## Baseline Model Features

### Temporal Features
1. **DAY_OF_WEEK**
- **Description**: Day of week
- **Data Source**: BTS Flight
- **Null %**: 0.00%
- **Rationale**: Temporal pattern - certain days (e.g., Mondays, Fridays) have higher delay rates due to business travel patterns and airport congestion

2. **MONTH**
- **Description**: Month of flight
- **Data Source**: BTS Flight
- **Null %**: 0.00%
- **Rationale**: Seasonal patterns - weather seasons, holidays, peak travel periods (summer, winter holidays) affect delay rates

3. **DEP_TIME_BLK**
- **Description**: CRS departure time block in hourly intervals
- **Data Source**: BTS Flight
- **Null %**: 0.00%
- **Rationale**: Time of day effects - rush hours (early morning, evening) have higher congestion and delay rates. Overnight operations may have different patterns. This feature captures time-of-day patterns without being too granular, and is available 2 hours before departure since it's based on the scheduled time, not the actual departure time. 


### Airport Factors
1. **ORIGIN** 
- **Description**: Origin airport code (IATA)
- **Data Source**: BTS Flight
- **Null %**: 0.00%
- **Rationale**: Airport-specific delay patterns - some airports are more prone to delays due to congestion, infrastructure, weather patterns, and operational capacity

2. **DEST** 
- **Description**: Destination airport code (IATA)
- **Data Source**: BTS Flight
- **Null %**: 0.00%
- **Rationale**: Destination airport characteristics - congestion at destination can cause delays, and weather at destination may affect departure decisions

### Weather Factors
1. **HourlyWindSpeed** (at origin, 2 hours before departure)
- **Description**: Horizontal wind speed rate in meters per second (scaled by 10, 9999=missing)
- **Data Source**: NOAA Weather
- **Null %**: 0.33%
- **Rationale**: High winds can cause delays and cancellations. Crosswinds above certain thresholds require different runway operations or may ground flights.

2. **HourlyVisibility** (at origin, 2 hours before departure)
- **Description**: Horizontal visibility distance in meters (999999=missing, >160000 entered as 160000)
- **Data Source**: NOAA Weather
- **Null %**: 0.29%
- **Rationale**: Low visibility (fog, haze, precipitation) causes significant delays as it affects takeoff and landing safety requirements.

3. **HourlyPrecipitation** (at origin, 2 hours before departure)
- **Description**: Liquid precipitation depth in millimeters (scaled by 10, 9999=missing)
- **Data Source**: NOAA Weather
- **Null %**: 11.35%
- **Rationale**: Precipitation directly impacts operations - rain, snow, and freezing precipitation can cause delays and cancellations.


### Carrier Factors
1. **OP_UNIQUE_CARRIER**
- **Description**: Unique carrier code - when same code used by multiple carriers, numeric suffix added (e.g., PA, PA(1))
- **Data Source**: BTS Flight
- **Null %**: 0.00%
- **Rationale**: Carrier-specific operational efficiency and reliability - different airlines have varying on-time performance, maintenance practices, and operational strategies.



#### Other factors considered but not currently used:
##### Weather Features
1. **HourlySkyConditions** (Null: 2.35%)
   - Cloud coverage and ceiling height - affects visibility and landing requirements

2. **HourlySeaLevelPressure** (Null: 10.95%)
   - Pressure systems indicate weather patterns and storm systems

##### Temporal Features
3. **QUARTER** (Null: 0.00%)
   - Quarter of year - captures broader seasonal trends beyond month

4. **DAY_OF_MONTH** (Null: 0.00%)
   - Day of month - may capture monthly patterns (beginning/end of month travel)

##### Flight Characteristics
5. **DISTANCE_GROUP** (Null: 0.00%)
    - Distance intervals in 250-mile increments - may be more stable than continuous distance

6. **CRS_ELAPSED_TIME** (Null: 0.00%)
    - Scheduled flight duration - longer flights may buffer delays differently

7. **CRS_DEP_TIME** (Null: 0.00%)
    - Scheduled departure time (HHMM) - more granular than time block

##### Airport Characteristics
8. **origin_type / dest_type** (Null: 0.00%)
    - Airport size/type (large_airport, medium_airport, small_airport) - capacity and congestion levels

9. **origin_region / dest_region** (Null: 0.00%)
    - Geographic region/state code - weather patterns, regulations, operational differences

10. **DISTANCE_GROUP** (Null: 0.00%)
    - Distance intervals in 250-mile increments - may be more stable than continuous distance

### Input Features

|   Temporal Features  |   Airport Factors    |  Weather Factors |  Flight Factors   | 
|-----------------|---------------|----------|---------------|
| DAY_OF_WEEK  | ORIGIN | HourlyWindSpeed | OP_UNIQUE_CARRIER | 
| MONTH  | DEST | HourlyVisibility | DIST |  |
| DEP_TIME_BLK  | |  HourlyPrecipitation |  |  
|   Total  |       |  |        |           
| 3  | 2 | 3 | 2 |  



## Helper Function for Classification Metrics

In [0]:
# Classification Metrics for Bucketed Flight Delay Predictions (4-Bucket Version)
# Converts continuous regression predictions into 4 discrete buckets for operational evaluation
# Buckets: Early (<0), OnTime (0-15), Delayed (15-60), Severe (60+)

from pyspark.sql import DataFrame
from pyspark.sql.functions import col, when
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator
import pandas as pd
import numpy as np


def compute_classification_metrics(predictions_df, label_col="DEP_DELAY", prediction_col="prediction"):
    """
    Compute classification metrics by bucketing continuous delay predictions into 4 buckets.
    
    Buckets:
    - Early: < 0 minutes
    - OnTime: 0-15 minutes
    - Delayed: 15-60 minutes
    - Severe: 60+ minutes
    
    Args:
        predictions_df: Spark DataFrame with actual delays and predictions
        label_col: Column name for actual delay values
        prediction_col: Column name for predicted delay values
    
    Returns:
        dict: Dictionary containing all classification and domain-specific metrics
        pd.DataFrame: Detailed metrics DataFrame with LaTeX equations
    """
    
    # ========================================
    # 1. On-Time Performance Prediction (OTPA)
    # ========================================
    # Binary: On-Time (<15 min) vs Delayed (>=15 min)
    
    df_otpa = predictions_df.withColumn(
        "actual_ontime",
        when(col(label_col) < 15, 1).otherwise(0)
    ).withColumn(
        "predicted_ontime",
        when(col(prediction_col) < 15, 1).otherwise(0)
    )
    
    # Compute confusion matrix for OTPA
    tp_otpa = df_otpa.filter((col("predicted_ontime") == 1) & (col("actual_ontime") == 1)).count()
    tn_otpa = df_otpa.filter((col("predicted_ontime") == 0) & (col("actual_ontime") == 0)).count()
    fp_otpa = df_otpa.filter((col("predicted_ontime") == 1) & (col("actual_ontime") == 0)).count()
    fn_otpa = df_otpa.filter((col("predicted_ontime") == 0) & (col("actual_ontime") == 1)).count()
    
    # OTPA Metrics
    otpa_precision = tp_otpa / (tp_otpa + fp_otpa) if (tp_otpa + fp_otpa) > 0 else 0.0
    otpa_recall = tp_otpa / (tp_otpa + fn_otpa) if (tp_otpa + fn_otpa) > 0 else 0.0
    otpa_f1 = 2 * (otpa_precision * otpa_recall) / (otpa_precision + otpa_recall) if (otpa_precision + otpa_recall) > 0 else 0.0
    otpa_accuracy = (tp_otpa + tn_otpa) / (tp_otpa + tn_otpa + fp_otpa + fn_otpa) if (tp_otpa + tn_otpa + fp_otpa + fn_otpa) > 0 else 0.0
    
    # ========================================
    # 2. Severe Delay Detection Rate (SDDR)
    # ========================================
    # Focus on delays >= 60 minutes
    
    df_severe = predictions_df.withColumn(
        "actual_severe",
        when(col(label_col) >= 60, 1).otherwise(0)
    ).withColumn(
        "predicted_severe",
        when(col(prediction_col) >= 60, 1).otherwise(0)
    )
    
    # Compute confusion matrix for severe delays
    tp_severe = df_severe.filter((col("predicted_severe") == 1) & (col("actual_severe") == 1)).count()
    fn_severe = df_severe.filter((col("predicted_severe") == 0) & (col("actual_severe") == 1)).count()
    fp_severe = df_severe.filter((col("predicted_severe") == 1) & (col("actual_severe") == 0)).count()
    tn_severe = df_severe.filter((col("predicted_severe") == 0) & (col("actual_severe") == 0)).count()
    
    # SDDR Metrics
    sddr_recall = tp_severe / (tp_severe + fn_severe) if (tp_severe + fn_severe) > 0 else 0.0
    sddr_precision = tp_severe / (tp_severe + fp_severe) if (tp_severe + fp_severe) > 0 else 0.0
    sddr_f1 = 2 * (sddr_precision * sddr_recall) / (sddr_precision + sddr_recall) if (sddr_precision + sddr_recall) > 0 else 0.0
    
    # ========================================
    # 3. 4-Bucket Classification
    # ========================================
    # Buckets: Early (<0), OnTime (0-15), Delayed (15-60), Severe (60+)
    
    df_buckets = predictions_df.withColumn(
        "actual_bucket",
        when(col(label_col) < 0, "Early")
        .when((col(label_col) >= 0) & (col(label_col) < 15), "OnTime")
        .when((col(label_col) >= 15) & (col(label_col) < 60), "Delayed")
        .otherwise("Severe")
    ).withColumn(
        "predicted_bucket",
        when(col(prediction_col) < 0, "Early")
        .when((col(prediction_col) >= 0) & (col(prediction_col) < 15), "OnTime")
        .when((col(prediction_col) >= 15) & (col(prediction_col) < 60), "Delayed")
        .otherwise("Severe")
    )
    
    # Compute per-bucket metrics
    buckets = ["Early", "OnTime", "Delayed", "Severe"]
    bucket_metrics = {}
    
    for bucket in buckets:
        tp = df_buckets.filter((col("predicted_bucket") == bucket) & (col("actual_bucket") == bucket)).count()
        fp = df_buckets.filter((col("predicted_bucket") == bucket) & (col("actual_bucket") != bucket)).count()
        fn = df_buckets.filter((col("predicted_bucket") != bucket) & (col("actual_bucket") == bucket)).count()
        tn = df_buckets.filter((col("predicted_bucket") != bucket) & (col("actual_bucket") != bucket)).count()
        
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
        accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0.0
        
        bucket_metrics[bucket] = {
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "accuracy": accuracy,
            "support": tp + fn  # Number of actual instances in this bucket
        }
    
    # Overall bucket accuracy
    total_correct = df_buckets.filter(col("predicted_bucket") == col("actual_bucket")).count()
    total_count = df_buckets.count()
    overall_bucket_accuracy = total_correct / total_count if total_count > 0 else 0.0
    
    # ========================================
    # 4. Compile Results
    # ========================================
    
    results = {
        # OTPA Metrics
        "otpa_accuracy": round(otpa_accuracy, 4),
        "otpa_precision": round(otpa_precision, 4),
        "otpa_recall": round(otpa_recall, 4),
        "otpa_f1": round(otpa_f1, 4),
        "otpa_tp": tp_otpa,
        "otpa_tn": tn_otpa,
        "otpa_fp": fp_otpa,
        "otpa_fn": fn_otpa,
        
        # SDDR Metrics
        "sddr_recall": round(sddr_recall, 4),
        "sddr_precision": round(sddr_precision, 4),
        "sddr_f1": round(sddr_f1, 4),
        "sddr_tp": tp_severe,
        "sddr_fn": fn_severe,
        "sddr_fp": fp_severe,
        "sddr_tn": tn_severe,
        
        # Bucket Metrics
        "bucket_accuracy": round(overall_bucket_accuracy, 4),
        "bucket_metrics": {k: {mk: round(mv, 4) if isinstance(mv, float) else mv 
                               for mk, mv in v.items()} 
                          for k, v in bucket_metrics.items()}
    }
    
    # ========================================
    # 5. Create Metrics DataFrame with LaTeX
    # ========================================
    
    metrics_data = [
        {
            "Metric": "OTPA Accuracy",
            "Value": round(otpa_accuracy, 4),
            "Category": "On-Time Performance",
            "LaTeX": r"$\text{OTPA} = \frac{TP + TN}{TP + TN + FP + FN}$",
            "Description": "Accuracy of predicting on-time (<15 min) vs delayed (≥15 min)"
        },
        {
            "Metric": "OTPA Precision",
            "Value": round(otpa_precision, 4),
            "Category": "On-Time Performance",
            "LaTeX": r"$\text{Precision} = \frac{TP}{TP + FP}$",
            "Description": "Proportion of predicted on-time flights that are actually on-time"
        },
        {
            "Metric": "OTPA Recall",
            "Value": round(otpa_recall, 4),
            "Category": "On-Time Performance",
            "LaTeX": r"$\text{Recall} = \frac{TP}{TP + FN}$",
            "Description": "Proportion of actual on-time flights correctly identified"
        },
        {
            "Metric": "OTPA F1-Score",
            "Value": round(otpa_f1, 4),
            "Category": "On-Time Performance",
            "LaTeX": r"$F1 = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$",
            "Description": "Harmonic mean of OTPA precision and recall"
        },
        {
            "Metric": "SDDR (Recall)",
            "Value": round(sddr_recall, 4),
            "Category": "Severe Delay Detection",
            "LaTeX": r"$\text{SDDR} = \frac{TP_{severe}}{TP_{severe} + FN_{severe}}$",
            "Description": "Proportion of severe delays (≥60 min) correctly identified"
        },
        {
            "Metric": "SDDR Precision",
            "Value": round(sddr_precision, 4),
            "Category": "Severe Delay Detection",
            "LaTeX": r"$\text{Precision} = \frac{TP_{severe}}{TP_{severe} + FP_{severe}}$",
            "Description": "Proportion of predicted severe delays that are actually severe"
        },
        {
            "Metric": "SDDR F1-Score",
            "Value": round(sddr_f1, 4),
            "Category": "Severe Delay Detection",
            "LaTeX": r"$F1 = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$",
            "Description": "Harmonic mean of SDDR precision and recall"
        },
        {
            "Metric": "Bucket Accuracy",
            "Value": round(overall_bucket_accuracy, 4),
            "Category": "4-Bucket Classification",
            "LaTeX": r"$\text{Accuracy} = \frac{\text{Correct Predictions}}{\text{Total Predictions}}$",
            "Description": "Overall accuracy across all 4 delay buckets"
        }
    ]
    
    # Add per-bucket metrics
    bucket_descriptions = {
        "Early": "Early departures (<0 min)",
        "OnTime": "On-time departures (0-15 min)",
        "Delayed": "Delayed departures (15-60 min)",
        "Severe": "Severely delayed departures (≥60 min)"
    }
    
    for bucket in buckets:
        metrics = bucket_metrics[bucket]
        metrics_data.append({
            "Metric": f"{bucket} Precision",
            "Value": round(metrics["precision"], 4),
            "Category": "Bucket-Specific",
            "LaTeX": r"$\text{Precision} = \frac{TP}{TP + FP}$",
            "Description": f"{bucket_descriptions[bucket]} - support: {metrics['support']}"
        })
        metrics_data.append({
            "Metric": f"{bucket} Recall",
            "Value": round(metrics["recall"], 4),
            "Category": "Bucket-Specific",
            "LaTeX": r"$\text{Recall} = \frac{TP}{TP + FN}$",
            "Description": f"{bucket_descriptions[bucket]} - support: {metrics['support']}"
        })
        metrics_data.append({
            "Metric": f"{bucket} F1-Score",
            "Value": round(metrics["f1"], 4),
            "Category": "Bucket-Specific",
            "LaTeX": r"$F1 = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$",
            "Description": f"{bucket_descriptions[bucket]} - support: {metrics['support']}"
        })
    
    metrics_df = pd.DataFrame(metrics_data)
    
    return results, metrics_df


def print_classification_summary(results, metrics_df):
    """
    Print a formatted summary of classification metrics.
    
    Args:
        results: Dictionary from compute_classification_metrics
        metrics_df: DataFrame from compute_classification_metrics
    """
    print("=" * 80)
    print("CLASSIFICATION METRICS SUMMARY (4-Bucket)")
    print("=" * 80)
    
    print("\n--- On-Time Performance Prediction Accuracy (OTPA) ---")
    print(f"Threshold: <15 minutes = On-Time, ≥15 minutes = Delayed")
    print(f"Accuracy:  {results['otpa_accuracy']:.4f}")
    print(f"Precision: {results['otpa_precision']:.4f}")
    print(f"Recall:    {results['otpa_recall']:.4f}")
    print(f"F1-Score:  {results['otpa_f1']:.4f}")
    print(f"Confusion Matrix: TP={results['otpa_tp']}, TN={results['otpa_tn']}, FP={results['otpa_fp']}, FN={results['otpa_fn']}")
    
    print("\n--- Severe Delay Detection Rate (SDDR) ---")
    print(f"Threshold: ≥60 minutes = Severe Delay")
    print(f"Recall (SDDR): {results['sddr_recall']:.4f}")
    print(f"Precision:     {results['sddr_precision']:.4f}")
    print(f"F1-Score:      {results['sddr_f1']:.4f}")
    print(f"Confusion Matrix: TP={results['sddr_tp']}, FN={results['sddr_fn']}, FP={results['sddr_fp']}, TN={results['sddr_tn']}")
    
    print("\n--- 4-Bucket Classification ---")
    print(f"Buckets:")
    print(f"  - Early: <0 minutes")
    print(f"  - OnTime: 0-15 minutes")
    print(f"  - Delayed: 15-60 minutes")
    print(f"  - Severe: ≥60 minutes")
    print(f"\nOverall Bucket Accuracy: {results['bucket_accuracy']:.4f}")
    print("\nPer-Bucket Performance:")
    for bucket, metrics in results['bucket_metrics'].items():
        print(f"  {bucket:10s} - Precision: {metrics['precision']:.4f}, Recall: {metrics['recall']:.4f}, "
              f"F1: {metrics['f1']:.4f}, Support: {metrics['support']}")
    
    print("\n" + "=" * 80)
    print("DETAILED METRICS TABLE")
    print("=" * 80)
    print(metrics_df.to_string(index=False))
    print("=" * 80)


# Example usage function to integrate with CV_12m.py
def add_classification_metrics_to_cv(predictions_df, split_name="validation"):
    """
    Wrapper function to compute and display classification metrics for a CV fold.
    
    Args:
        predictions_df: Spark DataFrame with predictions
        split_name: Name of the split (train/validation/test)
    
    Returns:
        results: Dictionary of metrics
        metrics_df: DataFrame of metrics with LaTeX
    """
    print(f"\n{'='*80}")
    print(f"Computing 4-Bucket Classification Metrics for {split_name.upper()} set")
    print(f"{'='*80}")
    
    results, metrics_df = compute_classification_metrics(predictions_df)
    print_classification_summary(results, metrics_df)
    
    return results, metrics_df

# Model 1: No-Cross-Validation/3-month/Data-Imputed

In [0]:
# OTPW
df_otpw = spark.read.format("csv").option("header","true").load(f"dbfs:/mnt/mids-w261/OTPW_3M_2015.csv")
display(df_otpw)

In [0]:
## Load the data into a dataframe
df = df_otpw

print(f"Total rows: {df.count()}")
display(df.limit(10))

# Experiment 1: Model 1 No-Cross-Validation/3-month/Data-Imputed
Updated to run with classification metrics

In [0]:
# ============================================================================
# Additional Imports for Main Pipeline
# ============================================================================

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, isnan, isnull, count, avg, regexp_replace, trim, length
from pyspark.sql.types import DoubleType, IntegerType, StringType
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    VectorAssembler,      # Combines features into single vector
    StringIndexer,        # Converts strings to numeric indices
    OneHotEncoder,        # Converts indices to binary vectors
    StandardScaler,       # Standardizes features (mean=0, std=1)
    Imputer               # Fills missing values with median
)
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
import pandas as pd
import numpy as np


# ============================================================================
# 1. Data Loading
# ============================================================================

# Load OTPW data from mounted DBFS
# This is a 3-month subset (Jan-Mar 2015) for faster iteration
# Full path: dbfs:/mnt/mids-w261/OTPW_3M_2015.csv
print("=" * 80)
print("STEP 1: LOADING DATA")
print("=" * 80)
df_otpw = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("dbfs:/mnt/mids-w261/OTPW_3M_2015.csv")

# Create working copy
df = df_otpw
print(f"✓ Loaded {df.count()} rows from OTPW_3M_2015.csv")


# ============================================================================
# 1b. Cluster Configuration
# ============================================================================

# Display cluster resources for reproducibility and debugging
# Important for:
# - Debugging OOM errors (check if cluster is undersized)
# - Reproducing results (document exact cluster configuration)
# - Cost tracking (understand resource usage)

print("\n" + "=" * 80)
print("CLUSTER CONFIGURATION")
print("=" * 80)

sc = spark.sparkContext

# Get number of executors (exclude driver node)
num_executors = len(sc._jsc.sc().statusTracker().getExecutorInfos()) - 1

# Get executor configuration from Spark conf
executor_memory = sc.getConf().get("spark.executor.memory", "Unknown")
executor_cores = sc.getConf().get("spark.executor.cores", "Unknown")
actual_cores = sc.defaultParallelism / num_executors if num_executors > 0 else "Unknown"

print(f"Number of Executors: {num_executors}")
print(f"Executor Memory: {executor_memory}")
print(f"Executor Cores: {executor_cores}")
print(f"Estimated Cores per Executor: {int(actual_cores) if isinstance(actual_cores, float) else actual_cores}")
print(f"Total Parallelism: {sc.defaultParallelism}")
print("=" * 80)


# ============================================================================
# 2. Feature Selection
# ============================================================================

# Define the 10 baseline features
# These represent various delay factors without data leakage
# (all known 2 hours before scheduled departure)
print("\n" + "=" * 80)
print("STEP 2: FEATURE SELECTION")
print("=" * 80)

baseline_features = [
    # Temporal Features (3 features)
    'DAY_OF_WEEK',           # Integer (1=Monday, 7=Sunday) - captures weekly patterns
    'MONTH',                 # Integer (1-12) - captures seasonal patterns
    'DEP_TIME_BLK',          # String (e.g., "0600-0659") - captures time-of-day patterns
    
    # Airport Features (2 features)
    'ORIGIN',                # String (e.g., "SFO") - origin airport code
    'DEST',                  # String (e.g., "JFK") - destination airport code
    
    # Flight Characteristics (2 features)
    'OP_UNIQUE_CARRIER',     # String (e.g., "AA") - operating carrier code
    'DISTANCE',              # Double (miles) - flight distance
    
    # Weather Features (3 features - at origin, 2h before departure)
    'HourlyWindSpeed',       # Double (mph) - wind speed
    'HourlyVisibility',      # Double (miles) - visibility
    'HourlyPrecipitation'    # Double (inches) - precipitation
]

# Target variable (what we're predicting)
target_var = 'DEP_DELAY'  # Departure delay in minutes

# Select only the columns we need (features + label)
# This reduces memory footprint and prevents accidental data leakage
feature_df = df.select(baseline_features + [target_var])

print(f"✓ Selected {len(baseline_features)} features + 1 target variable")
print(f"  - Temporal: 3 features")
print(f"  - Airport: 2 features")
print(f"  - Flight: 2 features")
print(f"  - Weather: 3 features")

# ============================================================================
# 3. Label Preparation
# ============================================================================

# Ensure label is DoubleType and non-null/non-NaN
# Spark ML LinearRegression requires DoubleType labels with no nulls
print("\n" + "=" * 80)
print("STEP 3: LABEL PREPARATION")
print("=" * 80)

# Cast label to DoubleType (required by Spark ML)
feature_df = feature_df.withColumn(target_var, col(target_var).cast(DoubleType()))

# Filter out rows with null or NaN labels
# These rows cannot be used for training
feature_df = feature_df.filter(~(col(target_var).isNull() | isnan(col(target_var))))

print(f"✓ Label '{target_var}' cast to DoubleType")
print(f"✓ Filtered out null/NaN labels")


# ============================================================================
# 4. Data Quality Check
# ============================================================================

print("\n" + "=" * 80)
print("STEP 4: DATA QUALITY CHECK")
print("=" * 80)

print(f"\nDataset Shape:")
print(f"  Total rows: {feature_df.count():,}")
print(f"  Total columns: {len(feature_df.columns)}")

# Check for nulls in each column
# This helps us understand which features need imputation
print("\nNull counts by column:")
for col_name in feature_df.columns:
    null_count = feature_df.filter(col(col_name).isNull() | isnan(col(col_name))).count()
    total_count = feature_df.count()
    null_pct = (null_count / total_count) * 100 if total_count > 0 else 0
    print(f"  {col_name:25s}: {null_count:8,} ({null_pct:5.2f}%)")

# Check data types to ensure correct schema
print("\nData types:")
feature_df.printSchema()

# ============================================================================
# 5. Numerical Feature Cleaning and Imputation Setup
# ============================================================================

print("\n" + "=" * 80)
print("STEP 5: NUMERICAL FEATURE CLEANING")
print("=" * 80)

# Define numerical features that need cleaning and imputation
# These are continuous variables that may contain non-numeric artifacts or nulls
numerical_features = [
    'HourlyWindSpeed',       # Weather: wind speed (may have "mph" suffix)
    'HourlyVisibility',      # Weather: visibility (may have "mi" suffix)
    'HourlyPrecipitation',   # Weather: precipitation (may have "in" suffix)
    'DISTANCE'               # Flight: distance (should be clean but verify)
]

# Clean numerical columns: remove non-numeric characters and cast to DoubleType
# This is necessary because source data may contain string artifacts like "12.5mph"
for _feat in numerical_features:
    # Step 1: Cast to string to enable regex operations
    # Step 2: Remove all non-numeric characters except +, -, and .
    #         Example: "12.5mph" -> "12.5"
    feature_df = feature_df.withColumn(
        _feat, 
        regexp_replace(col(_feat).cast(StringType()), r"[^0-9+\-\.]", "")
    )
    
    # Step 3: Convert empty strings to null (for imputation)
    #         Empty strings would cause cast errors
    feature_df = feature_df.withColumn(
        _feat, 
        when(length(trim(col(_feat))) == 0, None).otherwise(col(_feat))
    )
    
    # Step 4: Cast to DoubleType for modeling
    #         Spark ML requires numeric types for numerical features
    feature_df = feature_df.withColumn(_feat, col(_feat).cast(DoubleType()))

print(f"✓ Cleaned {len(numerical_features)} numerical features")

# Define Imputer transformers (median strategy)
# These will be added to the pipeline and run during fit/transform
# Median is more robust to outliers than mean
imputers = {}
for feature in numerical_features:
    imputers[feature] = Imputer(
        inputCols=[feature],                # Original column with nulls
        outputCols=[f"{feature}_imputed"],  # New column with imputed values
        strategy="median"                   # Use median (robust to outliers)
    )

print(f"✓ Created {len(imputers)} median imputers for pipeline")

# ============================================================================
# 6. Categorical Feature Preparation
# ============================================================================

print("\n" + "=" * 80)
print("STEP 6: CATEGORICAL FEATURE PREPARATION")
print("=" * 80)

# Define categorical features
# These will be encoded using StringIndexer + OneHotEncoder
categorical_features = [
    'DAY_OF_WEEK',        # Temporal: day of week (1-7)
    'MONTH',              # Temporal: month (1-12)
    'DEP_TIME_BLK',       # Temporal: departure time block
    'ORIGIN',             # Airport: origin code
    'DEST',               # Airport: destination code
    'OP_UNIQUE_CARRIER'   # Flight: carrier code
]

# Replace nulls with "UNKNOWN" for categorical features
# This prevents StringIndexer from failing on null values
for feature in categorical_features:
    # For numeric-coded categoricals (DAY_OF_WEEK, MONTH, DEP_TIME_BLK),
    # cast to string first to ensure consistent handling
    if feature in ['DAY_OF_WEEK', 'MONTH', 'DEP_TIME_BLK']:
        feature_df = feature_df.withColumn(
            f"{feature}_clean",
            when(col(feature).isNull(), "UNKNOWN").otherwise(col(feature).cast(StringType()))
        )
    else:
        # For string categoricals (ORIGIN, DEST, CARRIER), handle nulls only
        feature_df = feature_df.withColumn(
            f"{feature}_clean",
            when(col(feature).isNull(), "UNKNOWN").otherwise(col(feature))
        )

print(f"✓ Prepared {len(categorical_features)} categorical features")
print(f"  - Replaced nulls with 'UNKNOWN'")
print(f"  - Created '_clean' columns for pipeline")

# ============================================================================
# 7. Build ML Pipeline
# ============================================================================

print("\n" + "=" * 80)
print("STEP 7: BUILDING ML PIPELINE")
print("=" * 80)

# Pipeline Stages:
# 1. Imputers: Fill missing numerical values with median
# 2. StringIndexers: Convert categorical strings to numeric indices
# 3. OneHotEncoders: Convert indices to binary vectors
# 4. VectorAssembler: Combine all features into single vector
# 5. StandardScaler: Standardize features (mean=0, std=1)
# 6. LinearRegression: Train linear model

# Stage 1: Index categorical features
# StringIndexer converts strings to numeric indices (most frequent = 0)
indexers = []
for feature in categorical_features:
    indexer = StringIndexer(
        inputCol=f"{feature}_clean",       # Input: cleaned categorical column
        outputCol=f"{feature}_indexed",    # Output: numeric index
        handleInvalid="keep"               # Keep unknown categories (assign special index)
    )
    indexers.append(indexer)

print(f"✓ Created {len(indexers)} StringIndexers")

# Stage 2: One-hot encode categorical features
# OneHotEncoder converts indices to binary vectors (prevents ordinal assumption)
encoders = []
for feature in categorical_features:
    encoder = OneHotEncoder(
        inputCols=[f"{feature}_indexed"],  # Input: numeric index
        outputCols=[f"{feature}_encoded"], # Output: binary vector
        dropLast=True                      # Drop last category to avoid multicollinearity
    )
    encoders.append(encoder)

print(f"✓ Created {len(encoders)} OneHotEncoders")

# Stage 3: Assemble all features into a single vector
# VectorAssembler combines all features (numerical + encoded categorical) into one column
feature_columns = (
    [f"{feat}_imputed" for feat in numerical_features] +      # Imputed numerical features
    [f"{feat}_encoded" for feat in categorical_features]      # One-hot encoded categoricals
)

assembler = VectorAssembler(
    inputCols=feature_columns,    # All feature columns to combine
    outputCol="features",         # Output: single feature vector
    handleInvalid="skip"          # Skip rows with invalid values (e.g., NaN, Inf)
)

print(f"✓ Created VectorAssembler with {len(feature_columns)} input columns")

# Stage 4: Standardize features
# StandardScaler normalizes features to mean=0, std=1
# This improves convergence and makes coefficients comparable
scaler = StandardScaler(
    inputCol="features",          # Input: raw feature vector
    outputCol="scaled_features",  # Output: standardized feature vector
    withStd=True,                 # Scale to unit variance
    withMean=True                 # Center to zero mean
)

print(f"✓ Created StandardScaler (mean=0, std=1)")

# Stage 5: Linear Regression Model
# Baseline model with no regularization for interpretability
lr = LinearRegression(
    featuresCol="scaled_features",  # Input: standardized features
    labelCol=target_var,            # Target: DEP_DELAY
    maxIter=100,                    # Maximum iterations for convergence
    regParam=0.0,                   # No L2 regularization (baseline)
    elasticNetParam=0.0             # No L1 regularization (baseline)
)

print(f"✓ Created LinearRegression (maxIter=100, no regularization)")

# Assemble complete pipeline
# Order: Imputers -> StringIndexers -> OneHotEncoders -> VectorAssembler -> StandardScaler -> LinearRegression
imputer_stages = list(imputers.values())
pipeline_stages = imputer_stages + indexers + encoders + [assembler, scaler, lr]
pipeline = Pipeline(stages=pipeline_stages)

print(f"✓ Built complete pipeline with {len(pipeline_stages)} stages")
print(f"  - {len(imputer_stages)} Imputers")
print(f"  - {len(indexers)} StringIndexers")
print(f"  - {len(encoders)} OneHotEncoders")
print(f"  - 1 VectorAssembler")
print(f"  - 1 StandardScaler")
print(f"  - 1 LinearRegression")

# ============================================================================
# 8. Split Data into Train/Val/Test Sets
# ============================================================================

print("\n" + "=" * 80)
print("STEP 8: TRAIN/VAL/TEST SPLIT")
print("=" * 80)

# Split data (60% train, 20% validation, 20% test)
# seed=42 ensures reproducibility
train_df, val_df, test_df = feature_df.randomSplit([0.6, 0.2, 0.2], seed=42)

train_count = train_df.count()
val_count = val_df.count()
test_count = test_df.count()
total_count = train_count + val_count + test_count

print(f"✓ Split complete:")
print(f"  Training set:   {train_count:8,} rows ({train_count/total_count*100:.1f}%)")
print(f"  Validation set: {val_count:8,} rows ({val_count/total_count*100:.1f}%)")
print(f"  Test set:       {test_count:8,} rows ({test_count/total_count*100:.1f}%)")


# ============================================================================
# 9. Train the Model
# ============================================================================

print("\n" + "=" * 80)
print("STEP 9: MODEL TRAINING")
print("=" * 80)

print("Training baseline linear regression model...")
print("This may take a few minutes...")

# Fit the pipeline on training data
# This runs all stages: imputation -> encoding -> assembly -> scaling -> regression
model = pipeline.fit(train_df)

print("✓ Model training completed!")


# ============================================================================
# 10. Make Predictions
# ============================================================================

print("\n" + "=" * 80)
print("STEP 10: MAKING PREDICTIONS")
print("=" * 80)

# Transform validation and test sets using the fitted pipeline
# This applies all transformations and generates predictions
val_predictions = model.transform(val_df)
test_predictions = model.transform(test_df)

print("✓ Generated predictions on validation set")
print("✓ Generated predictions on test set")

# Display sample predictions from test set
print("\nSample predictions from test set (first 20 rows):")
test_predictions.select(
    target_var,           # Actual delay
    "prediction",         # Predicted delay
    "ORIGIN",            # Origin airport
    "DEST",              # Destination airport
    "DAY_OF_WEEK",       # Day of week
    "MONTH"              # Month
).show(20)


# ============================================================================
# 11. Evaluate Regression Performance
# ============================================================================

print("\n" + "=" * 80)
print("STEP 11: REGRESSION METRICS EVALUATION")
print("=" * 80)

# Define evaluators for different metrics
# RMSE: Root Mean Squared Error (penalizes large errors more)
evaluator_rmse = RegressionEvaluator(
    labelCol=target_var,
    predictionCol="prediction",
    metricName="rmse"
)

# MAE: Mean Absolute Error (average magnitude of errors)
evaluator_mae = RegressionEvaluator(
    labelCol=target_var,
    predictionCol="prediction",
    metricName="mae"
)

# R²: Coefficient of Determination (proportion of variance explained)
evaluator_r2 = RegressionEvaluator(
    labelCol=target_var,
    predictionCol="prediction",
    metricName="r2"
)

# Generate predictions on TRAINING set (for comparison)
train_predictions = model.transform(train_df)

# Calculate metrics for TRAINING set
train_rmse = evaluator_rmse.evaluate(train_predictions)
train_mae = evaluator_mae.evaluate(train_predictions)
train_r2 = evaluator_r2.evaluate(train_predictions)

# Calculate metrics for VALIDATION set
val_rmse = evaluator_rmse.evaluate(val_predictions)
val_mae = evaluator_mae.evaluate(val_predictions)
val_r2 = evaluator_r2.evaluate(val_predictions)

# Calculate metrics for TEST set
test_rmse = evaluator_rmse.evaluate(test_predictions)
test_mae = evaluator_mae.evaluate(test_predictions)
test_r2 = evaluator_r2.evaluate(test_predictions)

# Create regression metrics DataFrame (no LaTeX columns for simplicity)
regression_metrics_df = pd.DataFrame([
    {
        "split": "train",
        "rmse": round(train_rmse, 4),
        "mae": round(train_mae, 4),
        "r2": round(train_r2, 4),
        "mse": round(train_rmse**2, 4),
    },
    {
        "split": "validation",
        "rmse": round(val_rmse, 4),
        "mae": round(val_mae, 4),
        "r2": round(val_r2, 4),
        "mse": round(val_rmse**2, 4),
    },
    {
        "split": "test",
        "rmse": round(test_rmse, 4),
        "mae": round(test_mae, 4),
        "r2": round(test_r2, 4),
        "mse": round(test_rmse**2, 4),
    }
])

# Display results
print("=" * 80)
print("REGRESSION PERFORMANCE METRICS")
print("=" * 80)
print("\nTRAIN SET:")
print(f"  RMSE (Root Mean Squared Error):     {train_rmse:8.2f} minutes")
print(f"  MAE (Mean Absolute Error):          {train_mae:8.2f} minutes")
print(f"  R² (Coefficient of Determination):  {train_r2:8.4f}")

print("\nVALIDATION SET:")
print(f"  RMSE (Root Mean Squared Error):     {val_rmse:8.2f} minutes")
print(f"  MAE (Mean Absolute Error):          {val_mae:8.2f} minutes")
print(f"  R² (Coefficient of Determination):  {val_r2:8.4f}")

print("\nTEST SET:")
print(f"  RMSE (Root Mean Squared Error):     {test_rmse:8.2f} minutes")
print(f"  MAE (Mean Absolute Error):          {test_mae:8.2f} minutes")
print(f"  R² (Coefficient of Determination):  {test_r2:8.4f}")

print("=" * 80)
print("\nInterpretation:")
print(f"  - Train RMSE: {train_rmse:.2f} | Val RMSE: {val_rmse:.2f} | Test RMSE: {test_rmse:.2f}")
print(f"  - Overfitting check: Compare train vs val/test metrics")
print(f"  - Test set explains {test_r2*100:.2f}% of variance in delays (R²)")
print("=" * 80)

# Display regression metrics DataFrame
print("\n" + "=" * 80)
print("REGRESSION METRICS DATAFRAME")
print("=" * 80)
display(regression_metrics_df)

# ============================================================================
# 12. Extract Model Coefficients (Feature Importance)
# ============================================================================

print("\n" + "=" * 80)
print("STEP 12: FEATURE IMPORTANCE ANALYSIS")
print("=" * 80)

# Get the linear regression model from the pipeline
# It's the last stage after all transformations
lr_model = model.stages[-1]

# Extract coefficients and intercept
# Coefficients show the impact of each feature on the prediction
coefficients = lr_model.coefficients.toArray()
intercept = lr_model.intercept

print(f"✓ Extracted {len(coefficients)} coefficients")
print(f"✓ Intercept: {intercept:.4f}")

# Create a DataFrame with feature names and coefficients
# Extract expanded feature names from VectorAssembler metadata
# This gives us human-readable names for one-hot encoded features
try:
    attrs = []
    feats_meta = test_predictions.schema["features"].metadata
    if "ml_attr" in feats_meta and "attrs" in feats_meta["ml_attr"]:
        ml_attrs = feats_meta["ml_attr"]["attrs"]
        for t in ["binary", "numeric", "nominal"]:
            if t in ml_attrs:
                attrs.extend(ml_attrs[t])
        attrs = sorted(attrs, key=lambda x: x["idx"])  # sort by index
        feature_names = [a.get("name", f"feature_{a['idx']}") for a in attrs]
    else:
        feature_names = [f"feature_{i}" for i in range(len(coefficients))]

except Exception:
    feature_names = [f"feature_{i}" for i in range(len(coefficients))]

## Guard against length mismatch
if len(feature_names) != len(coefficients):
    feature_names = [f"feature_{i}" for i in range(len(coefficients))]

coef_df = pd.DataFrame({
    'feature': feature_names,
    'coefficient': coefficients
})

## Sort by absolute coefficient value
coef_df['abs_coefficient'] = coef_df['coefficient'].abs()
coef_df = coef_df.sort_values('abs_coefficient', ascending=False)

print("\nTop 10 Most Important Features (by absolute coefficient):")
print(coef_df.head(10).to_string(index=False))
print(f"\nIntercept: {intercept:.4f}")

# 13. Save Model (Optional)

# Save the trained model
# model.write().overwrite().save("/dbfs/FileStore/models/baseline_lr_model")

# Later, load the model:
# from pyspark.ml import PipelineModel
# loaded_model = PipelineModel.load("/dbfs/FileStore/models/baseline_lr_model")

# ============================================================================
# 14. Additional Analysis: Residuals
# ============================================================================

print("\n" + "=" * 80)
print("STEP 14: RESIDUAL ANALYSIS")
print("=" * 80)

# Calculate residuals (actual - predicted)
# Residuals help identify systematic errors in the model
test_predictions_with_residuals = test_predictions.withColumn(
    "residual",
    col(target_var) - col("prediction")
)

# Summary statistics of residuals
residual_stats = test_predictions_with_residuals.select(
    avg("residual").alias("mean_residual"),
    count("residual").alias("count")
).collect()

print(f"\nResidual Analysis (Test Set):")
print(f"  Mean Residual: {residual_stats[0]['mean_residual']:.4f} minutes")
print(f"  Count: {residual_stats[0]['count']:,}")
print(f"\nInterpretation:")
print(f"  - Mean residual close to 0 indicates unbiased predictions")
print(f"  - Current mean: {residual_stats[0]['mean_residual']:.4f} minutes")

# ============================================================================
# 15. Classification Metrics (4-Bucket)
# ============================================================================

print("\n" + "=" * 80)
print("STEP 15: CLASSIFICATION METRICS (4-BUCKET)")
print("=" * 80)

# Compute classification metrics for TRAIN set
print("\n" + "=" * 80)
print("TRAIN SET - CLASSIFICATION METRICS")
print("=" * 80)
train_class_results, train_class_metrics_df = compute_classification_metrics(
    predictions_df=train_predictions,
    label_col=target_var,
    prediction_col="prediction"
)
print_classification_summary(train_class_results, train_class_metrics_df)

# Compute classification metrics for VALIDATION set
print("\n" + "=" * 80)
print("VALIDATION SET - CLASSIFICATION METRICS")
print("=" * 80)
val_class_results, val_class_metrics_df = compute_classification_metrics(
    predictions_df=val_predictions,
    label_col=target_var,
    prediction_col="prediction"
)
print_classification_summary(val_class_results, val_class_metrics_df)

# Compute classification metrics for TEST set
print("\n" + "=" * 80)
print("TEST SET - CLASSIFICATION METRICS")
print("=" * 80)
test_class_results, test_class_metrics_df = compute_classification_metrics(
    predictions_df=test_predictions,
    label_col=target_var,
    prediction_col="prediction"
)
print_classification_summary(test_class_results, test_class_metrics_df)

# Create summary classification metrics DataFrame in the same style as CV tables
# Columns: split, otpa_accuracy, otpa_f1, sddr_recall, bucket_accuracy
classification_summary_df = pd.DataFrame([
    {
        "split": "train",
        "otpa_accuracy": round(train_class_results['otpa_accuracy'], 4),
        "otpa_f1": round(train_class_results['otpa_f1'], 4),
        "sddr_recall": round(train_class_results['sddr_recall'], 4),
        "bucket_accuracy": round(train_class_results['bucket_accuracy'], 4),
    },
    {
        "split": "validation",
        "otpa_accuracy": round(val_class_results['otpa_accuracy'], 4),
        "otpa_f1": round(val_class_results['otpa_f1'], 4),
        "sddr_recall": round(val_class_results['sddr_recall'], 4),
        "bucket_accuracy": round(val_class_results['bucket_accuracy'], 4),
    },
    {
        "split": "test",
        "otpa_accuracy": round(test_class_results['otpa_accuracy'], 4),
        "otpa_f1": round(test_class_results['otpa_f1'], 4),
        "sddr_recall": round(test_class_results['sddr_recall'], 4),
        "bucket_accuracy": round(test_class_results['bucket_accuracy'], 4),
    },
])

# Display summary classification metrics DataFrame
print("\n" + "=" * 80)
print("CLASSIFICATION METRICS SUMMARY")
print("=" * 80)
display(classification_summary_df)

In [0]:
displayHTML("""
<!DOCTYPE html>
<html>
<head>
  <script src="https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.min.js"></script>
  <script>
    mermaid.initialize({
      startOnLoad: true,
      theme: 'dark',
      themeVariables: {
        primaryColor: '#4a5568',
        primaryTextColor: '#fff',
        primaryBorderColor: '#cbd5e0',
        lineColor: '#cbd5e0',
        secondaryColor: '#2d3748',
        tertiaryColor: '#1a202c',
        background: '#1a202c',
        mainBkg: '#4a5568',
        secondBkg: '#2d3748',
        tertiaryBkg: '#1a202c'
      }
    });
  </script>
  <style>
    body { background-color: #1a202c; }
    .mermaid { background-color: #1a202c; }
    font-size: 500px;
  </style>
</head>
<body>
<div class="mermaid">
flowchart LR
    %% Input
    Input["<b>Input</b><br/>Raw DataFrame<br/>10 Features + Label"]

    %% Stage 1: Data Preparation
    subgraph S1["<b>Stage 1: Data Preparation</b>"]
        LabelClean["Cast DEP_DELAY to Double<br/>Filter Null/NaN Labels"]
        SelectFeat["Select 10 Baseline Features<br/>Temporal, Airport, Flight, Weather"]
        NumClean["Clean Numerical Features:<br/>Remove non-numeric chars<br/>Empty → Null<br/>Cast to Double"]
        LabelClean --> SelectFeat --> NumClean
    end

    %% Stage 2: Numerical Imputation (Median)
    subgraph S2["<b>Stage 2: Numerical Imputation (Median)</b>"]
        ImpWind["HourlyWindSpeed_imputed<br/>median(HourlyWindSpeed)"]
        ImpVis["HourlyVisibility_imputed<br/>median(HourlyVisibility)"]
        ImpPrec["HourlyPrecipitation_imputed<br/>median(HourlyPrecipitation)"]
        ImpDist["DISTANCE_imputed<br/>median(DISTANCE)"]
    end

    %% Stage 3: Categorical Encoding
    subgraph S3["<b>Stage 3: Categorical Encoding</b>"]
        DOW["DAY_OF_WEEK_clean<br/>StringIndexer + OHE"]
        Month["MONTH_clean<br/>StringIndexer + OHE"]
        TimeBlk["DEP_TIME_BLK_clean<br/>StringIndexer + OHE"]
        Origin["ORIGIN_clean<br/>StringIndexer + OHE"]
        Dest["DEST_clean<br/>StringIndexer + OHE"]
        Carrier["OP_UNIQUE_CARRIER_clean<br/>StringIndexer + OHE"]
    end

    %% Stage 4: Feature Assembly
    subgraph S4["<b>Stage 4: Feature Assembly</b>"]
        Assemble["VectorAssembler<br/>Combine Imputed Numerics + Encoded Categoricals"]
    end

    %% Stage 5: Standardization
    subgraph S5["<b>Stage 5: Standardization</b>"]
        Scale["StandardScaler<br/>features → scaled_features<br/>withMean=True, withStd=True"]
    end

    %% Stage 6: Linear Regression
    subgraph S6["<b>Stage 6: Model</b>"]
        LR["Linear Regression<br/>featuresCol=scaled_features<br/>labelCol=DEP_DELAY<br/>maxIter=100<br/>regParam=0.0, elasticNet=0.0"]
    end

    %% Output
    Output["<b>Output</b><br/>Predictions<br/>DEP_DELAY in minutes"]

    %% Connections
    Input --> S1
    S1 --> S2
    S1 --> S3
    S2 --> Assemble
    S3 --> Assemble
    Assemble --> Scale --> LR --> Output
</div>
</body>
</html>
""")

In [0]:
displayHTML("""
<!DOCTYPE html>
<html>
<head>
  <script src="https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.min.js"></script>
  <script>
    mermaid.initialize({
      startOnLoad: true,
      theme: 'dark',
      themeVariables: {
        primaryColor: '#4a5568',
        primaryTextColor: '#ffffff',
        primaryBorderColor: '#cbd5e0',
        lineColor: '#cbd5e0',
        secondaryColor: '#2d3748',
        tertiaryColor: '#1a202c',
        background: '#1a202c',
        mainBkg: '#4a5568',
        secondBkg: '#2d3748',
        tertiaryBkg: '#1a202c',
        fontSize: '24px'   // Larger base font for the whole diagram
      },
      flowchart: {
        useMaxWidth: true,
        htmlLabels: true,
        nodeSpacing: 40,
        rankSpacing: 60
      }
    });
  </script>
  <style>
    body {
      background-color: #1a202c;
    }
    .mermaid-wrapper {
      background-color: #1a202c;
      max-width: 1600px;    /* wider drawing area */
      margin: 0 auto;
      padding: 24px;
    }
    .mermaid {
      background-color: #1a202c;
      font-size: 24px;      /* extra scaling for diagram text */
    }
  </style>
</head>
<body>
<div class="mermaid-wrapper">
<div class="mermaid">
flowchart LR

    %% ==============================
    %% Input
    %% ==============================
    Input["<b>Input</b><br/>Raw 2015 Flights–Weather Data<br/>10 Features + Label"]

    %% ==============================
    %% Stage 1: Data Preparation
    %% ==============================
    subgraph S1["<b>Stage 1: Data Preparation</b>"]
        LabelClean["Clean Label<br/>DEP_DELAY → Double<br/>Drop Null / NaN Labels"]
        SelectFeat["Select 10 Baseline Features<br/>Temporal, Flight, Airport, Weather"]
        NumClean["Numeric Cleaning<br/>Strip Non‑Numeric<br/>Empty → Null<br/>Cast to Double"]
        LabelClean --> SelectFeat --> NumClean
    end

    %% ==============================
    %% Stage 2: Median Imputation
    %% ==============================
    subgraph S2["<b>Stage 2: Median Imputation</b>"]
        ImpNumerics["Imputer (Median)<br/>Wind, Visibility,<br/>Precipitation, Distance"]
    end

    %% ==============================
    %% Stage 3: Categorical Encoding
    %% ==============================
    subgraph S3["<b>Stage 3: Categorical Encoding</b>"]
        DOW["DAY_OF_WEEK<br/>StringIndexer + OHE"]
        Month["MONTH<br/>StringIndexer + OHE"]
        TimeBlk["DEP_TIME_BLK<br/>StringIndexer + OHE"]
        Origin["ORIGIN<br/>StringIndexer + OHE"]
        Dest["DEST<br/>StringIndexer + OHE"]
        Carrier["OP_UNIQUE_CARRIER<br/>StringIndexer + OHE"]
    end

    %% ==============================
    %% Stage 4: Feature Assembly
    %% ==============================
    subgraph S4["<b>Stage 4: Feature Assembly</b>"]
        Assemble["VectorAssembler<br/>Imputed Numerics + Encoded Categoricals<br/>→ features"]
    end

    %% ==============================
    %% Stage 5: Standardization
    %% ==============================
    subgraph S5["<b>Stage 5: Standardization</b>"]
        Scale["StandardScaler<br/>features → scaled_features<br/>withMean = True, withStd = True"]
    end

    %% ==============================
    %% Stage 6: Linear Regression
    %% ==============================
    subgraph S6["<b>Stage 6: Model</b>"]
        LR["Linear Regression<br/>labelCol = DEP_DELAY<br/>featuresCol = scaled_features<br/>maxIter = 100<br/>regParam = 0.0, elasticNet = 0.0"]
    end

    %% ==============================
    %% Output
    %% ==============================
    Output["<b>Output</b><br/>Predicted DEP_DELAY<br/>+ Classification Buckets"]

    %% ==============================
    %% Connections
    %% ==============================
    Input --> S1
    S1 --> S2
    S1 --> S3
    S2 --> Assemble
    S3 --> Assemble
    Assemble --> Scale --> LR --> Output
</div>
</div>
</body>
</html>
""")

# Experiment 1: Model 2 Cross-Validation/3-month/Data-Imputed
Cross-Validation/3-month/Data-Imputed

In [0]:
"""
================================================================================
Cross-Validation for Baseline Linear Regression (3-Month Dataset)
================================================================================

Purpose:
    Implements expanding window cross-validation for flight delay prediction
    using a baseline linear regression model with comprehensive feature engineering.

Key Features:
    - Expanding window CV: Each fold trains on progressively more historical data
    - Pre-checkpointed data: Uses coworker's fold pattern (OTPW_{version}_FOLD_{i}_{TRAIN/VAL/TEST})
    - Baseline pipeline: Numeric cleaning + median imputation + categorical OHE + standardization + LR
    - Comprehensive metrics: Regression (RMSE, MAE, R², MSE) + Classification (OTPA, SDDR, 4-bucket)
    - Memory management: Strategic caching with .count() to prevent OOM errors

Data Flow:
    1. Load pre-checkpointed folds from DBFS (3-month version)
    2. For each fold (except last):
       - Train model on expanding training set
       - Evaluate on validation set
       - Track best model by RMSE
    3. Evaluate best model on held-out test set
    4. Report comprehensive metrics and feature importance

Author: Emily Lieske
Date: November 2025
================================================================================
"""

# ============================================================================
# Imports
# ============================================================================

# PySpark SQL functions for data transformation
from pyspark.sql.functions import col, when, isnan, regexp_replace, trim, length
from pyspark.sql.types import DoubleType, StringType

# PySpark ML for pipeline construction and modeling
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder, StandardScaler, Imputer
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

# Standard libraries for metrics and timing
import numpy as np
import pandas as pd
import time


# ============================================================================
# Data Loading Functions
# ============================================================================

def _load_checkpointed_data(name, folder_path="dbfs:/student-groups/Group_4_2"):
    """
    Load a pre-checkpointed Parquet dataset from DBFS.
    
    Args:
        name (str): Dataset name (e.g., "OTPW_12M_FOLD_1_TRAIN")
        folder_path (str): DBFS path to the folder containing checkpointed data
        
    Returns:
        pyspark.sql.DataFrame: Loaded dataset
        
    Note:
        Pre-checkpointed data significantly speeds up CV by avoiding repeated
        data loading and splitting operations.
    """
    return spark.read.parquet(f"{folder_path}/{name}.parquet")


def _load_folds(n_folds=5, version="3M"):
    """
    Load all CV folds for expanding window cross-validation.
    
    Args:
        n_folds (int): Total number of folds (default: 5)
                       First n_folds-1 are train/val pairs
                       Last fold is train/test pair for final evaluation
        version (str): Dataset version ("3M", "12M", etc.)
        
    Returns:
        list of tuples: [(train_df, val_df), ..., (train_df, test_df)]
        
    Expanding Window Strategy:
        - Fold 1: Train on months 1-8, validate on month 9
        - Fold 2: Train on months 1-9, validate on month 10
        - Fold 3: Train on months 1-10, validate on month 11
        - Fold 4: Train on months 1-11, validate on month 12
        - Fold 5: Train on months 1-12, test on held-out data
        
    This mimics real-world deployment where models are retrained on
    progressively more historical data.
    """
    folds = []
    for i in range(1, n_folds + 1):
        # Load training data for this fold
        train_df = _load_checkpointed_data(f"OTPW_{version}_FOLD_{i}_TRAIN")
        
        if i != n_folds:
            # For folds 1 to n_folds-1: load validation set
            val_df = _load_checkpointed_data(f"OTPW_{version}_FOLD_{i}_VAL")
            folds.append((train_df, val_df))
        else:
            # For last fold: load held-out test set
            test_df = _load_checkpointed_data(f"OTPW_{version}_FOLD_{i}_TEST")
            folds.append((train_df, test_df))
    
    return folds


# ============================================================================
# Baseline Estimator Class
# ============================================================================

class BaselineEstimator:
    """
    Baseline Linear Regression Estimator with Feature Engineering Pipeline.
    
    This class encapsulates the entire feature engineering and modeling pipeline:
        1. Data preparation (label cleaning, feature selection)
        2. Numerical feature cleaning (remove non-numeric chars, handle nulls)
        3. Median imputation for numerical features
        4. Categorical encoding (StringIndexer + OneHotEncoder)
        5. Feature assembly and standardization
        6. Linear regression modeling
    
    Feature Families:
        - Temporal: DAY_OF_WEEK, MONTH, DEP_TIME_BLK
        - Airport: ORIGIN, DEST
        - Flight: OP_UNIQUE_CARRIER, DISTANCE
        - Weather: HourlyWindSpeed, HourlyVisibility, HourlyPrecipitation
    
    Why This Design:
        - Encapsulation: All preprocessing logic in one place
        - Reusability: Same pipeline for train/val/test
        - Spark ML compatibility: Uses Pipeline for efficient execution
    """
    
    def __init__(self, label_col="DEP_DELAY"):
        """
        Initialize the estimator with feature definitions.
        
        Args:
            label_col (str): Name of the target variable column
        """
        self.label_col = label_col
        self.pipeline = None
        self.model = None
        
        # Categorical features: Encoded using StringIndexer + OneHotEncoder
        # These capture temporal patterns, route characteristics, and carrier effects
        self.categorical_features = [
            "DAY_OF_WEEK",        # Day of week (1=Monday, 7=Sunday)
            "MONTH",              # Month of year (1-12)
            "DEP_TIME_BLK",       # Departure time block (e.g., "0600-0659")
            "ORIGIN",             # Origin airport code
            "DEST",               # Destination airport code
            "OP_UNIQUE_CARRIER"   # Operating carrier code
        ]
        
        # Numerical features: Imputed with median and standardized
        # These capture weather conditions and flight distance
        self.numerical_features = [
            "HourlyWindSpeed",       # Wind speed at origin (mph)
            "HourlyVisibility",      # Visibility at origin (miles)
            "HourlyPrecipitation",   # Precipitation at origin (inches)
            "DISTANCE"               # Flight distance (miles)
        ]

    def _prepare(self, df):
        """
        Prepare the DataFrame for modeling by cleaning the label and numerical features.
        
        Steps:
            1. Cast label to DoubleType (required by Spark ML)
            2. Filter out rows with null/NaN labels (Spark ML requirement)
            3. Select only required features + label
            4. Clean numerical features:
               - Remove non-numeric characters (e.g., "12.5mph" -> "12.5")
               - Convert empty strings to null
               - Cast to DoubleType
        
        Args:
            df (pyspark.sql.DataFrame): Input DataFrame
            
        Returns:
            pyspark.sql.DataFrame: Cleaned DataFrame
            
        Why This Matters:
            - Spark ML LinearRegression requires DoubleType labels with no nulls
            - Numerical features may contain string artifacts from source data
            - Explicit type casting prevents downstream pipeline errors
        """
        # Cast label to double and filter out null/NaN values
        # Spark ML does not accept null labels
        df = df.withColumn(self.label_col, col(self.label_col).cast(DoubleType()))
        df = df.filter(~(col(self.label_col).isNull() | isnan(col(self.label_col))))

        # Select only the columns we need (features + label)
        # This reduces memory footprint and prevents accidental data leakage
        selected = [c for c in (self.categorical_features + self.numerical_features + [self.label_col]) 
                    if c in df.columns]
        df = df.select(*selected)

        # Clean numerical features: remove non-numeric characters, handle empty strings
        for f in self.numerical_features:
            if f in df.columns:
                # Step 1: Cast to string to enable regex operations
                # Step 2: Remove all non-numeric characters except +, -, and .
                df = df.withColumn(f, regexp_replace(col(f).cast(StringType()), r"[^0-9+\-\.]", ""))
                
                # Step 3: Convert empty strings to null (for imputation)
                df = df.withColumn(f, when(length(trim(col(f))) == 0, None).otherwise(col(f)))
                
                # Step 4: Cast to DoubleType for modeling
                df = df.withColumn(f, col(f).cast(DoubleType()))
        
        return df

    def _build_pipeline(self, df):
        """
        Build the Spark ML Pipeline with all feature engineering stages.
        
        Pipeline Stages:
            1. Imputers: Median imputation for numerical features
            2. StringIndexers: Convert categorical strings to indices
            3. OneHotEncoders: Convert indices to binary vectors
            4. VectorAssembler: Combine all features into single vector
            5. StandardScaler: Standardize features (mean=0, std=1)
            6. LinearRegression: Train linear model
        
        Args:
            df (pyspark.sql.DataFrame): Prepared DataFrame
            
        Returns:
            pyspark.sql.DataFrame: DataFrame (may have additional columns from transformations)
            
        Why This Design:
            - Pipeline ensures consistent transformations across train/val/test
            - Median imputation is robust to outliers (better than mean)
            - OneHotEncoding with dropLast=True prevents multicollinearity
            - StandardScaler improves convergence for gradient descent
            - No regularization (regParam=0) for interpretable baseline
        """
        stages = []
        
        # ========================================
        # Stage 1: Median Imputation for Numerical Features
        # ========================================
        # Why median? More robust to outliers than mean
        # Missing weather data is common in aviation datasets
        imputers = [
            Imputer(inputCols=[f], outputCols=[f"{f}_imputed"], strategy="median")
            for f in self.numerical_features if f in df.columns
        ]
        stages.extend(imputers)

        # ========================================
        # Stage 2-3: Categorical Encoding (StringIndexer + OneHotEncoder)
        # ========================================
        # StringIndexer: Converts strings to numeric indices (most frequent = 0)
        # OneHotEncoder: Converts indices to binary vectors (prevents ordinal assumption)
        # handleInvalid="keep": Unseen categories in test set get their own index
        # dropLast=True: Drop last category to prevent multicollinearity
        for f in self.categorical_features:
            if f in df.columns:
                # For numeric-coded categoricals (DAY_OF_WEEK, MONTH, DEP_TIME_BLK),
                # cast to string first to ensure consistent handling
                if f in ["DAY_OF_WEEK", "MONTH", "DEP_TIME_BLK"]:
                    df = df.withColumn(f"{f}_clean", 
                                      when(col(f).isNull(), "UNKNOWN").otherwise(col(f).cast(StringType())))
                else:
                    # For string categoricals (ORIGIN, DEST, CARRIER), handle nulls only
                    df = df.withColumn(f"{f}_clean", 
                                      when(col(f).isNull(), "UNKNOWN").otherwise(col(f)))
                
                # Add StringIndexer stage
                stages.append(StringIndexer(inputCol=f"{f}_clean", 
                                           outputCol=f"{f}_indexed", 
                                           handleInvalid="keep"))
                
                # Add OneHotEncoder stage
                stages.append(OneHotEncoder(inputCols=[f"{f}_indexed"], 
                                           outputCols=[f"{f}_encoded"], 
                                           dropLast=True))

        # ========================================
        # Stage 4: Feature Assembly
        # ========================================
        # Combine all features (imputed numerical + encoded categorical) into single vector
        # handleInvalid="skip": Skip rows with invalid values (e.g., NaN after imputation)
        feature_columns = [f"{f}_imputed" for f in self.numerical_features if f in df.columns] + \
                          [f"{f}_encoded" for f in self.categorical_features if f in df.columns]
        assembler = VectorAssembler(inputCols=feature_columns, 
                                    outputCol="features", 
                                    handleInvalid="skip")
        
        # ========================================
        # Stage 5: Feature Standardization
        # ========================================
        # Standardize features to mean=0, std=1
        # withStd=True: Scale to unit variance
        # withMean=True: Center to zero mean
        # Why? Improves convergence and makes coefficients comparable
        scaler = StandardScaler(inputCol="features", 
                               outputCol="scaled_features", 
                               withStd=True, 
                               withMean=True)
        
        # ========================================
        # Stage 6: Linear Regression
        # ========================================
        # Baseline model: No regularization (regParam=0, elasticNetParam=0)
        # maxIter=100: Maximum iterations for convergence
        # Why no regularization? We want an interpretable baseline to understand
        # feature importance before adding complexity
        lr = LinearRegression(featuresCol="scaled_features", 
                            labelCol=self.label_col, 
                            maxIter=100, 
                            regParam=0.0, 
                            elasticNetParam=0.0)

        # Assemble all stages into pipeline
        stages.extend([assembler, scaler, lr])
        self.pipeline = Pipeline(stages=stages)
        
        return df

    def fit(self, df):
        """
        Fit the pipeline on training data.
        
        Args:
            df (pyspark.sql.DataFrame): Training DataFrame
            
        Returns:
            BaselineEstimator: self (for method chaining)
        """
        df_prep = self._prepare(df)
        df_prep = self._build_pipeline(df_prep)
        self.model = self.pipeline.fit(df_prep)
        return self

    def transform(self, df):
        """
        Transform data using the fitted pipeline.
        
        Args:
            df (pyspark.sql.DataFrame): DataFrame to transform (val/test)
            
        Returns:
            pyspark.sql.DataFrame: Transformed DataFrame with predictions
            
        Note:
            We rebuild the pipeline on the input DataFrame to ensure all
            transformation columns are present, but use the fitted model
            for predictions.
        """
        df_prep = self._prepare(df)
        df_prep = self._build_pipeline(df_prep)  # Rebuild to ensure cols present
        return self.model.transform(df_prep)


# ============================================================================
# Cross-Validation Runner
# ============================================================================

def run_cv(n_folds=5, version="3M", include_classification_metrics=True):
    """
    Run expanding window cross-validation for the baseline model.
    
    Process:
        1. Load pre-checkpointed folds (expanding window)
        2. For each fold (except last):
           a. Train model on training set
           b. Evaluate on both training and validation sets
           c. Compute regression and classification metrics
           d. Track the best model (by validation RMSE)
        3. Evaluate best model on held-out test set
        4. Report comprehensive metrics, feature importance, and runtime
    
    Args:
        n_folds (int): Number of folds (default 5, last fold is test)
        version (str): Dataset version ("3M", "12M", etc.)
        include_classification_metrics (bool): Whether to compute OTPA, SDDR, etc.
        
    Returns:
        tuple: (best_model, val_metrics_list, test_metrics_dict, metrics_df,
                metrics_defs, classification_df, test_class_results)
    
    Memory Management Strategy:
        - .cache().count(): Forces immediate materialization in controlled chunks
        - .unpersist(): Explicitly frees memory after each fold
        - This prevents OOM errors by avoiding lazy accumulation of cached data
    """
    # ========================================
    # Initialize Timer and Load Data
    # ========================================
    overall_start_time = time.time()
    folds = _load_folds(n_folds=n_folds, version=version)

    # ========================================
    # Initialize Evaluators
    # ========================================
    # Create evaluators for each regression metric
    # These will be reused across all folds for consistency
    eval_rmse = RegressionEvaluator(predictionCol="prediction", labelCol="DEP_DELAY", metricName="rmse")
    eval_mae  = RegressionEvaluator(predictionCol="prediction", labelCol="DEP_DELAY", metricName="mae")
    eval_r2   = RegressionEvaluator(predictionCol="prediction", labelCol="DEP_DELAY", metricName="r2")
    eval_mse  = RegressionEvaluator(predictionCol="prediction", labelCol="DEP_DELAY", metricName="mse")

    # ========================================
    # Initialize Metric Storage
    # ========================================
    metrics = []  # Validation metrics per fold (for backward compatibility)
    train_records = []  # Per-fold train metrics
    val_records = []    # Per-fold validation metrics
    classification_records = []  # Classification metrics per fold
    best_model = None  # Best model (by validation RMSE)
    best_rmse = float("inf")  # Track best validation RMSE

    # ========================================
    # Initialize Estimator
    # ========================================
    est = BaselineEstimator(label_col="DEP_DELAY")

    # ========================================
    # Cross-Validation Loop
    # ========================================
    # Train/validate on first n_folds-1; last fold used as held-out test
    # folds[:-1] excludes the last fold (which is train/test pair)
    for idx, (train_df, val_df) in enumerate(folds[:-1], start=1):
        print(f"--- Fold {idx}/{n_folds - 1} ---")
        
        # ========================================
        # Cache DataFrames for Performance
        # ========================================
        # .cache().count() forces immediate materialization
        # This prevents memory accumulation by materializing in controlled chunks
        # The paradox: .count() seems expensive but actually prevents OOM!
        train_df.cache().count()
        val_df.cache().count()
        
        # ========================================
        # Train Model
        # ========================================
        model = est.fit(train_df)
        
        # ========================================
        # Generate Predictions
        # ========================================
        # Generate predictions on both validation and training sets
        # Training metrics help detect overfitting
        
        # Validation predictions
        val_preds = model.transform(val_df)
        val_preds.cache().count()  # Materialize to prevent recomputation
        
        # Train predictions
        train_preds = model.transform(train_df)
        train_preds.cache().count()  # Materialize to prevent recomputation

        # ========================================
        # Compute Regression Metrics (Validation)
        # ========================================
        val_rmse = eval_rmse.evaluate(val_preds)
        val_mae  = eval_mae.evaluate(val_preds)
        val_r2   = eval_r2.evaluate(val_preds)
        val_mse  = eval_mse.evaluate(val_preds)
        
        # Store validation metrics
        metrics.append({"fold": idx, "rmse": val_rmse, "mae": val_mae, "r2": val_r2, "mse": val_mse})
        val_records.append({"split": "validation", "fold": idx, "rmse": val_rmse, "mae": val_mae, "r2": val_r2, "mse": val_mse})

        # ========================================
        # Compute Regression Metrics (Training)
        # ========================================
        # Training metrics help detect overfitting
        # If train metrics >> val metrics, model is overfitting
        tr_rmse = eval_rmse.evaluate(train_preds)
        tr_mae  = eval_mae.evaluate(train_preds)
        tr_r2   = eval_r2.evaluate(train_preds)
        tr_mse  = eval_mse.evaluate(train_preds)
        train_records.append({"split": "train", "fold": idx, "rmse": tr_rmse, "mae": tr_mae, "r2": tr_r2, "mse": tr_mse})

        # ========================================
        # Track Best Model
        # ========================================
        # Select model with lowest validation RMSE
        # This model will be used for final test evaluation
        if val_rmse < best_rmse:
            best_rmse = val_rmse
            best_model = model

        # Print validation metrics
        print(f"RMSE: {val_rmse:.2f}  MAE: {val_mae:.2f}  R²: {val_r2:.4f}  MSE: {val_mse:.2f}")
        
        # ========================================
        # Compute Classification Metrics (Optional)
        # ========================================
        # Classification metrics provide operational insights:
        # - OTPA: On-Time Performance Accuracy (<15 min threshold)
        # - SDDR: Severe Delay Detection Rate (≥60 min threshold)
        # - Bucket Accuracy: 4-bucket classification (Early, OnTime, Delayed, Severe)
        if include_classification_metrics:
            val_class_results, _ = compute_classification_metrics(val_preds)
            classification_records.append({
                "split": "validation",
                "fold": idx,
                "otpa_accuracy": val_class_results["otpa_accuracy"],
                "otpa_f1": val_class_results["otpa_f1"],
                "sddr_recall": val_class_results["sddr_recall"],
                "bucket_accuracy": val_class_results["bucket_accuracy"]
            })
            print(f"  OTPA Acc: {val_class_results['otpa_accuracy']:.4f}  "
                  f"OTPA F1: {val_class_results['otpa_f1']:.4f}  "
                  f"SDDR: {val_class_results['sddr_recall']:.4f}")
        
        # ========================================
        # Free Memory
        # ========================================
        # Explicitly unpersist cached DataFrames to free memory
        # Critical for preventing OOM in subsequent folds
        train_preds.unpersist()
        val_preds.unpersist()
        train_df.unpersist()
        val_df.unpersist()

    # ========================================
    # Validation Summary
    # ========================================
    # Report aggregate statistics across all validation folds
    # This helps assess model stability and generalization
    print("-------------------------------")
    print("Validation Summary:")
    print(f"Best RMSE: {best_rmse:.2f}")
    print(f"Mean RMSE: {np.mean([m['rmse'] for m in metrics]):.2f}  |  Std: {np.std([m['rmse'] for m in metrics]):.2f}")
    print(f"Mean MAE:  {np.mean([m['mae'] for m in metrics]):.2f}")
    print(f"Mean R²:   {np.mean([m['r2'] for m in metrics]):.4f}")
    print(f"Mean MSE:  {np.mean([m['mse'] for m in metrics]):.2f}")

    # ========================================
    # Held-Out Test Evaluation
    # ========================================
    # Evaluate best model on held-out test set (last fold)
    # This provides an unbiased estimate of production performance
    train_df, test_df = folds[-1]
    test_df.cache().count()  # Cache and materialize test data
    
    # Generate predictions on test set
    test_preds = best_model.transform(test_df)
    test_preds.cache().count()  # Cache and materialize predictions
    
    # Compute regression metrics on test set
    test_rmse = eval_rmse.evaluate(test_preds)
    test_mae  = eval_mae.evaluate(test_preds)
    test_r2   = eval_r2.evaluate(test_preds)
    test_mse  = eval_mse.evaluate(test_preds)
    
    # Free memory
    test_preds.unpersist()
    test_df.unpersist()

    # Print test results
    print("-------------------------------")
    print("Held-out Test:")
    print(f"RMSE: {test_rmse:.2f}  MAE: {test_mae:.2f}  R²: {test_r2:.4f}  MSE: {test_mse:.2f}")
    
    # ========================================
    # Test Set Classification Metrics
    # ========================================
    # Compute classification metrics for test set if requested
    # Provides operational insights for deployment decisions
    if include_classification_metrics:
        test_class_results, test_class_df = compute_classification_metrics(test_preds)
        print_classification_summary(test_class_results, test_class_df)
        classification_records.append({
            "split": "test",
            "fold": n_folds,
            "otpa_accuracy": test_class_results["otpa_accuracy"],
            "otpa_f1": test_class_results["otpa_f1"],
            "sddr_recall": test_class_results["sddr_recall"],
            "bucket_accuracy": test_class_results["bucket_accuracy"]
        })

    # ========================================
    # Build Consolidated Metrics DataFrames
    # ========================================
    # Combine train/val/test metrics into single DataFrame for easy comparison
    test_record = [{"split": "test", "fold": n_folds, "rmse": test_rmse, "mae": test_mae, "r2": test_r2, "mse": test_mse}]
    metrics_df = pd.DataFrame(train_records + val_records + test_record)
    
    # Build classification metrics DataFrame if computed
    classification_df = pd.DataFrame(classification_records) if include_classification_metrics else None
    
    # Round to 4 decimal places for readability
    metrics_df['rmse'] = metrics_df['rmse'].round(4)
    metrics_df['mae'] = metrics_df['mae'].round(4)
    metrics_df['r2'] = metrics_df['r2'].round(4)
    metrics_df['mse'] = metrics_df['mse'].round(4)

    # ========================================
    # Metrics Definitions with LaTeX
    # ========================================
    # Provide LaTeX equations for each metric for documentation/reporting
    metrics_defs = pd.DataFrame([
        {"metric": "RMSE", "latex": r"\\mathrm{RMSE} = \\sqrt{\\tfrac{1}{N} \\sum_{i=1}^{N} (y_i - \\hat{y}_i)^2}"},
        {"metric": "MAE",  "latex": r"\\mathrm{MAE} = \\tfrac{1}{N} \\sum_{i=1}^{N} |y_i - \\hat{y}_i|"},
        {"metric": "R^2",  "latex": r"R^2 = 1 - \\dfrac{\\sum_{i=1}^{N} (y_i - \\hat{y}_i)^2}{\\sum_{i=1}^{N} (y_i - \\bar{y})^2}"},
        {"metric": "MSE",  "latex": r"\\mathrm{MSE} = \\tfrac{1}{N} \\sum_{i=1}^{N} (y_i - \\hat{y}_i)^2"},
    ])

    # ========================================
    # Extract Feature Importance (Coefficients)
    # ========================================
    # Extract and display top 10 most important features by absolute coefficient
    # Helps understand which features drive predictions
    try:
        # Extract coefficients from the linear regression model (last stage in pipeline)
        coefficients = best_model.model.stages[-1].coefficients.toArray()
        
        # Try to extract feature names from metadata (includes OHE expansions)
        feats_meta = test_preds.schema["features"].metadata
        attrs = []
        if "ml_attr" in feats_meta and "attrs" in feats_meta["ml_attr"]:
            ml_attrs = feats_meta["ml_attr"]["attrs"]
            # Collect attributes from all types (binary, numeric, nominal)
            for t in ["binary", "numeric", "nominal"]:
                if t in ml_attrs:
                    attrs.extend(ml_attrs[t])
            # Sort by vector index to match coefficient order
            attrs = sorted(attrs, key=lambda x: x["idx"])
            feature_names = [a.get("name", f"feature_{a['idx']}") for a in attrs]
        else:
            # Fallback: use generic names if metadata unavailable
            feature_names = [f"feature_{i}" for i in range(len(coefficients))]
        
        # Safety check: ensure lengths match
        if len(feature_names) != len(coefficients):
            feature_names = [f"feature_{i}" for i in range(len(coefficients))]
        
        # Create DataFrame with feature names and coefficients
        coef_df = pd.DataFrame({"feature": feature_names, "coefficient": coefficients})
        coef_df["abs_coefficient"] = coef_df["coefficient"].abs()
        coef_df = coef_df.sort_values("abs_coefficient", ascending=False)
        
        # Round coefficients to 4 decimal places
        top_coef_df = coef_df.head(10).copy()
        top_coef_df['coefficient'] = top_coef_df['coefficient'].round(4)
        top_coef_df['abs_coefficient'] = top_coef_df['abs_coefficient'].round(4)
        
        print("\nTop 10 Most Important Features (by absolute coefficient):")
        print(top_coef_df.to_string(index=False))
    except Exception as e:
        print(f"[Info] Skipping coefficient listing: {e}")
    
    # ========================================
    # Print Total Runtime
    # ========================================
    # Report total execution time for performance tracking
    overall_elapsed = time.time() - overall_start_time
    hours, remainder = divmod(overall_elapsed, 3600)
    minutes, seconds = divmod(remainder, 60)
    print("\n" + "="*80)
    print(f"TOTAL RUNTIME: {int(hours)}h {int(minutes)}m {seconds:.2f}s ({overall_elapsed:.2f} seconds)")
    print("="*80)

    # ========================================
    # Return Results
    # ========================================
    return best_model, metrics, {"rmse": test_rmse, "mae": test_mae, "r2": test_r2, "mse": test_mse}, metrics_df, metrics_defs, classification_df, test_class_results if include_classification_metrics else None


# ============================================================================
# Main Execution Block
# ============================================================================

if __name__ == "__main__":
    """
    Main execution entry point for the cross-validation script.
    
    Workflow:
        1. Print cluster configuration (for reproducibility and debugging)
        2. Run expanding window cross-validation
        3. Display comprehensive results (regression + classification metrics)
    
    Note:
        This block only runs when the script is executed directly,
        not when imported as a module.
    """
    
    # ========================================
    # Print Cluster Configuration
    # ========================================
    # Display cluster resources to ensure adequate capacity and enable reproducibility
    # Important for:
    # - Debugging OOM errors (check if cluster is undersized)
    # - Reproducing results (document exact cluster configuration)
    # - Cost tracking (understand resource usage)
    
    sc = spark.sparkContext
    
    # Get number of executors (exclude driver node)
    # statusTracker().getExecutorInfos() returns list of all executors
    num_executors = len(sc._jsc.sc().statusTracker().getExecutorInfos()) - 1
    
    # Get executor configuration from Spark conf
    executor_memory = sc.getConf().get("spark.executor.memory", "Unknown")
    executor_cores = sc.getConf().get("spark.executor.cores", "Unknown")
    actual_cores = sc.defaultParallelism / num_executors if num_executors > 0 else "Unknown"
    print(f"Estimated cores per executor: {int(actual_cores)}")
    
    # Print cluster configuration
    print("\n" + "="*80)
    print("CLUSTER CONFIGURATION")
    print("="*80)
    print(f"Cluster Size: {num_executors} executors, {executor_cores} cores per executor, {executor_memory} RAM per executor")
    print("="*80 + "\n")
    
    # ========================================
    # Run Cross-Validation
    # ========================================
    # Execute expanding window CV with comprehensive metrics
    # Parameters:
    # - n_folds=5: 4 validation folds + 1 held-out test fold
    # - version="3M": Use 3-month dataset
    # - include_classification_metrics=True: Compute OTPA, SDDR, bucket accuracy
    
    best_model, val_metrics_list, test_metrics_dict, metrics_df, metrics_defs, classification_df, test_class_results = run_cv(
        n_folds=5, 
        version="3M",
        include_classification_metrics=True
    )
    
    # ========================================
    # Display Regression Metrics
    # ========================================
    # Show comprehensive regression metrics across train/val/test splits
    # metrics_df: Per-fold metrics (split, fold, RMSE, MAE, R², MSE)
    # metrics_defs: LaTeX equations for each metric (for documentation)
    
    print("\n" + "="*80)
    print("REGRESSION METRICS")
    print("="*80)
    display(metrics_df)  # Use display() for interactive Databricks table
    display(metrics_defs)
    
    # ========================================
    # Display Classification Metrics
    # ========================================
    # Show operational classification metrics if computed
    # Includes OTPA (on-time accuracy), SDDR (severe delay detection),
    # and 4-bucket classification accuracy
    
    if classification_df is not None:
        print("\n" + "="*80)
        print("CLASSIFICATION METRICS SUMMARY")
        print("="*80)
        display(classification_df)  # Use display() for interactive Databricks table

In [0]:
displayHTML("""
<!DOCTYPE html>
<html>
<head>
  <script src="https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.min.js"></script>
  <script>
    mermaid.initialize({
      startOnLoad: true,
      theme: 'dark',
      themeVariables: {
        primaryColor: '#4a5568',
        primaryTextColor: '#fff',
        primaryBorderColor: '#cbd5e0',
        lineColor: '#cbd5e0',
        secondaryColor: '#2d3748',
        tertiaryColor: '#1a202c',
        background: '#1a202c',
        mainBkg: '#4a5568',
        secondBkg: '#2d3748',
        tertiaryBkg: '#1a202c'
      }
    });
  </script>
  <style>
    body { background-color: #1a202c; }
    .mermaid { background-color: #1a202c; }
  </style>
</head>
<body>
<div class="mermaid">
flowchart LR

    %% =========================
    %% Inputs (Pre-checkpointed Folds)
    %% =========================
    Folds["<b>Input</b><br/>Pre-checkpointed 3M Folds<br/>OTPW_3M_FOLD_i_{TRAIN/VAL/TEST}.parquet"]

    %% =========================
    %% Baseline Estimator Pipeline (per fold)
    %% =========================
    subgraph PIPE["<b>Baseline Estimator: Feature Pipeline</b>"]
        direction LR

        %% Stage 1: Data Preparation
        subgraph S1["Stage 1: Data Preparation"]
            LabelClean["Cast DEP_DELAY to Double<br/>Filter Null/NaN Labels"]
            SelectFeat["Select 10 Baseline Features<br/>Temporal, Airport, Flight, Weather"]
            NumClean["Clean Numerical Features:<br/>Remove non-numeric chars<br/>Empty → Null<br/>Cast to Double"]
            LabelClean --> SelectFeat --> NumClean
        end

        %% Stage 2: Numerical Imputation (Median)
        subgraph S2["Stage 2: Numerical Imputation (Median)"]
            ImpWind["HourlyWindSpeed_imputed<br/>median(HourlyWindSpeed)"]
            ImpVis["HourlyVisibility_imputed<br/>median(HourlyVisibility)"]
            ImpPrec["HourlyPrecipitation_imputed<br/>median(HourlyPrecipitation)"]
            ImpDist["DISTANCE_imputed<br/>median(DISTANCE)"]
        end

        %% Stage 3: Categorical Encoding
        subgraph S3["Stage 3: Categorical Encoding"]
            DOW["DAY_OF_WEEK_clean<br/>StringIndexer + OHE"]
            Month["MONTH_clean<br/>StringIndexer + OHE"]
            TimeBlk["DEP_TIME_BLK_clean<br/>StringIndexer + OHE"]
            Origin["ORIGIN_clean<br/>StringIndexer + OHE"]
            Dest["DEST_clean<br/>StringIndexer + OHE"]
            Carrier["OP_UNIQUE_CARRIER_clean<br/>StringIndexer + OHE"]
        end

        %% Stage 4: Feature Assembly
        subgraph S4["Stage 4: Feature Assembly"]
            Assemble["VectorAssembler<br/>Combine Imputed Numerics + Encoded Categoricals"]
        end

        %% Stage 5: Standardization
        subgraph S5["Stage 5: Standardization"]
            Scale["StandardScaler<br/>features → scaled_features<br/>withMean=True, withStd=True"]
        end

        %% Stage 6: Linear Regression
        subgraph S6["Stage 6: Model"]
            LR["Linear Regression<br/>featuresCol=scaled_features<br/>labelCol=DEP_DELAY<br/>maxIter=100<br/>regParam=0.0, elasticNet=0.0"]
        end
    end

    %% Connections inside pipeline
    S1 --> S2
    S1 --> S3
    S2 --> Assemble
    S3 --> Assemble
    Assemble --> Scale --> LR

    %% =========================
    %% Expanding Window CV Flow
    %% =========================
    subgraph CV["<b>Expanding Window Cross-Validation</b>"]
        direction TB
        Fold1["Fold 1:<br/>Train: Months 1–8<br/>Val: Month 9"]
        Fold2["Fold 2:<br/>Train: Months 1–9<br/>Val: Month 10"]
        Fold3["Fold 3:<br/>Train: Months 1–10<br/>Val: Month 11"]
        Fold4["Fold 4:<br/>Train: Months 1–11<br/>Val: Month 12"]
        Fold5["Fold 5:<br/>Train: Months 1–12<br/>Test: Held-out"]

        Fold1 --> Fold2 --> Fold3 --> Fold4 --> Fold5
    end

    %% =========================
    %% Metrics & Output
    %% =========================
    subgraph METRICS["<b>Metrics & Outputs</b>"]
        direction TB
        RegMetrics["Regression Metrics per Fold<br/>RMSE, MAE, R², MSE<br/>Train & Validation"]
        ClassMetrics["Classification Metrics per Fold<br/>OTPA Accuracy, OTPA F1<br/>SDDR Recall, Bucket Accuracy"]
        TestEval["Held-out Test Evaluation<br/>Best Model by Val RMSE<br/>Final Test RMSE/MAE/R²/MSE<br/>+ Test Classification Metrics"]
    end

    %% Global connections
    Folds --> PIPE
    PIPE --> CV
    CV --> METRICS
</div>
</body>
</html>
""")

# Experiment 2: Model 3

# Modeling Pipeline Visualization (Cross Validation Baseline with Data Imputatuion)

## Experiment 2: Model 3
Linear regression, with cross validation and null imputation on the 1 year data set, calculating linear regression and categorical metrics

# Model 3 - Cross-Validation/12-month/Data-Imputed/Default-Data
Updated to reference 4 bucket classification metrics and print cluster info

In [0]:
displayHTML("""
<!DOCTYPE html>
<html>
<head>
  <script src="https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.min.js"></script>
  <script>
    mermaid.initialize({
      startOnLoad: true,
      theme: 'dark',
      themeVariables: {
        primaryColor: '#4a5568',
        primaryTextColor: '#fff',
        primaryBorderColor: '#cbd5e0',
        lineColor: '#cbd5e0',
        secondaryColor: '#2d3748',
        tertiaryColor: '#1a202c',
        background: '#1a202c',
        mainBkg: '#4a5568',
        secondBkg: '#2d3748',
        tertiaryBkg: '#1a202c'
      }
    });
  </script>
  <style>
    body { background-color: #1a202c; }
    .mermaid { background-color: #1a202c; }
  </style>
</head>
<body>
<div class="mermaid">
flowchart LR

    %% =========================
    %% Inputs (Pre-checkpointed 12M Folds)
    %% =========================
    Folds["<b>Input</b><br/>Pre-checkpointed 12M Folds<br/>OTPW_12M_FOLD_i_{TRAIN/VAL/TEST}.parquet"]

    %% =========================
    %% Baseline Estimator Pipeline (per fold)
    %% =========================
    subgraph PIPE["<b>Baseline Estimator: Feature Pipeline</b>"]
        direction LR

        %% Stage 1: Data Preparation
        subgraph S1["Stage 1: Data Preparation"]
            LabelClean["Cast DEP_DELAY to Double<br/>Filter Null/NaN Labels"]
            SelectFeat["Select 10 Baseline Features<br/>Temporal, Airport, Flight, Weather"]
            NumClean["Clean Numerical Features:<br/>Remove non-numeric chars<br/>Empty → Null<br/>Cast to Double"]
            LabelClean --> SelectFeat --> NumClean
        end

        %% Stage 2: Numerical Imputation (Median)
        subgraph S2["Stage 2: Numerical Imputation (Median)"]
            ImpWind["HourlyWindSpeed_imputed<br/>median(HourlyWindSpeed)"]
            ImpVis["HourlyVisibility_imputed<br/>median(HourlyVisibility)"]
            ImpPrec["HourlyPrecipitation_imputed<br/>median(HourlyPrecipitation)"]
            ImpDist["DISTANCE_imputed<br/>median(DISTANCE)"]
        end

        %% Stage 3: Categorical Encoding
        subgraph S3["Stage 3: Categorical Encoding"]
            DOW["DAY_OF_WEEK_clean<br/>StringIndexer + OHE"]
            Month["MONTH_clean<br/>StringIndexer + OHE"]
            TimeBlk["DEP_TIME_BLK_clean<br/>StringIndexer + OHE"]
            Origin["ORIGIN_clean<br/>StringIndexer + OHE"]
            Dest["DEST_clean<br/>StringIndexer + OHE"]
            Carrier["OP_UNIQUE_CARRIER_clean<br/>StringIndexer + OHE"]
        end

        %% Stage 4: Feature Assembly
        subgraph S4["Stage 4: Feature Assembly"]
            Assemble["VectorAssembler<br/>Combine Imputed Numerics + Encoded Categoricals"]
        end

        %% Stage 5: Standardization
        subgraph S5["Stage 5: Standardization"]
            Scale["StandardScaler<br/>features → scaled_features<br/>withMean=True, withStd=True"]
        end

        %% Stage 6: Linear Regression
        subgraph S6["Stage 6: Model"]
            LR["Linear Regression<br/>featuresCol=scaled_features<br/>labelCol=DEP_DELAY<br/>maxIter=100<br/>regParam=0.0, elasticNet=0.0"]
        end
    end

    %% Connections inside pipeline
    S1 --> S2
    S1 --> S3
    S2 --> Assemble
    S3 --> Assemble
    Assemble --> Scale --> LR

    %% =========================
    %% Expanding Window CV Flow
    %% =========================
    subgraph CV["<b>Expanding Window Cross-Validation (12 Months)</b>"]
        direction TB
        Fold1["Fold 1:<br/>Train: Months 1–8<br/>Val: Month 9"]
        Fold2["Fold 2:<br/>Train: Months 1–9<br/>Val: Month 10"]
        Fold3["Fold 3:<br/>Train: Months 1–10<br/>Val: Month 11"]
        Fold4["Fold 4:<br/>Train: Months 1–11<br/>Val: Month 12"]
        Fold5["Fold 5:<br/>Train: Months 1–12<br/>Test: Held-out"]

        Fold1 --> Fold2 --> Fold3 --> Fold4 --> Fold5
    end

    %% =========================
    %% Metrics & Output
    %% =========================
    subgraph METRICS["<b>Metrics & Outputs</b>"]
        direction TB
        RegMetrics["Regression Metrics per Fold<br/>RMSE, MAE, R², MSE<br/>Train & Validation"]
        ClassMetrics["Classification Metrics per Fold<br/>OTPA Accuracy, OTPA F1<br/>SDDR Recall, Bucket Accuracy"]
        TestEval["Held-out Test Evaluation<br/>Best Model by Val RMSE<br/>Final Test RMSE/MAE/R²/MSE<br/>+ Test Classification Metrics"]
    end

    %% Global connections
    Folds --> PIPE
    PIPE --> CV
    CV --> METRICS

</div>
</body>
</html>
""")

In [0]:
"""
================================================================================
Cross-Validation for Baseline Linear Regression (12-Month Dataset)
================================================================================

Purpose:
    Implements expanding window cross-validation for flight delay prediction
    using a baseline linear regression model with comprehensive feature engineering.

Key Features:
    - Expanding window CV: Each fold trains on progressively more historical data
    - Pre-checkpointed data: Uses coworker's fold pattern (OTPW_{version}_FOLD_{i}_{TRAIN/VAL/TEST})
    - Baseline pipeline: Numeric cleaning + median imputation + categorical OHE + standardization + LR
    - Comprehensive metrics: Regression (RMSE, MAE, R², MSE) + Classification (OTPA, SDDR, 4-bucket)
    - Memory management: Strategic caching with .count() to prevent OOM errors

Data Flow:
    1. Load pre-checkpointed folds from DBFS
    2. For each fold (except last):
       - Train model on expanding training set
       - Evaluate on validation set
       - Track best model by RMSE
    3. Evaluate best model on held-out test set
    4. Report comprehensive metrics and feature importance

Author: Emily Lieske
Date: November 2025
================================================================================
"""

# ============================================================================
# Imports
# ============================================================================

# PySpark SQL functions for data transformation
from pyspark.sql.functions import col, when, isnan, regexp_replace, trim, length
from pyspark.sql.types import DoubleType, StringType

# PySpark ML for pipeline construction and modeling
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder, StandardScaler, Imputer
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

# Standard libraries for metrics and timing
import numpy as np
import pandas as pd
import time


# ============================================================================
# Data Loading Functions
# ============================================================================

def _load_checkpointed_data(name, folder_path="dbfs:/student-groups/Group_4_2"):
    """
    Load a pre-checkpointed Parquet dataset from DBFS.
    
    Args:
        name (str): Dataset name (e.g., "OTPW_12M_FOLD_1_TRAIN")
        folder_path (str): DBFS path to the folder containing checkpointed data
        
    Returns:
        pyspark.sql.DataFrame: Loaded dataset
        
    Note:
        Pre-checkpointed data significantly speeds up CV by avoiding repeated
        data loading and splitting operations.
    """
    return spark.read.parquet(f"{folder_path}/{name}.parquet")


def _load_folds(n_folds=5, version="12M"):
    """
    Load all CV folds for expanding window cross-validation.
    
    Args:
        n_folds (int): Total number of folds (default: 5)
                       First n_folds-1 are train/val pairs
                       Last fold is train/test pair for final evaluation
        version (str): Dataset version ("3M", "12M", etc.)
        
    Returns:
        list of tuples: [(train_df, val_df), ..., (train_df, test_df)]
        
    Expanding Window Strategy:
        - Fold 1: Train on months 1-8, validate on month 9
        - Fold 2: Train on months 1-9, validate on month 10
        - Fold 3: Train on months 1-10, validate on month 11
        - Fold 4: Train on months 1-11, validate on month 12
        - Fold 5: Train on months 1-12, test on held-out data
        
    This mimics real-world deployment where models are retrained on
    progressively more historical data.
    """
    folds = []
    for i in range(1, n_folds + 1):
        # Load training data for this fold
        train_df = _load_checkpointed_data(f"OTPW_{version}_FOLD_{i}_TRAIN")
        
        if i != n_folds:
            # For folds 1 to n_folds-1: load validation set
            val_df = _load_checkpointed_data(f"OTPW_{version}_FOLD_{i}_VAL")
            folds.append((train_df, val_df))
        else:
            # For last fold: load held-out test set
            test_df = _load_checkpointed_data(f"OTPW_{version}_FOLD_{i}_TEST")
            folds.append((train_df, test_df))
    
    return folds


# ============================================================================
# Baseline Estimator Class
# ============================================================================

class BaselineEstimator:
    """
    Baseline Linear Regression Estimator with Feature Engineering Pipeline.
    
    This class encapsulates the entire feature engineering and modeling pipeline:
        1. Data preparation (label cleaning, feature selection)
        2. Numerical feature cleaning (remove non-numeric chars, handle nulls)
        3. Median imputation for numerical features
        4. Categorical encoding (StringIndexer + OneHotEncoder)
        5. Feature assembly and standardization
        6. Linear regression modeling
    
    Feature Families:
        - Temporal: DAY_OF_WEEK, MONTH, DEP_TIME_BLK
        - Airport: ORIGIN, DEST
        - Flight: OP_UNIQUE_CARRIER, DISTANCE
        - Weather: HourlyWindSpeed, HourlyVisibility, HourlyPrecipitation
    
    Why This Design:
        - Encapsulation: All preprocessing logic in one place
        - Reusability: Same pipeline for train/val/test
        - Spark ML compatibility: Uses Pipeline for efficient execution
    """
    
    def __init__(self, label_col="DEP_DELAY"):
        """
        Initialize the estimator with feature definitions.
        
        Args:
            label_col (str): Name of the target variable column
        """
        self.label_col = label_col
        self.pipeline = None
        self.model = None
        
        # Categorical features: Encoded using StringIndexer + OneHotEncoder
        # These capture temporal patterns, route characteristics, and carrier effects
        self.categorical_features = [
            "DAY_OF_WEEK",        # Day of week (1=Monday, 7=Sunday)
            "MONTH",              # Month of year (1-12)
            "DEP_TIME_BLK",       # Departure time block (e.g., "0600-0659")
            "ORIGIN",             # Origin airport code
            "DEST",               # Destination airport code
            "OP_UNIQUE_CARRIER"   # Operating carrier code
        ]
        
        # Numerical features: Imputed with median and standardized
        # These capture weather conditions and flight distance
        self.numerical_features = [
            "HourlyWindSpeed",       # Wind speed at origin (mph)
            "HourlyVisibility",      # Visibility at origin (miles)
            "HourlyPrecipitation",   # Precipitation at origin (inches)
            "DISTANCE"               # Flight distance (miles)
        ]

    def _prepare(self, df):
        """
        Prepare the DataFrame for modeling by cleaning the label and numerical features.
        
        Steps:
            1. Cast label to DoubleType (required by Spark ML)
            2. Filter out rows with null/NaN labels (Spark ML requirement)
            3. Select only required features + label
            4. Clean numerical features:
               - Remove non-numeric characters (e.g., "12.5mph" -> "12.5")
               - Convert empty strings to null
               - Cast to DoubleType
        
        Args:
            df (pyspark.sql.DataFrame): Input DataFrame
            
        Returns:
            pyspark.sql.DataFrame: Cleaned DataFrame
            
        Why This Matters:
            - Spark ML LinearRegression requires DoubleType labels with no nulls
            - Numerical features may contain string artifacts from source data
            - Explicit type casting prevents downstream pipeline errors
        """
        # Cast label to double and filter out null/NaN values
        # Spark ML does not accept null labels
        df = df.withColumn(self.label_col, col(self.label_col).cast(DoubleType()))
        df = df.filter(~(col(self.label_col).isNull() | isnan(col(self.label_col))))

        # Select only the columns we need (features + label)
        # This reduces memory footprint and prevents accidental data leakage
        selected = [c for c in (self.categorical_features + self.numerical_features + [self.label_col]) 
                    if c in df.columns]
        df = df.select(*selected)

        # Clean numerical features: remove non-numeric characters, handle empty strings
        for f in self.numerical_features:
            if f in df.columns:
                # Step 1: Cast to string to enable regex operations
                # Step 2: Remove all non-numeric characters except +, -, and .
                df = df.withColumn(f, regexp_replace(col(f).cast(StringType()), r"[^0-9+\-\.]", ""))
                
                # Step 3: Convert empty strings to null (for imputation)
                df = df.withColumn(f, when(length(trim(col(f))) == 0, None).otherwise(col(f)))
                
                # Step 4: Cast to DoubleType for modeling
                df = df.withColumn(f, col(f).cast(DoubleType()))
        
        return df

    def _build_pipeline(self, df):
        """
        Build the Spark ML Pipeline with all feature engineering stages.
        
        Pipeline Stages:
            1. Imputers: Median imputation for numerical features
            2. StringIndexers: Convert categorical strings to indices
            3. OneHotEncoders: Convert indices to binary vectors
            4. VectorAssembler: Combine all features into single vector
            5. StandardScaler: Standardize features (mean=0, std=1)
            6. LinearRegression: Train linear model
        
        Args:
            df (pyspark.sql.DataFrame): Prepared DataFrame
            
        Returns:
            pyspark.sql.DataFrame: DataFrame (may have additional columns from transformations)
            
        Why This Design:
            - Pipeline ensures consistent transformations across train/val/test
            - Median imputation is robust to outliers (better than mean)
            - OneHotEncoding with dropLast=True prevents multicollinearity
            - StandardScaler improves convergence for gradient descent
            - No regularization (regParam=0) for interpretable baseline
        """
        stages = []
        
        # ========================================
        # Stage 1: Median Imputation for Numerical Features
        # ========================================
        # Why median? More robust to outliers than mean
        # Missing weather data is common in aviation datasets
        imputers = [
            Imputer(inputCols=[f], outputCols=[f"{f}_imputed"], strategy="median")
            for f in self.numerical_features if f in df.columns
        ]
        stages.extend(imputers)

        # ========================================
        # Stage 2-3: Categorical Encoding (StringIndexer + OneHotEncoder)
        # ========================================
        # StringIndexer: Converts strings to numeric indices (most frequent = 0)
        # OneHotEncoder: Converts indices to binary vectors (prevents ordinal assumption)
        # handleInvalid="keep": Unseen categories in test set get their own index
        # dropLast=True: Drop last category to prevent multicollinearity
        for f in self.categorical_features:
            if f in df.columns:
                # For numeric-coded categoricals (DAY_OF_WEEK, MONTH, DEP_TIME_BLK),
                # cast to string first to ensure consistent handling
                if f in ["DAY_OF_WEEK", "MONTH", "DEP_TIME_BLK"]:
                    df = df.withColumn(f"{f}_clean", 
                                      when(col(f).isNull(), "UNKNOWN").otherwise(col(f).cast(StringType())))
                else:
                    # For string categoricals (ORIGIN, DEST, CARRIER), handle nulls only
                    df = df.withColumn(f"{f}_clean", 
                                      when(col(f).isNull(), "UNKNOWN").otherwise(col(f)))
                
                # Add StringIndexer stage
                stages.append(StringIndexer(inputCol=f"{f}_clean", 
                                           outputCol=f"{f}_indexed", 
                                           handleInvalid="keep"))
                
                # Add OneHotEncoder stage
                stages.append(OneHotEncoder(inputCols=[f"{f}_indexed"], 
                                           outputCols=[f"{f}_encoded"], 
                                           dropLast=True))

        # ========================================
        # Stage 4: Feature Assembly
        # ========================================
        # Combine all features (imputed numerical + encoded categorical) into single vector
        # handleInvalid="skip": Skip rows with invalid values (e.g., NaN after imputation)
        feature_columns = [f"{f}_imputed" for f in self.numerical_features if f in df.columns] + \
                          [f"{f}_encoded" for f in self.categorical_features if f in df.columns]
        assembler = VectorAssembler(inputCols=feature_columns, 
                                    outputCol="features", 
                                    handleInvalid="skip")
        
        # ========================================
        # Stage 5: Feature Standardization
        # ========================================
        # Standardize features to mean=0, std=1
        # withStd=True: Scale to unit variance
        # withMean=True: Center to zero mean
        # Why? Improves convergence and makes coefficients comparable
        scaler = StandardScaler(inputCol="features", 
                               outputCol="scaled_features", 
                               withStd=True, 
                               withMean=True)
        
        # ========================================
        # Stage 6: Linear Regression
        # ========================================
        # Baseline model: No regularization (regParam=0, elasticNetParam=0)
        # maxIter=100: Maximum iterations for convergence
        # Why no regularization? We want an interpretable baseline to understand
        # feature importance before adding complexity
        lr = LinearRegression(featuresCol="scaled_features", 
                            labelCol=self.label_col, 
                            maxIter=100, 
                            regParam=0.0, 
                            elasticNetParam=0.0)

        # Assemble all stages into pipeline
        stages.extend([assembler, scaler, lr])
        self.pipeline = Pipeline(stages=stages)
        
        return df

    def fit(self, df):
        """
        Fit the pipeline on training data.
        
        Args:
            df (pyspark.sql.DataFrame): Training DataFrame
            
        Returns:
            BaselineEstimator: self (for method chaining)
        """
        df_prep = self._prepare(df)
        df_prep = self._build_pipeline(df_prep)
        self.model = self.pipeline.fit(df_prep)
        return self

    def transform(self, df):
        """
        Transform data using the fitted pipeline.
        
        Args:
            df (pyspark.sql.DataFrame): DataFrame to transform (val/test)
            
        Returns:
            pyspark.sql.DataFrame: Transformed DataFrame with predictions
            
        Note:
            We rebuild the pipeline on the input DataFrame to ensure all
            transformation columns are present, but use the fitted model
            for predictions.
        """
        df_prep = self._prepare(df)
        df_prep = self._build_pipeline(df_prep)  # Rebuild to ensure cols present
        return self.model.transform(df_prep)


# ============================================================================
# Cross-Validation Runner
# ============================================================================

def run_cv(n_folds=5, version="12M", include_classification_metrics=True):
    """
    Run expanding window cross-validation for the baseline model.
    
    Process:
        1. Load pre-checkpointed folds (expanding window)
        2. For each fold (except last):
           a. Train model on training set
           b. Evaluate on both training and validation sets
           c. Compute regression and classification metrics
           d. Track the best model (by validation RMSE)
        3. Evaluate best model on held-out test set
        4. Report comprehensive metrics, feature importance, and runtime
    
    Args:
        n_folds (int): Number of folds (default 5, last fold is test)
        version (str): Dataset version ("3M", "12M", etc.)
        include_classification_metrics (bool): Whether to compute OTPA, SDDR, etc.
        
    Returns:
        tuple: (best_model, val_metrics_list, test_metrics_dict, metrics_df,
                metrics_defs, classification_df, test_class_results)
    
    Memory Management Strategy:
        - .cache().count(): Forces immediate materialization in controlled chunks
        - .unpersist(): Explicitly frees memory after each fold
        - This prevents OOM errors by avoiding lazy accumulation of cached data
    """
    # ========================================
    # Initialize Timer and Load Data
    # ========================================
    overall_start_time = time.time()
    folds = _load_folds(n_folds=n_folds, version=version)

    # ========================================
    # Initialize Evaluators
    # ========================================
    # Create evaluators for each regression metric
    # These will be reused across all folds for consistency
    eval_rmse = RegressionEvaluator(predictionCol="prediction", labelCol="DEP_DELAY", metricName="rmse")
    eval_mae  = RegressionEvaluator(predictionCol="prediction", labelCol="DEP_DELAY", metricName="mae")
    eval_r2   = RegressionEvaluator(predictionCol="prediction", labelCol="DEP_DELAY", metricName="r2")
    eval_mse  = RegressionEvaluator(predictionCol="prediction", labelCol="DEP_DELAY", metricName="mse")

    # ========================================
    # Initialize Metric Storage
    # ========================================
    metrics = []  # Validation metrics per fold (for backward compatibility)
    train_records = []  # Per-fold train metrics
    val_records = []    # Per-fold validation metrics
    classification_records = []  # Classification metrics per fold
    best_model = None  # Best model (by validation RMSE)
    best_rmse = float("inf")  # Track best validation RMSE

    # ========================================
    # Initialize Estimator
    # ========================================
    est = BaselineEstimator(label_col="DEP_DELAY")

    # ========================================
    # Cross-Validation Loop
    # ========================================
    # Train/validate on first n_folds-1; last fold used as held-out test
    # folds[:-1] excludes the last fold (which is train/test pair)
    for idx, (train_df, val_df) in enumerate(folds[:-1], start=1):
        print(f"--- Fold {idx}/{n_folds - 1} ---")
        
        # ========================================
        # Cache DataFrames for Performance
        # ========================================
        # .cache().count() forces immediate materialization
        # This prevents memory accumulation by materializing in controlled chunks
        # The paradox: .count() seems expensive but actually prevents OOM!
        train_df.cache().count()
        val_df.cache().count()
        
        # ========================================
        # Train Model
        # ========================================
        model = est.fit(train_df)
        
        # ========================================
        # Generate Predictions
        # ========================================
        # Generate predictions on both validation and training sets
        # Training metrics help detect overfitting
        
        # Validation predictions
        val_preds = model.transform(val_df)
        val_preds.cache().count()  # Materialize to prevent recomputation
        
        # Train predictions
        train_preds = model.transform(train_df)
        train_preds.cache().count()  # Materialize to prevent recomputation

        # ========================================
        # Compute Regression Metrics (Validation)
        # ========================================
        val_rmse = eval_rmse.evaluate(val_preds)
        val_mae  = eval_mae.evaluate(val_preds)
        val_r2   = eval_r2.evaluate(val_preds)
        val_mse  = eval_mse.evaluate(val_preds)
        
        # Store validation metrics
        metrics.append({"fold": idx, "rmse": val_rmse, "mae": val_mae, "r2": val_r2, "mse": val_mse})
        val_records.append({"split": "validation", "fold": idx, "rmse": val_rmse, "mae": val_mae, "r2": val_r2, "mse": val_mse})

        # ========================================
        # Compute Regression Metrics (Training)
        # ========================================
        # Training metrics help detect overfitting
        # If train metrics >> val metrics, model is overfitting
        tr_rmse = eval_rmse.evaluate(train_preds)
        tr_mae  = eval_mae.evaluate(train_preds)
        tr_r2   = eval_r2.evaluate(train_preds)
        tr_mse  = eval_mse.evaluate(train_preds)
        train_records.append({"split": "train", "fold": idx, "rmse": tr_rmse, "mae": tr_mae, "r2": tr_r2, "mse": tr_mse})

        # ========================================
        # Track Best Model
        # ========================================
        # Select model with lowest validation RMSE
        # This model will be used for final test evaluation
        if val_rmse < best_rmse:
            best_rmse = val_rmse
            best_model = model

        # Print validation metrics
        print(f"RMSE: {val_rmse:.2f}  MAE: {val_mae:.2f}  R²: {val_r2:.4f}  MSE: {val_mse:.2f}")
        
        # ========================================
        # Compute Classification Metrics (Optional)
        # ========================================
        # Classification metrics provide operational insights:
        # - OTPA: On-Time Performance Accuracy (<15 min threshold)
        # - SDDR: Severe Delay Detection Rate (≥60 min threshold)
        # - Bucket Accuracy: 4-bucket classification (Early, OnTime, Delayed, Severe)
        if include_classification_metrics:
            val_class_results, _ = compute_classification_metrics(val_preds)
            classification_records.append({
                "split": "validation",
                "fold": idx,
                "otpa_accuracy": val_class_results["otpa_accuracy"],
                "otpa_f1": val_class_results["otpa_f1"],
                "sddr_recall": val_class_results["sddr_recall"],
                "bucket_accuracy": val_class_results["bucket_accuracy"]
            })
            print(f"  OTPA Acc: {val_class_results['otpa_accuracy']:.4f}  "
                  f"OTPA F1: {val_class_results['otpa_f1']:.4f}  "
                  f"SDDR: {val_class_results['sddr_recall']:.4f}")
        
        # ========================================
        # Free Memory
        # ========================================
        # Explicitly unpersist cached DataFrames to free memory
        # Critical for preventing OOM in subsequent folds
        train_preds.unpersist()
        val_preds.unpersist()
        train_df.unpersist()
        val_df.unpersist()

    # ========================================
    # Validation Summary
    # ========================================
    # Report aggregate statistics across all validation folds
    # This helps assess model stability and generalization
    print("-------------------------------")
    print("Validation Summary:")
    print(f"Best RMSE: {best_rmse:.2f}")
    print(f"Mean RMSE: {np.mean([m['rmse'] for m in metrics]):.2f}  |  Std: {np.std([m['rmse'] for m in metrics]):.2f}")
    print(f"Mean MAE:  {np.mean([m['mae'] for m in metrics]):.2f}")
    print(f"Mean R²:   {np.mean([m['r2'] for m in metrics]):.4f}")
    print(f"Mean MSE:  {np.mean([m['mse'] for m in metrics]):.2f}")

    # ========================================
    # Held-Out Test Evaluation
    # ========================================
    # Evaluate best model on held-out test set (last fold)
    # This provides an unbiased estimate of production performance
    train_df, test_df = folds[-1]
    test_df.cache().count()  # Cache and materialize test data
    
    # Generate predictions on test set
    test_preds = best_model.transform(test_df)
    test_preds.cache().count()  # Cache and materialize predictions
    
    # Compute regression metrics on test set
    test_rmse = eval_rmse.evaluate(test_preds)
    test_mae  = eval_mae.evaluate(test_preds)
    test_r2   = eval_r2.evaluate(test_preds)
    test_mse  = eval_mse.evaluate(test_preds)
    
    # Free memory
    test_preds.unpersist()
    test_df.unpersist()

    # Print test results
    print("-------------------------------")
    print("Held-out Test:")
    print(f"RMSE: {test_rmse:.2f}  MAE: {test_mae:.2f}  R²: {test_r2:.4f}  MSE: {test_mse:.2f}")
    
    # ========================================
    # Test Set Classification Metrics
    # ========================================
    # Compute classification metrics for test set if requested
    # Provides operational insights for deployment decisions
    if include_classification_metrics:
        test_class_results, test_class_df = compute_classification_metrics(test_preds)
        print_classification_summary(test_class_results, test_class_df)
        classification_records.append({
            "split": "test",
            "fold": n_folds,
            "otpa_accuracy": test_class_results["otpa_accuracy"],
            "otpa_f1": test_class_results["otpa_f1"],
            "sddr_recall": test_class_results["sddr_recall"],
            "bucket_accuracy": test_class_results["bucket_accuracy"]
        })

    # ========================================
    # Build Consolidated Metrics DataFrames
    # ========================================
    # Combine train/val/test metrics into single DataFrame for easy comparison
    test_record = [{"split": "test", "fold": n_folds, "rmse": test_rmse, "mae": test_mae, "r2": test_r2, "mse": test_mse}]
    metrics_df = pd.DataFrame(train_records + val_records + test_record)
    
    # Build classification metrics DataFrame if computed
    classification_df = pd.DataFrame(classification_records) if include_classification_metrics else None
    
    # Round to 4 decimal places for readability
    metrics_df['rmse'] = metrics_df['rmse'].round(4)
    metrics_df['mae'] = metrics_df['mae'].round(4)
    metrics_df['r2'] = metrics_df['r2'].round(4)
    metrics_df['mse'] = metrics_df['mse'].round(4)

    # ========================================
    # Metrics Definitions with LaTeX
    # ========================================
    # Provide LaTeX equations for each metric for documentation/reporting
    metrics_defs = pd.DataFrame([
        {"metric": "RMSE", "latex": r"\\mathrm{RMSE} = \\sqrt{\\tfrac{1}{N} \\sum_{i=1}^{N} (y_i - \\hat{y}_i)^2}"},
        {"metric": "MAE",  "latex": r"\\mathrm{MAE} = \\tfrac{1}{N} \\sum_{i=1}^{N} |y_i - \\hat{y}_i|"},
        {"metric": "R^2",  "latex": r"R^2 = 1 - \\dfrac{\\sum_{i=1}^{N} (y_i - \\hat{y}_i)^2}{\\sum_{i=1}^{N} (y_i - \\bar{y})^2}"},
        {"metric": "MSE",  "latex": r"\\mathrm{MSE} = \\tfrac{1}{N} \\sum_{i=1}^{N} (y_i - \\hat{y}_i)^2"},
    ])

    # ========================================
    # Extract Feature Importance (Coefficients)
    # ========================================
    # Extract and display top 10 most important features by absolute coefficient
    # Helps understand which features drive predictions
    try:
        # Extract coefficients from the linear regression model (last stage in pipeline)
        coefficients = best_model.model.stages[-1].coefficients.toArray()
        
        # Try to extract feature names from metadata (includes OHE expansions)
        feats_meta = test_preds.schema["features"].metadata
        attrs = []
        if "ml_attr" in feats_meta and "attrs" in feats_meta["ml_attr"]:
            ml_attrs = feats_meta["ml_attr"]["attrs"]
            # Collect attributes from all types (binary, numeric, nominal)
            for t in ["binary", "numeric", "nominal"]:
                if t in ml_attrs:
                    attrs.extend(ml_attrs[t])
            # Sort by vector index to match coefficient order
            attrs = sorted(attrs, key=lambda x: x["idx"])
            feature_names = [a.get("name", f"feature_{a['idx']}") for a in attrs]
        else:
            # Fallback: use generic names if metadata unavailable
            feature_names = [f"feature_{i}" for i in range(len(coefficients))]
        
        # Safety check: ensure lengths match
        if len(feature_names) != len(coefficients):
            feature_names = [f"feature_{i}" for i in range(len(coefficients))]
        
        # Create DataFrame with feature names and coefficients
        coef_df = pd.DataFrame({"feature": feature_names, "coefficient": coefficients})
        coef_df["abs_coefficient"] = coef_df["coefficient"].abs()
        coef_df = coef_df.sort_values("abs_coefficient", ascending=False)
        
        # Round coefficients to 4 decimal places
        top_coef_df = coef_df.head(10).copy()
        top_coef_df['coefficient'] = top_coef_df['coefficient'].round(4)
        top_coef_df['abs_coefficient'] = top_coef_df['abs_coefficient'].round(4)
        
        print("\nTop 10 Most Important Features (by absolute coefficient):")
        print(top_coef_df.to_string(index=False))
    except Exception as e:
        print(f"[Info] Skipping coefficient listing: {e}")
    
    # ========================================
    # Print Total Runtime
    # ========================================
    # Report total execution time for performance tracking
    overall_elapsed = time.time() - overall_start_time
    hours, remainder = divmod(overall_elapsed, 3600)
    minutes, seconds = divmod(remainder, 60)
    print("\n" + "="*80)
    print(f"TOTAL RUNTIME: {int(hours)}h {int(minutes)}m {seconds:.2f}s ({overall_elapsed:.2f} seconds)")
    print("="*80)

    # ========================================
    # Return Results
    # ========================================
    return best_model, metrics, {"rmse": test_rmse, "mae": test_mae, "r2": test_r2, "mse": test_mse}, metrics_df, metrics_defs, classification_df, test_class_results if include_classification_metrics else None


# ============================================================================
# Main Execution Block
# ============================================================================

if __name__ == "__main__":
    """
    Main execution entry point for the cross-validation script.
    
    Workflow:
        1. Print cluster configuration (for reproducibility and debugging)
        2. Run expanding window cross-validation
        3. Display comprehensive results (regression + classification metrics)
    
    Note:
        This block only runs when the script is executed directly,
        not when imported as a module.
    """
    
    # ========================================
    # Print Cluster Configuration
    # ========================================
    # Display cluster resources to ensure adequate capacity and enable reproducibility
    # Important for:
    # - Debugging OOM errors (check if cluster is undersized)
    # - Reproducing results (document exact cluster configuration)
    # - Cost tracking (understand resource usage)
    
    sc = spark.sparkContext
    
    # Get number of executors (exclude driver node)
    # statusTracker().getExecutorInfos() returns list of all executors
    num_executors = len(sc._jsc.sc().statusTracker().getExecutorInfos()) - 1
    
    # Get executor configuration from Spark conf
    executor_memory = sc.getConf().get("spark.executor.memory", "Unknown")
    executor_cores = sc.getConf().get("spark.executor.cores", "Unknown")
    actual_cores = sc.defaultParallelism / num_executors if num_executors > 0 else "Unknown"
    print(f"Estimated cores per executor: {int(actual_cores)}")
    
    # Print cluster configuration
    print("\n" + "="*80)
    print("CLUSTER CONFIGURATION")
    print("="*80)
    print(f"Cluster Size: {num_executors} executors, {executor_cores} cores per executor, {executor_memory} RAM per executor")
    print("="*80 + "\n")
    
    # ========================================
    # Run Cross-Validation
    # ========================================
    # Execute expanding window CV with comprehensive metrics
    # Parameters:
    # - n_folds=5: 4 validation folds + 1 held-out test fold
    # - version="12M": Use 12-month dataset
    # - include_classification_metrics=True: Compute OTPA, SDDR, bucket accuracy
    
    best_model, val_metrics_list, test_metrics_dict, metrics_df, metrics_defs, classification_df, test_class_results = run_cv(
        n_folds=5, 
        version="12M",
        include_classification_metrics=True
    )
    
    # ========================================
    # Display Regression Metrics
    # ========================================
    # Show comprehensive regression metrics across train/val/test splits
    # metrics_df: Per-fold metrics (split, fold, RMSE, MAE, R², MSE)
    # metrics_defs: LaTeX equations for each metric (for documentation)
    
    print("\n" + "="*80)
    print("REGRESSION METRICS")
    print("="*80)
    display(metrics_df)  # Use display() for interactive Databricks table
    display(metrics_defs)
    
    # ========================================
    # Display Classification Metrics
    # ========================================
    # Show operational classification metrics if computed
    # Includes OTPA (on-time accuracy), SDDR (severe delay detection),
    # and 4-bucket classification accuracy
    
    if classification_df is not None:
        print("\n" + "="*80)
        print("CLASSIFICATION METRICS SUMMARY")
        print("="*80)
        display(classification_df)  # Use display() for interactive Databricks table

Our baseline linear regression model was trained on a 12-month flight delay dataset using expanding window cross-validation with 4 folds. The model achieved a test RMSE of 35.96 minutes with an R² of 0.0077, indicating that the baseline features explain approximately 0.77% of the variance in flight delays. While regression performance is modest, the model demonstrates strong on-time prediction accuracy (OTPA: 69.52%) but struggles with severe delay detection (SDDR: 0.03%).

Key Observations:
Best fold: Fold 4 achieved the lowest RMSE (30.38 minutes)
Consistency: Standard deviation of 3.49 minutes indicates reasonable stability across folds
Generalization: Test RMSE (35.96) is close to mean validation RMSE (35.86), suggesting no overfitting

#### On-Time Performance Prediction Accuracy (OTPA)
| Metric     | Value   | Interpretation                                         |
|------------|---------|-------------------------------------------------------|
| Accuracy   | 69.52%  | Correctly classifies 7 out of 10 flights as on-time vs delayed |
| Precision  | 87.32%  | When predicting on-time, correct 87% of the time      |
| Recall     | 74.46%  | Identifies 74% of actual on-time flights              |
| F1-Score   | 80.38%  | Balanced performance for on-time prediction           |

#### Severe Delay Detection Rate (SDDR)
| Metric           | Value   | Interpretation                                      |
|------------------|---------|-----------------------------------------------------|
| Recall (SDDR)    | 0.03%   | Detects only 14 out of 43,939 severe delays         |
| Precision        | 17.72%  | When predicting severe delay, correct 18% of the time|
| F1-Score         | 0.06%   | Very poor performance on severe delays              |

Key Observations:
The model essentially fails to predict severe delays, identifying less than 0.03% of them.

#### 4-Bucket Classification Performance
| Bucket   | Delay Range | Precision | Recall  | F1-Score | Support   |
|----------|-------------|-----------|---------|----------|-----------|
| Early    | < 0 min     | 78.31%    | 8.36%   | 15.10%   | 548,981   |
| OnTime   | 0-15 min    | 23.37%    | 63.11%  | 34.11%   | 221,627   |
| Delayed  | 15-60 min   | 17.09%    | 42.84%  | 24.44%   | 104,515   |
| Severe   | ≥ 60 min    | 17.72%    | 0.03%   | 0.06%    | 43,939    |
| Overall  | -           | -         | -       | -        | 919,062   |


Overall Bucket Accuracy: 25.08%
Key Observations:
Early flights: High precision but very low recall (misses 92% of early departures)
OnTime flights: Best overall performance with 63% recall
Delayed flights: Moderate recall (43%) but low precision
Severe delays: Catastrophically poor performance across all metrics

#### Top 10 Most Important Features
| Rank | Feature                | Coefficient | Abs. Coefficient | Interpretation                          |
|------|------------------------|-------------|------------------|------------------------------------------|
| 1    | DEP_TIME_BLK_0600-0659 | -2.2436     | 2.2436           | Early morning flights → shorter delays   |
| 2    | HourlyVisibility       | -2.1598     | 2.1598           | Better visibility → shorter delays       |
| 3    | DEP_TIME_BLK_0700-0759 | -1.8192     | 1.8192           | Morning flights → shorter delays         |
| 4    | DEP_TIME_BLK_1900-1959 | +1.6132     | 1.6132           | Evening flights → longer delays          |
| 5    | DEP_TIME_BLK_1800-1859 | +1.5372     | 1.5372           | Late afternoon flights → longer delays   |
| 6    | DEP_TIME_BLK_2000-2059 | +1.4628     | 1.4628           | Night flights → longer delays            |
| 7    | DEP_TIME_BLK_0800-0859 | -1.3745     | 1.3745           | Mid-morning flights → shorter delays     |
| 8    | DEP_TIME_BLK_1700-1759 | +1.3518     | 1.3518           | Rush hour flights → longer delays        |
| 9    | DEP_TIME_BLK_0001-0559 | -1.2453     | 1.2453           | Red-eye flights → shorter delays         |
| 10   | MONTH_6 (June)         | +1.2103     | 1.2103           | Summer travel → longer delays            |

Key Insights:
Time of day dominates: 8 of top 10 features are departure time blocks
Clear pattern: Morning flights (6-9 AM) have shorter delays; evening flights (5-8 PM) have longer delays
Weather matters: Visibility is the 2nd most important feature
Seasonal effect: June shows increased delays (summer travel season)

# Detailed Classification Metrics Table

| Metric | Value | Category | Formula | Description |
|--------|-------|----------|---------|-------------|
| **OTPA Accuracy** | 0.6952 | On-Time Performance | $\text{OTPA} = \frac{TP + TN}{TP + TN + FP + FN}$ | Accuracy of predicting on-time (<15 min) vs delayed (≥15 min) |
| **OTPA Precision** | 0.8732 | On-Time Performance | $\text{Precision} = \frac{TP}{TP + FP}$ | Proportion of predicted on-time flights that are actually on-time |
| **OTPA Recall** | 0.7446 | On-Time Performance | $\text{Recall} = \frac{TP}{TP + FN}$ | Proportion of actual on-time flights correctly identified |
| **OTPA F1-Score** | 0.8038 | On-Time Performance | $F1 = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$ | Harmonic mean of OTPA precision and recall |
| **SDDR (Recall)** | 0.0003 | Severe Delay Detection | $\text{SDDR} = \frac{TP_{severe}}{TP_{severe} + FN_{severe}}$ | Proportion of severe delays (≥60 min) correctly identified |
| **SDDR Precision** | 0.1772 | Severe Delay Detection | $\text{Precision} = \frac{TP_{severe}}{TP_{severe} + FP_{severe}}$ | Proportion of predicted severe delays that are actually severe |
| **SDDR F1-Score** | 0.0006 | Severe Delay Detection | $F1 = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$ | Harmonic mean of SDDR precision and recall |
| **Bucket Accuracy** | 0.2508 | 4-Bucket Classification | $\text{Accuracy} = \frac{\text{Correct Predictions}}{\text{Total Predictions}}$ | Overall accuracy across all 4 delay buckets |
| **Early Precision** | 0.7831 | Bucket-Specific | $\text{Precision} = \frac{TP}{TP + FP}$ | Early departures (<0 min) - support: 548,981 |
| **Early Recall** | 0.0836 | Bucket-Specific | $\text{Recall} = \frac{TP}{TP + FN}$ | Early departures (<0 min) - support: 548,981 |
| **Early F1-Score** | 0.1510 | Bucket-Specific | $F1 = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$ | Early departures (<0 min) - support: 548,981 |
| **OnTime Precision** | 0.2337 | Bucket-Specific | $\text{Precision} = \frac{TP}{TP + FP}$ | On-time departures (0-15 min) - support: 221,627 |
| **OnTime Recall** | 0.6311 | Bucket-Specific | $\text{Recall} = \frac{TP}{TP + FN}$ | On-time departures (0-15 min) - support: 221,627 |
| **OnTime F1-Score** | 0.3411 | Bucket-Specific | $F1 = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$ | On-time departures (0-15 min) - support: 221,627 |
| **Delayed Precision** | 0.1709 | Bucket-Specific | $\text{Precision} = \frac{TP}{TP + FP}$ | Delayed departures (15-60 min) - support: 104,515 |
| **Delayed Recall** | 0.4284 | Bucket-Specific | $\text{Recall} = \frac{TP}{TP + FN}$ | Delayed departures (15-60 min) - support: 104,515 |
| **Delayed F1-Score** | 0.2444 | Bucket-Specific | $F1 = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$ | Delayed departures (15-60 min) - support: 104,515 |
| **Severe Precision** | 0.1772 | Bucket-Specific | $\text{Precision} = \frac{TP}{TP + FP}$ | Severely delayed departures (≥60 min) - support: 43,939 |
| **Severe Recall** | 0.0003 | Bucket-Specific | $\text{Recall} = \frac{TP}{TP + FN}$ | Severely delayed departures (≥60 min) - support: 43,939 |
| **Severe F1-Score** | 0.0006 | Bucket-Specific | $F1 = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$ | Severely delayed departures (≥60 min) - support: 43,939 |

Key Highlights
- Strong OTPA Performance: 69.52% accuracy with 87.32% precision for on-time predictions
- Critical SDDR Issue: Only 0.03% recall for severe delays (14 out of 43,939 detected)
- Bucket Distribution: Early (548K), OnTime (222K), Delayed (105K), Severe (44K)
- Best Bucket: OnTime has highest F1-score (0.3411) with 63% recall

# Model 4 - Cross-Validation/12-month/Dropped-Null/Default-Data

In [0]:
"""
================================================================================
Cross-Validation for Baseline Linear Regression (12-Month Dataset)
================================================================================

Purpose:
    Implements expanding window cross-validation for flight delay prediction
    using a baseline linear regression model with comprehensive feature engineering.

Key Features:
    - Expanding window CV: Each fold trains on progressively more historical data
    - Pre-checkpointed data: Uses coworker's fold pattern (OTPW_{version}_FOLD_{i}_{TRAIN/VAL/TEST})
    - Baseline pipeline: Numeric cleaning + median imputation + categorical OHE + standardization + LR
    - Comprehensive metrics: Regression (RMSE, MAE, R², MSE) + Classification (OTPA, SDDR, 4-bucket)
    - Memory management: Strategic caching with .count() to prevent OOM errors

Data Flow:
    1. Load pre-checkpointed folds from DBFS
    2. For each fold (except last):
       - Train model on expanding training set
       - Evaluate on validation set
       - Track best model by RMSE
    3. Evaluate best model on held-out test set
    4. Report comprehensive metrics and feature importance

Author: Emily Lieske
Date: November 2025
================================================================================
"""

# ============================================================================
# Imports
# ============================================================================

# PySpark SQL functions for data transformation
from pyspark.sql.functions import col, when, isnan, regexp_replace, trim, length
from pyspark.sql.types import DoubleType, StringType

# PySpark ML for pipeline construction and modeling
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder, StandardScaler, Imputer
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

# Standard libraries for metrics and timing
import numpy as np
import pandas as pd
import time
import gc  # Garbage collection for memory management


# ============================================================================
# Data Loading Functions
# ============================================================================

def _load_checkpointed_data(name, folder_path="dbfs:/student-groups/Group_4_2"):
    """
    Load a pre-checkpointed Parquet dataset from DBFS.
    
    Args:
        name (str): Dataset name (e.g., "OTPW_12M_FOLD_1_TRAIN")
        folder_path (str): DBFS path to the folder containing checkpointed data
        
    Returns:
        pyspark.sql.DataFrame: Loaded dataset
        
    Note:
        Pre-checkpointed data significantly speeds up CV by avoiding repeated
        data loading and splitting operations.
    """
    return spark.read.parquet(f"{folder_path}/{name}.parquet")


def _load_folds(n_folds=5, version="12M"):
    """
    Load all CV folds for expanding window cross-validation.
    
    Args:
        n_folds (int): Total number of folds (default: 5)
                       First n_folds-1 are train/val pairs
                       Last fold is train/test pair for final evaluation
        version (str): Dataset version ("3M", "12M", etc.)
        
    Returns:
        list of tuples: [(train_df, val_df), ..., (train_df, test_df)]
        
    Expanding Window Strategy:
        - Fold 1: Train on months 1-8, validate on month 9
        - Fold 2: Train on months 1-9, validate on month 10
        - Fold 3: Train on months 1-10, validate on month 11
        - Fold 4: Train on months 1-11, validate on month 12
        - Fold 5: Train on months 1-12, test on held-out data
        
    This mimics real-world deployment where models are retrained on
    progressively more historical data.
    """
    folds = []
    for i in range(1, n_folds + 1):
        # Load training data for this fold
        train_df = _load_checkpointed_data(f"OTPW_{version}_FOLD_{i}_TRAIN")
        
        if i != n_folds:
            # For folds 1 to n_folds-1: load validation set
            val_df = _load_checkpointed_data(f"OTPW_{version}_FOLD_{i}_VAL")
            folds.append((train_df, val_df))
        else:
            # For last fold: load held-out test set
            test_df = _load_checkpointed_data(f"OTPW_{version}_FOLD_{i}_TEST")
            folds.append((train_df, test_df))
    
    return folds


# ============================================================================
# Baseline Estimator Class
# ============================================================================

class BaselineEstimator:
    """
    Baseline Linear Regression Estimator with Feature Engineering Pipeline.
    
    This class encapsulates the entire feature engineering and modeling pipeline:
        1. Data preparation (label cleaning, feature selection)
        2. Numerical feature cleaning (remove non-numeric chars, handle nulls)
        3. Median imputation for numerical features
        4. Categorical encoding (StringIndexer + OneHotEncoder)
        5. Feature assembly and standardization
        6. Linear regression modeling
    
    Feature Families:
        - Temporal: DAY_OF_WEEK, MONTH, DEP_TIME_BLK
        - Airport: ORIGIN, DEST
        - Flight: OP_UNIQUE_CARRIER, DISTANCE
        - Weather: HourlyWindSpeed, HourlyVisibility, HourlyPrecipitation
    
    Why This Design:
        - Encapsulation: All preprocessing logic in one place
        - Reusability: Same pipeline for train/val/test
        - Spark ML compatibility: Uses Pipeline for efficient execution
    """
    
    def __init__(self, label_col="DEP_DELAY"):
        """
        Initialize the estimator with feature definitions.
        
        Args:
            label_col (str): Name of the target variable column
        """
        self.label_col = label_col
        self.pipeline = None
        self.model = None
        
        # Categorical features: Encoded using StringIndexer + OneHotEncoder
        # These capture temporal patterns, route characteristics, and carrier effects
        self.categorical_features = [
            "DAY_OF_WEEK",        # Day of week (1=Monday, 7=Sunday)
            "MONTH",              # Month of year (1-12)
            "DEP_TIME_BLK",       # Departure time block (e.g., "0600-0659")
            "ORIGIN",             # Origin airport code
            "DEST",               # Destination airport code
            "OP_UNIQUE_CARRIER"   # Operating carrier code
        ]
        
        # Numerical features: Imputed with median and standardized
        # These capture weather conditions and flight distance
        self.numerical_features = [
            "HourlyWindSpeed",       # Wind speed at origin (mph)
            "HourlyVisibility",      # Visibility at origin (miles)
            "HourlyPrecipitation",   # Precipitation at origin (inches)
            "DISTANCE"               # Flight distance (miles)
        ]

    def _prepare(self, df):
        """
        Prepare the DataFrame for modeling by cleaning the label and numerical features.
        
        Steps:
            1. Cast label to DoubleType (required by Spark ML)
            2. Filter out rows with null/NaN labels (Spark ML requirement)
            3. Select only required features + label
            4. Clean numerical features:
               - Remove non-numeric characters (e.g., "12.5mph" -> "12.5")
               - Convert empty strings to null
               - Cast to DoubleType
        
        Args:
            df (pyspark.sql.DataFrame): Input DataFrame
            
        Returns:
            pyspark.sql.DataFrame: Cleaned DataFrame
            
        Why This Matters:
            - Spark ML LinearRegression requires DoubleType labels with no nulls
            - Numerical features may contain string artifacts from source data
            - Explicit type casting prevents downstream pipeline errors
        """
        # Cast label to double and filter out null/NaN values
        # Spark ML does not accept null labels
        df = df.withColumn(self.label_col, col(self.label_col).cast(DoubleType()))
        df = df.filter(~(col(self.label_col).isNull() | isnan(col(self.label_col))))

        # Select only the columns we need (features + label)
        # This reduces memory footprint and prevents accidental data leakage
        selected = [c for c in (self.categorical_features + self.numerical_features + [self.label_col]) 
                    if c in df.columns]
        df = df.select(*selected)

        # Clean numerical features: remove non-numeric characters, handle empty strings
        for f in self.numerical_features:
            if f in df.columns:
                # Step 1: Cast to string to enable regex operations
                # Step 2: Remove all non-numeric characters except +, -, and .
                df = df.withColumn(f, regexp_replace(col(f).cast(StringType()), r"[^0-9+\-\.]", ""))
                
                # Step 3: Convert empty strings to null (for imputation)
                df = df.withColumn(f, when(length(trim(col(f))) == 0, None).otherwise(col(f)))
                
                # Step 4: Cast to DoubleType for modeling
                df = df.withColumn(f, col(f).cast(DoubleType()))
        
        return df

    def _build_pipeline(self, df):
        """
        Build the Spark ML Pipeline with all feature engineering stages.
        
        Pipeline Stages:
            1. Drop nulls: Remove rows with missing numerical features
            2. StringIndexers: Convert categorical strings to indices
            3. OneHotEncoders: Convert indices to binary vectors
            4. VectorAssembler: Combine all features into single vector
            5. StandardScaler: Standardize features (mean=0, std=1)
            6. LinearRegression: Train linear model
        
        Args:
            df (pyspark.sql.DataFrame): Prepared DataFrame
            
        Returns:
            pyspark.sql.DataFrame: DataFrame (may have additional columns from transformations)
            
        Why This Design:
            - Pipeline ensures consistent transformations across train/val/test
            - Dropping nulls instead of imputing for cleaner baseline
            - OneHotEncoding with dropLast=True prevents multicollinearity
            - StandardScaler improves convergence for gradient descent
            - No regularization (regParam=0) for interpretable baseline
        """
        stages = []
        
        # ========================================
        # Stage 1: Drop Null Values in Numerical Features
        # ========================================
        # Drop rows with any null values in numerical features
        # This ensures we only train on complete cases
        for f in self.numerical_features:
            if f in df.columns:
                df = df.filter(col(f).isNotNull())
        
        # Repartition after filtering to rebalance data across executors
        # Filtering can create skewed partitions (some empty, some overloaded)
        # Repartitioning ensures efficient parallel processing
        # Use 8 partitions to match available cores (2 executors × 4 cores = 8 total)
        df = df.repartition(8)
        
        # Rename numerical features to match expected naming convention
        # (no "_imputed" suffix since we're not imputing)
        for f in self.numerical_features:
            if f in df.columns:
                df = df.withColumnRenamed(f, f"{f}_imputed")

        # ========================================
        # Stage 2: Categorical Encoding (StringIndexer + OneHotEncoder)
        # ========================================
        # StringIndexer: Converts strings to numeric indices (most frequent = 0)
        # OneHotEncoder: Converts indices to binary vectors (prevents ordinal assumption)
        # handleInvalid="keep": Unseen categories in test set get their own index
        # dropLast=True: Drop last category to prevent multicollinearity
        for f in self.categorical_features:
            if f in df.columns:
                # For numeric-coded categoricals (DAY_OF_WEEK, MONTH, DEP_TIME_BLK),
                # cast to string first to ensure consistent handling
                if f in ["DAY_OF_WEEK", "MONTH", "DEP_TIME_BLK"]:
                    df = df.withColumn(f"{f}_clean", 
                                      when(col(f).isNull(), "UNKNOWN").otherwise(col(f).cast(StringType())))
                else:
                    # For string categoricals (ORIGIN, DEST, CARRIER), handle nulls only
                    df = df.withColumn(f"{f}_clean", 
                                      when(col(f).isNull(), "UNKNOWN").otherwise(col(f)))
                
                # Add StringIndexer stage
                stages.append(StringIndexer(inputCol=f"{f}_clean", 
                                           outputCol=f"{f}_indexed", 
                                           handleInvalid="keep"))
                
                # Add OneHotEncoder stage
                stages.append(OneHotEncoder(inputCols=[f"{f}_indexed"], 
                                           outputCols=[f"{f}_encoded"], 
                                           dropLast=True))

        # ========================================
        # Stage 3: Feature Assembly
        # ========================================
        # Combine all features (numerical + encoded categorical) into single vector
        # handleInvalid="skip": Skip rows with invalid values
        feature_columns = [f"{f}_imputed" for f in self.numerical_features if f in df.columns] + \
                          [f"{f}_encoded" for f in self.categorical_features if f in df.columns]
        assembler = VectorAssembler(inputCols=feature_columns, 
                                    outputCol="features", 
                                    handleInvalid="skip")
        
        # ========================================
        # Stage 4: Feature Standardization
        # ========================================
        # Standardize features to mean=0, std=1
        # withStd=True: Scale to unit variance
        # withMean=True: Center to zero mean
        # Why? Improves convergence and makes coefficients comparable
        scaler = StandardScaler(inputCol="features", 
                               outputCol="scaled_features", 
                               withStd=True, 
                               withMean=True)
        
        # ========================================
        # Stage 5: Linear Regression
        # ========================================
        # Baseline model: No regularization (regParam=0, elasticNetParam=0)
        # maxIter=100: Maximum iterations for convergence
        # Why no regularization? We want an interpretable baseline to understand
        # feature importance before adding complexity
        lr = LinearRegression(featuresCol="scaled_features", 
                            labelCol=self.label_col, 
                            maxIter=100, 
                            regParam=0.0, 
                            elasticNetParam=0.0)

        # Assemble all stages into pipeline
        stages.extend([assembler, scaler, lr])
        self.pipeline = Pipeline(stages=stages)
        
        return df

    def fit(self, df):
        """
        Fit the pipeline on training data.
        
        Args:
            df (pyspark.sql.DataFrame): Training DataFrame
            
        Returns:
            BaselineEstimator: self (for method chaining)
        """
        df_prep = self._prepare(df)
        df_prep = self._build_pipeline(df_prep)
        self.model = self.pipeline.fit(df_prep)
        return self

    def transform(self, df):
        """
        Transform data using the fitted pipeline.
        
        Args:
            df (pyspark.sql.DataFrame): DataFrame to transform (val/test)
            
        Returns:
            pyspark.sql.DataFrame: Transformed DataFrame with predictions
            
        Note:
            We rebuild the pipeline on the input DataFrame to ensure all
            transformation columns are present, but use the fitted model
            for predictions.
        """
        df_prep = self._prepare(df)
        df_prep = self._build_pipeline(df_prep)  # Rebuild to ensure cols present
        return self.model.transform(df_prep)


# ============================================================================
# Cross-Validation Runner
# ============================================================================

def run_cv(n_folds=5, version="12M", include_classification_metrics=True):
    """
    Run expanding window cross-validation for the baseline model.
    
    Process:
        1. Load pre-checkpointed folds (expanding window)
        2. For each fold (except last):
           a. Train model on training set
           b. Evaluate on both training and validation sets
           c. Compute regression and classification metrics
           d. Track the best model (by validation RMSE)
        3. Evaluate best model on held-out test set
        4. Report comprehensive metrics, feature importance, and runtime
    
    Args:
        n_folds (int): Number of folds (default 5, last fold is test)
        version (str): Dataset version ("3M", "12M", etc.)
        include_classification_metrics (bool): Whether to compute OTPA, SDDR, etc.
        
    Returns:
        tuple: (best_model, val_metrics_list, test_metrics_dict, metrics_df,
                metrics_defs, classification_df, test_class_results)
    
    Memory Management Strategy:
        - .cache().count(): Forces immediate materialization in controlled chunks
        - .unpersist(): Explicitly frees memory after each fold
        - This prevents OOM errors by avoiding lazy accumulation of cached data
    """
    # ========================================
    # Initialize Timer and Load Data
    # ========================================
    overall_start_time = time.time()
    folds = _load_folds(n_folds=n_folds, version=version)

    # ========================================
    # Initialize Evaluators
    # ========================================
    # Create evaluators for each regression metric
    # These will be reused across all folds for consistency
    eval_rmse = RegressionEvaluator(predictionCol="prediction", labelCol="DEP_DELAY", metricName="rmse")
    eval_mae  = RegressionEvaluator(predictionCol="prediction", labelCol="DEP_DELAY", metricName="mae")
    eval_r2   = RegressionEvaluator(predictionCol="prediction", labelCol="DEP_DELAY", metricName="r2")
    eval_mse  = RegressionEvaluator(predictionCol="prediction", labelCol="DEP_DELAY", metricName="mse")

    # ========================================
    # Initialize Metric Storage
    # ========================================
    metrics = []  # Validation metrics per fold (for backward compatibility)
    train_records = []  # Per-fold train metrics
    val_records = []    # Per-fold validation metrics
    classification_records = []  # Classification metrics per fold
    best_model = None  # Best model (by validation RMSE)
    best_rmse = float("inf")  # Track best validation RMSE

    # ========================================
    # Initialize Estimator
    # ========================================
    est = BaselineEstimator(label_col="DEP_DELAY")

    # ========================================
    # Cross-Validation Loop
    # ========================================
    # Train/validate on first n_folds-1; last fold used as held-out test
    # folds[:-1] excludes the last fold (which is train/test pair)
    for idx, (train_df, val_df) in enumerate(folds[:-1], start=1):
        print(f"--- Fold {idx}/{n_folds - 1} ---")
        
        # ========================================
        # Cache DataFrames for Performance
        # ========================================
        # Caching disabled for Model 3B (drop nulls) to prevent OOM
        # The filtered dataset is smaller but repartitioning causes memory pressure
        # train_df.cache().count()
        # val_df.cache().count()
        
        # ========================================
        # Train Model
        # ========================================
        model = est.fit(train_df)
        
        # ========================================
        # Generate Predictions
        # ========================================
        # Generate predictions on both validation and training sets
        # Training metrics help detect overfitting
        
        # Validation predictions
        val_preds = model.transform(val_df)
        # val_preds.cache().count()  # Disabled to prevent OOM
        
        # Train predictions
        train_preds = model.transform(train_df)
        # train_preds.cache().count()  # Disabled to prevent OOM

        # ========================================
        # Compute Regression Metrics (Validation)
        # ========================================
        val_rmse = eval_rmse.evaluate(val_preds)
        val_mae  = eval_mae.evaluate(val_preds)
        val_r2   = eval_r2.evaluate(val_preds)
        val_mse  = eval_mse.evaluate(val_preds)
        
        # Store validation metrics
        metrics.append({"fold": idx, "rmse": val_rmse, "mae": val_mae, "r2": val_r2, "mse": val_mse})
        val_records.append({"split": "validation", "fold": idx, "rmse": val_rmse, "mae": val_mae, "r2": val_r2, "mse": val_mse})

        # ========================================
        # Compute Regression Metrics (Training)
        # ========================================
        # Training metrics help detect overfitting
        # If train metrics >> val metrics, model is overfitting
        tr_rmse = eval_rmse.evaluate(train_preds)
        tr_mae  = eval_mae.evaluate(train_preds)
        tr_r2   = eval_r2.evaluate(train_preds)
        tr_mse  = eval_mse.evaluate(train_preds)
        train_records.append({"split": "train", "fold": idx, "rmse": tr_rmse, "mae": tr_mae, "r2": tr_r2, "mse": tr_mse})

        # ========================================
        # Track Best Model
        # ========================================
        # Select model with lowest validation RMSE
        # This model will be used for final test evaluation
        if val_rmse < best_rmse:
            best_rmse = val_rmse
            best_model = model

        # Print validation metrics
        print(f"RMSE: {val_rmse:.2f}  MAE: {val_mae:.2f}  R²: {val_r2:.4f}  MSE: {val_mse:.2f}")
        
        # ========================================
        # Compute Classification Metrics (Optional)
        # ========================================
        # Classification metrics provide operational insights:
        # - OTPA: On-Time Performance Accuracy (<15 min threshold)
        # - SDDR: Severe Delay Detection Rate (≥60 min threshold)
        # - Bucket Accuracy: 4-bucket classification (Early, OnTime, Delayed, Severe)
        if include_classification_metrics:
            val_class_results, _ = compute_classification_metrics(val_preds)
            classification_records.append({
                "split": "validation",
                "fold": idx,
                "otpa_accuracy": val_class_results["otpa_accuracy"],
                "otpa_f1": val_class_results["otpa_f1"],
                "sddr_recall": val_class_results["sddr_recall"],
                "bucket_accuracy": val_class_results["bucket_accuracy"]
            })
            print(f"  OTPA Acc: {val_class_results['otpa_accuracy']:.4f}  "
                  f"OTPA F1: {val_class_results['otpa_f1']:.4f}  "
                  f"SDDR: {val_class_results['sddr_recall']:.4f}")
        
        # ========================================
        # Free Memory
        # ========================================
        # Explicitly unpersist cached DataFrames to free memory
        # Critical for preventing OOM in subsequent folds
        # Disabled since caching is disabled
        # train_preds.unpersist()
        # val_preds.unpersist()
        # train_df.unpersist()
        # val_df.unpersist()
        
        # Force Python garbage collection to free driver memory
        gc.collect()
        
        # Clear Spark SQL cache to free executor memory
        spark.catalog.clearCache()

    # ========================================
    # Validation Summary
    # ========================================
    # Report aggregate statistics across all validation folds
    # This helps assess model stability and generalization
    print("-------------------------------")
    print("Validation Summary:")
    print(f"Best RMSE: {best_rmse:.2f}")
    print(f"Mean RMSE: {np.mean([m['rmse'] for m in metrics]):.2f}  |  Std: {np.std([m['rmse'] for m in metrics]):.2f}")
    print(f"Mean MAE:  {np.mean([m['mae'] for m in metrics]):.2f}")
    print(f"Mean R²:   {np.mean([m['r2'] for m in metrics]):.4f}")
    print(f"Mean MSE:  {np.mean([m['mse'] for m in metrics]):.2f}")

    # ========================================
    # Held-Out Test Evaluation
    # ========================================
    # Evaluate best model on held-out test set (last fold)
    # This provides an unbiased estimate of production performance
    train_df, test_df = folds[-1]
    # test_df.cache().count()  # Disabled to prevent OOM
    
    # Generate predictions on test set
    test_preds = best_model.transform(test_df)
    # test_preds.cache().count()  # Disabled to prevent OOM
    
    # Compute regression metrics on test set
    test_rmse = eval_rmse.evaluate(test_preds)
    test_mae  = eval_mae.evaluate(test_preds)
    test_r2   = eval_r2.evaluate(test_preds)
    test_mse  = eval_mse.evaluate(test_preds)
    
    # Free memory
    # Disabled since caching is disabled
    # test_preds.unpersist()
    # test_df.unpersist()

    # Print test results
    print("-------------------------------")
    print("Held-out Test:")
    print(f"RMSE: {test_rmse:.2f}  MAE: {test_mae:.2f}  R²: {test_r2:.4f}  MSE: {test_mse:.2f}")
    
    # ========================================
    # Test Set Classification Metrics
    # ========================================
    # Compute classification metrics for test set if requested
    # Provides operational insights for deployment decisions
    if include_classification_metrics:
        test_class_results, test_class_df = compute_classification_metrics(test_preds)
        print_classification_summary(test_class_results, test_class_df)
        classification_records.append({
            "split": "test",
            "fold": n_folds,
            "otpa_accuracy": test_class_results["otpa_accuracy"],
            "otpa_f1": test_class_results["otpa_f1"],
            "sddr_recall": test_class_results["sddr_recall"],
            "bucket_accuracy": test_class_results["bucket_accuracy"]
        })

    # ========================================
    # Build Consolidated Metrics DataFrames
    # ========================================
    # Combine train/val/test metrics into single DataFrame for easy comparison
    test_record = [{"split": "test", "fold": n_folds, "rmse": test_rmse, "mae": test_mae, "r2": test_r2, "mse": test_mse}]
    metrics_df = pd.DataFrame(train_records + val_records + test_record)
    
    # Build classification metrics DataFrame if computed
    classification_df = pd.DataFrame(classification_records) if include_classification_metrics else None
    
    # Round to 4 decimal places for readability
    metrics_df['rmse'] = metrics_df['rmse'].round(4)
    metrics_df['mae'] = metrics_df['mae'].round(4)
    metrics_df['r2'] = metrics_df['r2'].round(4)
    metrics_df['mse'] = metrics_df['mse'].round(4)

    # ========================================
    # Metrics Definitions with LaTeX
    # ========================================
    # Provide LaTeX equations for each metric for documentation/reporting
    metrics_defs = pd.DataFrame([
        {"metric": "RMSE", "latex": r"\\mathrm{RMSE} = \\sqrt{\\tfrac{1}{N} \\sum_{i=1}^{N} (y_i - \\hat{y}_i)^2}"},
        {"metric": "MAE",  "latex": r"\\mathrm{MAE} = \\tfrac{1}{N} \\sum_{i=1}^{N} |y_i - \\hat{y}_i|"},
        {"metric": "R^2",  "latex": r"R^2 = 1 - \\dfrac{\\sum_{i=1}^{N} (y_i - \\hat{y}_i)^2}{\\sum_{i=1}^{N} (y_i - \\bar{y})^2}"},
        {"metric": "MSE",  "latex": r"\\mathrm{MSE} = \\tfrac{1}{N} \\sum_{i=1}^{N} (y_i - \\hat{y}_i)^2"},
    ])

    # ========================================
    # Extract Feature Importance (Coefficients)
    # ========================================
    # Extract and display top 10 most important features by absolute coefficient
    # Helps understand which features drive predictions
    try:
        # Extract coefficients from the linear regression model (last stage in pipeline)
        coefficients = best_model.model.stages[-1].coefficients.toArray()
        
        # Try to extract feature names from metadata (includes OHE expansions)
        feats_meta = test_preds.schema["features"].metadata
        attrs = []
        if "ml_attr" in feats_meta and "attrs" in feats_meta["ml_attr"]:
            ml_attrs = feats_meta["ml_attr"]["attrs"]
            # Collect attributes from all types (binary, numeric, nominal)
            for t in ["binary", "numeric", "nominal"]:
                if t in ml_attrs:
                    attrs.extend(ml_attrs[t])
            # Sort by vector index to match coefficient order
            attrs = sorted(attrs, key=lambda x: x["idx"])
            feature_names = [a.get("name", f"feature_{a['idx']}") for a in attrs]
        else:
            # Fallback: use generic names if metadata unavailable
            feature_names = [f"feature_{i}" for i in range(len(coefficients))]
        
        # Safety check: ensure lengths match
        if len(feature_names) != len(coefficients):
            feature_names = [f"feature_{i}" for i in range(len(coefficients))]
        
        # Create DataFrame with feature names and coefficients
        coef_df = pd.DataFrame({"feature": feature_names, "coefficient": coefficients})
        coef_df["abs_coefficient"] = coef_df["coefficient"].abs()
        coef_df = coef_df.sort_values("abs_coefficient", ascending=False)
        
        # Round coefficients to 4 decimal places
        top_coef_df = coef_df.head(10).copy()
        top_coef_df['coefficient'] = top_coef_df['coefficient'].round(4)
        top_coef_df['abs_coefficient'] = top_coef_df['abs_coefficient'].round(4)
        
        print("\nTop 10 Most Important Features (by absolute coefficient):")
        print(top_coef_df.to_string(index=False))
    except Exception as e:
        print(f"[Info] Skipping coefficient listing: {e}")
    
    # ========================================
    # Print Total Runtime
    # ========================================
    # Report total execution time for performance tracking
    overall_elapsed = time.time() - overall_start_time
    hours, remainder = divmod(overall_elapsed, 3600)
    minutes, seconds = divmod(remainder, 60)
    print("\n" + "="*80)
    print(f"TOTAL RUNTIME: {int(hours)}h {int(minutes)}m {seconds:.2f}s ({overall_elapsed:.2f} seconds)")
    print("="*80)

    # ========================================
    # Return Results
    # ========================================
    return best_model, metrics, {"rmse": test_rmse, "mae": test_mae, "r2": test_r2, "mse": test_mse}, metrics_df, metrics_defs, classification_df, test_class_results if include_classification_metrics else None


# ============================================================================
# Main Execution Block
# ============================================================================

if __name__ == "__main__":
    """
    Main execution entry point for the cross-validation script.
    
    Workflow:
        1. Print cluster configuration (for reproducibility and debugging)
        2. Run expanding window cross-validation
        3. Display comprehensive results (regression + classification metrics)
    
    Note:
        This block only runs when the script is executed directly,
        not when imported as a module.
    """
    
    # ========================================
    # Print Cluster Configuration
    # ========================================
    # Display cluster resources to ensure adequate capacity and enable reproducibility
    # Important for:
    # - Debugging OOM errors (check if cluster is undersized)
    # - Reproducing results (document exact cluster configuration)
    # - Cost tracking (understand resource usage)
    
    sc = spark.sparkContext
    
    # Get number of executors (exclude driver node)
    # statusTracker().getExecutorInfos() returns list of all executors
    num_executors = len(sc._jsc.sc().statusTracker().getExecutorInfos()) - 1
    
    # Get executor configuration from Spark conf
    executor_memory = sc.getConf().get("spark.executor.memory", "Unknown")
    executor_cores = sc.getConf().get("spark.executor.cores", "Unknown")
    actual_cores = sc.defaultParallelism / num_executors if num_executors > 0 else "Unknown"
    print(f"Estimated cores per executor: {int(actual_cores)}")
    
    # Print cluster configuration
    print("\n" + "="*80)
    print("CLUSTER CONFIGURATION")
    print("="*80)
    print(f"Cluster Size: {num_executors} executors, {executor_cores} cores per executor, {executor_memory} RAM per executor")
    print("="*80 + "\n")
    
    # ========================================
    # Run Cross-Validation
    # ========================================
    # Execute expanding window CV with comprehensive metrics
    # Parameters:
    # - n_folds=5: 4 validation folds + 1 held-out test fold
    # - version="12M": Use 12-month dataset
    # - include_classification_metrics=True: Compute OTPA, SDDR, bucket accuracy
    
    best_model, val_metrics_list, test_metrics_dict, metrics_df, metrics_defs, classification_df, test_class_results = run_cv(
        n_folds=5, 
        version="12M",
        include_classification_metrics=True
    )
    
    # ========================================
    # Display Regression Metrics
    # ========================================
    # Show comprehensive regression metrics across train/val/test splits
    # metrics_df: Per-fold metrics (split, fold, RMSE, MAE, R², MSE)
    # metrics_defs: LaTeX equations for each metric (for documentation)
    
    print("\n" + "="*80)
    print("REGRESSION METRICS")
    print("="*80)
    display(metrics_df)  # Use display() for interactive Databricks table
    display(metrics_defs)
    
    # ========================================
    # Display Classification Metrics
    # ========================================
    # Show operational classification metrics if computed
    # Includes OTPA (on-time accuracy), SDDR (severe delay detection),
    # and 4-bucket classification accuracy
    
    if classification_df is not None:
        print("\n" + "="*80)
        print("CLASSIFICATION METRICS SUMMARY")
        print("="*80)
        display(classification_df)  # Use display() for interactive Databricks table

In [0]:
displayHTML("""
<!DOCTYPE html>
<html>
<head>
  <script src="https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.min.js"></script>
  <script>
    mermaid.initialize({
      startOnLoad: true,
      theme: 'dark',
      themeVariables: {
        primaryColor: '#4a5568',
        primaryTextColor: '#fff',
        primaryBorderColor: '#cbd5e0',
        lineColor: '#cbd5e0',
        secondaryColor: '#2d3748',
        tertiaryColor: '#1a202c',
        background: '#1a202c',
        mainBkg: '#4a5568',
        secondBkg: '#2d3748',
        tertiaryBkg: '#1a202c'
      }
    });
  </script>
  <style>
    body { background-color: #1a202c; }
    .mermaid { background-color: #1a202c; }
  </style>
</head>
<body>
<div class="mermaid">
flowchart LR

    %% =========================
    %% Inputs (Pre-checkpointed 12M Folds)
    %% =========================
    Folds["<b>Input</b><br/>Pre-checkpointed 12M Folds<br/>OTPW_12M_FOLD_i_{TRAIN/VAL/TEST}.parquet"]

    %% =========================
    %% Baseline Estimator Pipeline (per fold)
    %% =========================
    subgraph PIPE["<b>Baseline Estimator: Feature Pipeline (Drop Nulls)</b>"]
        direction LR

        %% Stage 1: Data Preparation
        subgraph S1["Stage 1: Data Preparation"]
            LabelClean["Cast DEP_DELAY to Double<br/>Filter Null/NaN Labels"]
            SelectFeat["Select 10 Baseline Features<br/>Temporal, Airport, Flight, Weather"]
            NumClean["Clean Numerical Features:<br/>Remove non-numeric chars<br/>Empty → Null<br/>Cast to Double"]
            LabelClean --> SelectFeat --> NumClean
        end

        %% Stage 2: Drop Null Numerical Rows
        subgraph S2["Stage 2: Drop Null Numerical Rows"]
            DropNulls["Drop rows where any of:<br/>HourlyWindSpeed, HourlyVisibility,<br/>HourlyPrecipitation, DISTANCE<br/>is Null"]
        end

        %% Stage 3: Categorical Encoding
        subgraph S3["Stage 3: Categorical Encoding"]
            DOW["DAY_OF_WEEK_clean<br/>StringIndexer + OHE"]
            Month["MONTH_clean<br/>StringIndexer + OHE"]
            TimeBlk["DEP_TIME_BLK_clean<br/>StringIndexer + OHE"]
            Origin["ORIGIN_clean<br/>StringIndexer + OHE"]
            Dest["DEST_clean<br/>StringIndexer + OHE"]
            Carrier["OP_UNIQUE_CARRIER_clean<br/>StringIndexer + OHE"]
        end

        %% Stage 4: Feature Assembly
        subgraph S4["Stage 4: Feature Assembly"]
            Assemble["VectorAssembler<br/>Combine Cleaned Numerics + Encoded Categoricals"]
        end

        %% Stage 5: Standardization
        subgraph S5["Stage 5: Standardization"]
            Scale["StandardScaler<br/>features → scaled_features<br/>withMean=True, withStd=True"]
        end

        %% Stage 6: Linear Regression
        subgraph S6["Stage 6: Model"]
            LR["Linear Regression<br/>featuresCol=scaled_features<br/>labelCol=DEP_DELAY<br/>maxIter=100<br/>regParam=0.0, elasticNet=0.0"]
        end
    end

    %% Connections inside pipeline
    S1 --> S2
    S1 --> S3
    S2 --> Assemble
    S3 --> Assemble
    Assemble --> Scale --> LR

    %% =========================
    %% Expanding Window CV Flow
    %% =========================
    subgraph CV["<b>Expanding Window Cross-Validation (12 Months)</b>"]
        direction TB
        Fold1["Fold 1:<br/>Train: Months 1–8<br/>Val: Month 9"]
        Fold2["Fold 2:<br/>Train: Months 1–9<br/>Val: Month 10"]
        Fold3["Fold 3:<br/>Train: Months 1–10<br/>Val: Month 11"]
        Fold4["Fold 4:<br/>Train: Months 1–11<br/>Val: Month 12"]
        Fold5["Fold 5:<br/>Train: Months 1–12<br/>Test: Held-out"]

        Fold1 --> Fold2 --> Fold3 --> Fold4 --> Fold5
    end

    %% =========================
    %% Metrics & Output
    %% =========================
    subgraph METRICS["<b>Metrics & Outputs</b>"]
        direction TB
        RegMetrics["Regression Metrics per Fold<br/>RMSE, MAE, R², MSE<br/>Train & Validation"]
        ClassMetrics["Classification Metrics per Fold<br/>OTPA Accuracy, OTPA F1<br/>SDDR Recall, Bucket Accuracy"]
        TestEval["Held-out Test Evaluation<br/>Best Model by Val RMSE<br/>Final Test RMSE/MAE/R²/MSE<br/>+ Test Classification Metrics"]
    end

    %% Global connections
    Folds --> PIPE
    PIPE --> CV
    CV --> METRICS

</div>
</body>
</html>
""")

# Model 5 - Cross-Validation/12-month/Dropped-Null/Custom-Data

In [0]:
# Load the 1-year joined dataset from the mount
df_1y = spark.read.parquet("/mnt/mids-w261/student-groups/Group_4_2/processed/flights_weather_joined_1y")

# View the top 10 rows and all columns
display(df_1y.limit(10))

In [0]:
# View column names only
df_1y.columns

### Model 5:Cross-Validation/12-month/Dropped-Null/Custom-Data
Using custom joined data, modified cross validation function to work with the one year data

In [0]:
"""
================================================================================
Cross-Validation for Baseline Linear Regression (12-Month Dataset)
================================================================================

Purpose:
    Implements expanding window cross-validation for flight delay prediction
    using a baseline linear regression model with comprehensive feature engineering.

Key Features:
    - Expanding window CV: Each fold trains on progressively more historical data
    - Pre-checkpointed data: Uses coworker's fold pattern (OTPW_{version}_FOLD_{i}_{TRAIN/VAL/TEST})
    - Baseline pipeline: Numeric cleaning + median imputation + categorical OHE + standardization + LR
    - Comprehensive metrics: Regression (RMSE, MAE, R², MSE) + Classification (OTPA, SDDR, 4-bucket)
    - Memory management: Strategic caching with .count() to prevent OOM errors

Data Flow:
    1. Load pre-checkpointed folds from DBFS
    2. For each fold (except last):
       - Train model on expanding training set
       - Evaluate on validation set
       - Track best model by RMSE
    3. Evaluate best model on held-out test set
    4. Report comprehensive metrics and feature importance

Author: Emily Lieske
Date: November 2025
================================================================================
"""

# ============================================================================
# Imports
# ============================================================================

# PySpark SQL functions for data transformation
from pyspark.sql.functions import col, when, isnan, regexp_replace, trim, length
from pyspark.sql.types import DoubleType, StringType

# PySpark ML for pipeline construction and modeling
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder, StandardScaler, Imputer
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

# Standard libraries for metrics and timing
import numpy as np
import pandas as pd
import time
import gc  # Garbage collection for memory management


# ============================================================================
# Data Loading Functions
# ============================================================================

def _load_checkpointed_data(name, folder_path="dbfs:/student-groups/Group_4_2"):
    """
    Load a pre-checkpointed Parquet dataset from DBFS.
    
    Args:
        name (str): Dataset name (e.g., "OTPW_12M_FOLD_1_TRAIN")
        folder_path (str): DBFS path to the folder containing checkpointed data
        
    Returns:
        pyspark.sql.DataFrame: Loaded dataset
        
    Note:
        Pre-checkpointed data significantly speeds up CV by avoiding repeated
        data loading and splitting operations.
    """
    return spark.read.parquet(f"{folder_path}/{name}.parquet")


def _build_time_folds_from_df(df, n_val_folds=3):
    """
    Build expanding-window folds directly from a single 1-year DataFrame.
    
    We use the `month` column (1–12) to create time-aware folds:
        - Fold 1: Train on months <= 8, validate on month 9
        - Fold 2: Train on months <= 9, validate on month 10
        - Fold 3: Train on months <= 10, validate on month 11
      Held-out test: Train on months <= 11, test on month 12
    
    Returns:
        folds: list of (train_df, val_df) for validation folds
        test_fold: (train_df, test_df) for final held-out evaluation
    """
    folds = []
    # Validation folds for months 8, 9, 10, 11.
    # Each fold trains on all months up to train_last_month (inclusive)
    # and validates on the next month (val_month).
    #
    # This mirrors the 4 validation folds in Model 3:
    #   Fold 1: train <= 7,  val == 8
    #   Fold 2: train <= 8,  val == 9
    #   Fold 3: train <= 9,  val == 10
    #   Fold 4: train <= 10, val == 11
    fold_specs = [(7, 8), (8, 9), (9, 10), (10, 11)]
    for train_last_month, val_month in fold_specs:
        # All history up to train_last_month becomes the training window
        train_df = df.filter(col("month") <= train_last_month)
        # The following month becomes the validation window
        val_df = df.filter(col("month") == val_month)
        folds.append((train_df, val_df))
    
    # Held-out test fold:
    # Train on all data up to month 11, and test on month 12 only.
    test_train_df = df.filter(col("month") <= 11)
    test_df = df.filter(col("month") == 12)
    test_fold = (test_train_df, test_df)
    
    return folds, test_fold


# ============================================================================
# Baseline Estimator Class
# ============================================================================

class BaselineEstimator:
    """
    Baseline Linear Regression Estimator with Feature Engineering Pipeline.
    
    This class encapsulates the entire feature engineering and modeling pipeline:
        1. Data preparation (label cleaning, feature selection)
        2. Numerical feature cleaning (remove non-numeric chars, handle nulls)
        3. Median imputation for numerical features
        4. Categorical encoding (StringIndexer + OneHotEncoder)
        5. Feature assembly and standardization
        6. Linear regression modeling
    
    Feature Families:
        - Temporal: day_of_week, month, dep_time_blk
        - Airport: origin, dest
        - Flight: op_unique_carrier, distance
        - Weather: hourlywindspeed, hourlyvisibility, hourlyprecipitation
    
    Why This Design:
        - Encapsulation: All preprocessing logic in one place
        - Reusability: Same pipeline for train/val/test
        - Spark ML compatibility: Uses Pipeline for efficient execution
    """
    
    def __init__(self, label_col="dep_delay"):
        """
        Initialize the estimator with feature definitions.
        
        Args:
            label_col (str): Name of the target variable column
        """
        self.label_col = label_col
        self.pipeline = None
        self.model = None
        
        # Categorical features: Encoded using StringIndexer + OneHotEncoder
        # These capture temporal patterns, route characteristics, and carrier effects
        self.categorical_features = [
            "day_of_week",        # Day of week (1=Monday, 7=Sunday)
            "month",              # Month of year (1-12)
            "dep_time_blk",       # Departure time block (e.g., "0600-0659")
            "origin",             # Origin airport code
            "dest",               # Destination airport code
            "op_unique_carrier"   # Operating carrier code
        ]
        
        # Numerical features: Weather conditions and flight distance
        self.numerical_features = [
            "hourlywindspeed",       # Wind speed at origin (mph)
            "hourlyvisibility",      # Visibility at origin (miles)
            "hourlyprecipitation",   # Precipitation at origin (inches)
            "distance"               # Flight distance (miles)
        ]

    def _prepare(self, df):
        """
        Prepare the DataFrame for modeling by cleaning the label and numerical features.
        
        Steps:
            1. Cast label to DoubleType (required by Spark ML)
            2. Filter out rows with null/NaN labels (Spark ML requirement)
            3. Select only required features + label
            4. Clean numerical features:
               - Remove non-numeric characters (e.g., "12.5mph" -> "12.5")
               - Convert empty strings to null
               - Cast to DoubleType
        
        Args:
            df (pyspark.sql.DataFrame): Input DataFrame
            
        Returns:
            pyspark.sql.DataFrame: Cleaned DataFrame
            
        Why This Matters:
            - Spark ML LinearRegression requires DoubleType labels with no nulls
            - Numerical features may contain string artifacts from source data
            - Explicit type casting prevents downstream pipeline errors
        """
        # Cast label to double and filter out null/NaN values
        # Spark ML does not accept null labels
        df = df.withColumn(self.label_col, col(self.label_col).cast(DoubleType()))
        df = df.filter(~(col(self.label_col).isNull() | isnan(col(self.label_col))))

        # Select only the columns we need (features + label)
        # This reduces memory footprint and prevents accidental data leakage
        selected = [c for c in (self.categorical_features + self.numerical_features + [self.label_col]) 
                    if c in df.columns]
        df = df.select(*selected)

        # Clean numerical features: remove non-numeric characters, handle empty strings
        for f in self.numerical_features:
            if f in df.columns:
                # Step 1: Cast to string to enable regex operations
                # Step 2: Remove all non-numeric characters except +, -, and .
                df = df.withColumn(f, regexp_replace(col(f).cast(StringType()), r"[^0-9+\-\.]", ""))
                
                # Step 3: Convert empty strings to null (for imputation)
                df = df.withColumn(f, when(length(trim(col(f))) == 0, None).otherwise(col(f)))
                
                # Step 4: Cast to DoubleType for modeling
                df = df.withColumn(f, col(f).cast(DoubleType()))
        
        return df

    def _build_pipeline(self, df):
        """
        Build the Spark ML Pipeline with all feature engineering stages.
        
        Pipeline Stages:
            1. Drop nulls: Remove rows with missing numerical features
            2. StringIndexers: Convert categorical strings to indices
            3. OneHotEncoders: Convert indices to binary vectors
            4. VectorAssembler: Combine all features into single vector
            5. StandardScaler: Standardize features (mean=0, std=1)
            6. LinearRegression: Train linear model
        
        Args:
            df (pyspark.sql.DataFrame): Prepared DataFrame
            
        Returns:
            pyspark.sql.DataFrame: DataFrame (may have additional columns from transformations)
            
        Why This Design:
            - Pipeline ensures consistent transformations across train/val/test
            - Dropping nulls instead of imputing for cleaner baseline
            - OneHotEncoding with dropLast=True prevents multicollinearity
            - StandardScaler improves convergence for gradient descent
            - No regularization (regParam=0) for interpretable baseline
        """
        stages = []
        
        # ========================================
        # Stage 1: Drop Null Values in Numerical Features
        # ========================================
        # Drop rows with any null values in numerical features
        # This ensures we only train on complete cases
        for f in self.numerical_features:
            if f in df.columns:
                df = df.filter(col(f).isNotNull())
        
        # Repartition after filtering to rebalance data across executors
        # Filtering can create skewed partitions (some empty, some overloaded)
        # Repartitioning ensures efficient parallel processing
        # Use 8 partitions to match available cores (2 executors × 4 cores = 8 total)
        df = df.repartition(8)
        
        # Rename numerical features to match expected naming convention
        # (no "_imputed" suffix since we're not imputing)
        for f in self.numerical_features:
            if f in df.columns:
                df = df.withColumnRenamed(f, f"{f}_imputed")

        # ========================================
        # Stage 2: Categorical Encoding (StringIndexer + OneHotEncoder)
        # ========================================
        # StringIndexer: Converts strings to numeric indices (most frequent = 0)
        # OneHotEncoder: Converts indices to binary vectors (prevents ordinal assumption)
        # handleInvalid="keep": Unseen categories in test set get their own index
        # dropLast=True: Drop last category to prevent multicollinearity
        for f in self.categorical_features:
            if f in df.columns:
                # For numeric-coded categoricals (day_of_week, month, dep_time_blk),
                # cast to string first to ensure consistent handling
                if f in ["day_of_week", "month", "dep_time_blk"]:
                    df = df.withColumn(f"{f}_clean", 
                                      when(col(f).isNull(), "UNKNOWN").otherwise(col(f).cast(StringType())))
                else:
                    # For string categoricals (ORIGIN, DEST, CARRIER), handle nulls only
                    df = df.withColumn(f"{f}_clean", 
                                      when(col(f).isNull(), "UNKNOWN").otherwise(col(f)))
                
                # Add StringIndexer stage
                stages.append(StringIndexer(inputCol=f"{f}_clean", 
                                           outputCol=f"{f}_indexed", 
                                           handleInvalid="keep"))
                
                # Add OneHotEncoder stage
                stages.append(OneHotEncoder(inputCols=[f"{f}_indexed"], 
                                           outputCols=[f"{f}_encoded"], 
                                           dropLast=True))

        # ========================================
        # Stage 3: Feature Assembly
        # ========================================
        # Combine all features (numerical + encoded categorical) into single vector
        # handleInvalid="skip": Skip rows with invalid values
        feature_columns = [f"{f}_imputed" for f in self.numerical_features if f in df.columns] + \
                          [f"{f}_encoded" for f in self.categorical_features if f in df.columns]
        assembler = VectorAssembler(inputCols=feature_columns, 
                                    outputCol="features", 
                                    handleInvalid="skip")
        
        # ========================================
        # Stage 4: Feature Standardization
        # ========================================
        # Standardize features to mean=0, std=1
        # withStd=True: Scale to unit variance
        # withMean=True: Center to zero mean
        # Why? Improves convergence and makes coefficients comparable
        scaler = StandardScaler(inputCol="features", 
                               outputCol="scaled_features", 
                               withStd=True, 
                               withMean=True)
        
        # ========================================
        # Stage 5: Linear Regression
        # ========================================
        # Baseline model: No regularization (regParam=0, elasticNetParam=0)
        # maxIter=100: Maximum iterations for convergence
        # Why no regularization? We want an interpretable baseline to understand
        # feature importance before adding complexity
        lr = LinearRegression(featuresCol="scaled_features", 
                            labelCol=self.label_col, 
                            maxIter=100, 
                            regParam=0.0, 
                            elasticNetParam=0.0)

        # Assemble all stages into pipeline
        stages.extend([assembler, scaler, lr])
        self.pipeline = Pipeline(stages=stages)
        
        return df

    def fit(self, df):
        """
        Fit the pipeline on training data.
        
        Args:
            df (pyspark.sql.DataFrame): Training DataFrame
            
        Returns:
            BaselineEstimator: self (for method chaining)
        """
        df_prep = self._prepare(df)
        df_prep = self._build_pipeline(df_prep)
        self.model = self.pipeline.fit(df_prep)
        return self

    def transform(self, df):
        """
        Transform data using the fitted pipeline.
        
        Args:
            df (pyspark.sql.DataFrame): DataFrame to transform (val/test)
            
        Returns:
            pyspark.sql.DataFrame: Transformed DataFrame with predictions
            
        Note:
            We rebuild the pipeline on the input DataFrame to ensure all
            transformation columns are present, but use the fitted model
            for predictions.
        """
        df_prep = self._prepare(df)
        df_prep = self._build_pipeline(df_prep)  # Rebuild to ensure cols present
        return self.model.transform(df_prep)


# ============================================================================
# Cross-Validation Runner
# ============================================================================

def run_cv(df_1y, include_classification_metrics=True):
    """
    Run expanding window cross-validation for the baseline model on 1-year data.
    
    Process:
        1. Build time-aware folds directly from the 1-year DataFrame (expanding window)
        2. For each validation fold:
           a. Train model on training set
           b. Evaluate on both training and validation sets
           c. Compute regression and classification metrics
           d. Track the best model (by validation RMSE)
        3. Evaluate best model on held-out test set (month = 12)
        4. Report comprehensive metrics, feature importance, and runtime
    
    Args:
        df_1y (DataFrame): Full 1-year joined dataset with `month` and `dep_delay`
        include_classification_metrics (bool): Whether to compute OTPA, SDDR, etc.
        
    Returns:
        tuple: (best_model, val_metrics_list, test_metrics_dict, metrics_df,
                metrics_defs, classification_df, test_class_results)
    
    Memory Management Strategy:
        - .cache().count(): Forces immediate materialization in controlled chunks
        - .unpersist(): Explicitly frees memory after each fold
        - This prevents OOM errors by avoiding lazy accumulation of cached data
    """
    # ========================================
    # Initialize Timer and Build Folds
    # ========================================
    overall_start_time = time.time()
    # Build time-aware (expanding window) folds from the 1-year DataFrame.
    # - folds: list of (train_df, val_df) pairs for validation months 9,10,11
    # - test_fold: (train_df, test_df) pair for held-out month 12
    folds, test_fold = _build_time_folds_from_df(df_1y)

    # ========================================
    # Initialize Evaluators
    # ========================================
    # Create evaluators for each regression metric
    # These will be reused across all folds for consistency
    eval_rmse = RegressionEvaluator(predictionCol="prediction", labelCol="dep_delay", metricName="rmse")
    eval_mae  = RegressionEvaluator(predictionCol="prediction", labelCol="dep_delay", metricName="mae")
    eval_r2   = RegressionEvaluator(predictionCol="prediction", labelCol="dep_delay", metricName="r2")
    eval_mse  = RegressionEvaluator(predictionCol="prediction", labelCol="dep_delay", metricName="mse")

    # ========================================
    # Initialize Metric Storage
    # ========================================
    metrics = []  # Validation metrics per fold (for backward compatibility)
    train_records = []  # Per-fold train metrics across months
    val_records = []    # Per-fold validation metrics across months
    classification_records = []  # Classification metrics per validation/test fold
    best_model = None  # Best model (by validation RMSE)
    best_rmse = float("inf")  # Track best validation RMSE seen so far

    # ========================================
    # Initialize Estimator
    # ========================================
    est = BaselineEstimator(label_col="dep_delay")

    # ========================================
    # Cross-Validation Loop
    # ========================================
    # Train/validate on all validation folds; a separate held-out test fold is used later.
    # Each fold corresponds to a different validation month (9, 10, 11).
    for idx, (train_df, val_df) in enumerate(folds, start=1):
        print(f"--- Fold {idx}/{len(folds)} ---")
        
        # ========================================
        # Cache DataFrames for Performance (optional)
        # ========================================
        # In this model we leave caching commented out to reduce memory pressure
        # on small clusters. Uncomment if you need more performance and have RAM.
        # ========================================
        # Train Model
        # ========================================
        model = est.fit(train_df)
        
        # ========================================
        # Generate Predictions
        # ========================================
        # Generate predictions on both validation and training sets.
        # Training metrics help detect overfitting (train << val).
        
        # Validation predictions
        val_preds = model.transform(val_df)
        # val_preds.cache().count()  # Disabled to prevent OOM
        
        # Train predictions
        train_preds = model.transform(train_df)
        # train_preds.cache().count()  # Disabled to prevent OOM

        # ========================================
        # Compute Regression Metrics (Validation)
        # ========================================
        val_rmse = eval_rmse.evaluate(val_preds)
        val_mae  = eval_mae.evaluate(val_preds)
        val_r2   = eval_r2.evaluate(val_preds)
        val_mse  = eval_mse.evaluate(val_preds)
        
        # Store validation metrics
        metrics.append({"fold": idx, "rmse": val_rmse, "mae": val_mae, "r2": val_r2, "mse": val_mse})
        val_records.append({"split": "validation", "fold": idx, "rmse": val_rmse, "mae": val_mae, "r2": val_r2, "mse": val_mse})

        # ========================================
        # Compute Regression Metrics (Training)
        # ========================================
        # Training metrics help detect overfitting
        # If train metrics >> val metrics, model is overfitting
        tr_rmse = eval_rmse.evaluate(train_preds)
        tr_mae  = eval_mae.evaluate(train_preds)
        tr_r2   = eval_r2.evaluate(train_preds)
        tr_mse  = eval_mse.evaluate(train_preds)
        train_records.append({"split": "train", "fold": idx, "rmse": tr_rmse, "mae": tr_mae, "r2": tr_r2, "mse": tr_mse})

        # ========================================
        # Track Best Model
        # ========================================
        # Select model with lowest validation RMSE
        # This model will be used for final test evaluation
        if val_rmse < best_rmse:
            best_rmse = val_rmse
            best_model = model

        # Print validation metrics
        print(f"RMSE: {val_rmse:.2f}  MAE: {val_mae:.2f}  R²: {val_r2:.4f}  MSE: {val_mse:.2f}")
        
        # ========================================
        # Compute Classification Metrics (Optional)
        # ========================================
        # Classification metrics provide operational insights:
        # - OTPA: On-Time Performance Accuracy (<15 min threshold)
        # - SDDR: Severe Delay Detection Rate (≥60 min threshold)
        # - Bucket Accuracy: 4-bucket classification (Early, OnTime, Delayed, Severe)
        if include_classification_metrics:
            val_class_results, _ = compute_classification_metrics(val_preds)
            classification_records.append({
                "split": "validation",
                "fold": idx,
                "otpa_accuracy": val_class_results["otpa_accuracy"],
                "otpa_f1": val_class_results["otpa_f1"],
                "sddr_recall": val_class_results["sddr_recall"],
                "bucket_accuracy": val_class_results["bucket_accuracy"]
            })
            print(f"  OTPA Acc: {val_class_results['otpa_accuracy']:.4f}  "
                  f"OTPA F1: {val_class_results['otpa_f1']:.4f}  "
                  f"SDDR: {val_class_results['sddr_recall']:.4f}")
        
        # ========================================
        # Free Memory
        # ========================================
        # Explicitly unpersist cached DataFrames to free memory
        # Critical for preventing OOM in subsequent folds
        # Disabled since caching is disabled
        # train_preds.unpersist()
        # val_preds.unpersist()
        # train_df.unpersist()
        # val_df.unpersist()
        
        # Force Python garbage collection to free driver memory
        gc.collect()
        
        # Clear Spark SQL cache to free executor memory
        spark.catalog.clearCache()

    # ========================================
    # Validation Summary
    # ========================================
    # Report aggregate statistics across all validation folds
    # This helps assess model stability and generalization
    print("-------------------------------")
    print("Validation Summary:")
    print(f"Best RMSE: {best_rmse:.2f}")
    print(f"Mean RMSE: {np.mean([m['rmse'] for m in metrics]):.2f}  |  Std: {np.std([m['rmse'] for m in metrics]):.2f}")
    print(f"Mean MAE:  {np.mean([m['mae'] for m in metrics]):.2f}")
    print(f"Mean R²:   {np.mean([m['r2'] for m in metrics]):.4f}")
    print(f"Mean MSE:  {np.mean([m['mse'] for m in metrics]):.2f}")

    # ========================================
    # Held-Out Test Evaluation
    # ========================================
    # Evaluate best model on held-out test set (month = 12)
    # This provides an unbiased estimate of production performance
    train_df, test_df = test_fold
    # test_df.cache().count()  # Disabled to prevent OOM
    
    # Generate predictions on test set
    test_preds = best_model.transform(test_df)
    # test_preds.cache().count()  # Disabled to prevent OOM
    
    # Compute regression metrics on test set
    test_rmse = eval_rmse.evaluate(test_preds)
    test_mae  = eval_mae.evaluate(test_preds)
    test_r2   = eval_r2.evaluate(test_preds)
    test_mse  = eval_mse.evaluate(test_preds)
    
    # Free memory
    # Disabled since caching is disabled
    # test_preds.unpersist()
    # test_df.unpersist()

    # Print test results
    print("-------------------------------")
    print("Held-out Test:")
    print(f"RMSE: {test_rmse:.2f}  MAE: {test_mae:.2f}  R²: {test_r2:.4f}  MSE: {test_mse:.2f}")
    
    # ========================================
    # Test Set Classification Metrics
    # ========================================
    # Compute classification metrics for test set if requested
    # Provides operational insights for deployment decisions
    if include_classification_metrics:
        test_class_results, test_class_df = compute_classification_metrics(test_preds)
        print_classification_summary(test_class_results, test_class_df)
        classification_records.append({
            "split": "test",
            "fold": len(folds) + 1,  # treat held-out test as the 5th fold
            "otpa_accuracy": test_class_results["otpa_accuracy"],
            "otpa_f1": test_class_results["otpa_f1"],
            "sddr_recall": test_class_results["sddr_recall"],
            "bucket_accuracy": test_class_results["bucket_accuracy"]
        })

    # ========================================
    # Build Consolidated Metrics DataFrames
    # ========================================
    # Combine train/val/test metrics into single DataFrame for easy comparison
    # Use fold index len(folds)+1 (=5) for the held-out test split to mirror 5-fold CV layout.
    test_record = [{"split": "test", "fold": len(folds) + 1, "rmse": test_rmse, "mae": test_mae, "r2": test_r2, "mse": test_mse}]
    metrics_df = pd.DataFrame(train_records + val_records + test_record)
    
    # Build classification metrics DataFrame if computed
    classification_df = pd.DataFrame(classification_records) if include_classification_metrics else None
    
    # Round to 4 decimal places for readability
    metrics_df['rmse'] = metrics_df['rmse'].round(4)
    metrics_df['mae'] = metrics_df['mae'].round(4)
    metrics_df['r2'] = metrics_df['r2'].round(4)
    metrics_df['mse'] = metrics_df['mse'].round(4)

    # ========================================
    # Metrics Definitions with LaTeX
    # ========================================
    # Provide LaTeX equations for each metric for documentation/reporting
    metrics_defs = pd.DataFrame([
        {"metric": "RMSE", "latex": r"\\mathrm{RMSE} = \\sqrt{\\tfrac{1}{N} \\sum_{i=1}^{N} (y_i - \\hat{y}_i)^2}"},
        {"metric": "MAE",  "latex": r"\\mathrm{MAE} = \\tfrac{1}{N} \\sum_{i=1}^{N} |y_i - \\hat{y}_i|"},
        {"metric": "R^2",  "latex": r"R^2 = 1 - \\dfrac{\\sum_{i=1}^{N} (y_i - \\hat{y}_i)^2}{\\sum_{i=1}^{N} (y_i - \\bar{y})^2}"},
        {"metric": "MSE",  "latex": r"\\mathrm{MSE} = \\tfrac{1}{N} \\sum_{i=1}^{N} (y_i - \\hat{y}_i)^2"},
    ])

    # ========================================
    # Extract Feature Importance (Coefficients)
    # ========================================
    # Extract and display top 10 most important features by absolute coefficient
    # Helps understand which features drive predictions
    try:
        # Extract coefficients from the linear regression model (last stage in pipeline)
        coefficients = best_model.model.stages[-1].coefficients.toArray()
        
        # Try to extract feature names from metadata (includes OHE expansions)
        feats_meta = test_preds.schema["features"].metadata
        attrs = []
        if "ml_attr" in feats_meta and "attrs" in feats_meta["ml_attr"]:
            ml_attrs = feats_meta["ml_attr"]["attrs"]
            # Collect attributes from all types (binary, numeric, nominal)
            for t in ["binary", "numeric", "nominal"]:
                if t in ml_attrs:
                    attrs.extend(ml_attrs[t])
            # Sort by vector index to match coefficient order
            attrs = sorted(attrs, key=lambda x: x["idx"])
            feature_names = [a.get("name", f"feature_{a['idx']}") for a in attrs]
        else:
            # Fallback: use generic names if metadata unavailable
            feature_names = [f"feature_{i}" for i in range(len(coefficients))]
        
        # Safety check: ensure lengths match
        if len(feature_names) != len(coefficients):
            feature_names = [f"feature_{i}" for i in range(len(coefficients))]
        
        # Create DataFrame with feature names and coefficients
        coef_df = pd.DataFrame({"feature": feature_names, "coefficient": coefficients})
        coef_df["abs_coefficient"] = coef_df["coefficient"].abs()
        coef_df = coef_df.sort_values("abs_coefficient", ascending=False)
        
        # Round coefficients to 4 decimal places
        top_coef_df = coef_df.head(10).copy()
        top_coef_df['coefficient'] = top_coef_df['coefficient'].round(4)
        top_coef_df['abs_coefficient'] = top_coef_df['abs_coefficient'].round(4)
        
        print("\nTop 10 Most Important Features (by absolute coefficient):")
        print(top_coef_df.to_string(index=False))
    except Exception as e:
        print(f"[Info] Skipping coefficient listing: {e}")
    
    # ========================================
    # Print Total Runtime
    # ========================================
    # Report total execution time for performance tracking
    overall_elapsed = time.time() - overall_start_time
    hours, remainder = divmod(overall_elapsed, 3600)
    minutes, seconds = divmod(remainder, 60)
    print("\n" + "="*80)
    print(f"TOTAL RUNTIME: {int(hours)}h {int(minutes)}m {seconds:.2f}s ({overall_elapsed:.2f} seconds)")
    print("="*80)

    # ========================================
    # Return Results
    # ========================================
    return best_model, metrics, {"rmse": test_rmse, "mae": test_mae, "r2": test_r2, "mse": test_mse}, metrics_df, metrics_defs, classification_df, test_class_results if include_classification_metrics else None


# ============================================================================
# Main Execution Block
# ============================================================================

if __name__ == "__main__":
    """
    Main execution entry point for the cross-validation script.
    
    Workflow:
        1. Print cluster configuration (for reproducibility and debugging)
        2. Run expanding window cross-validation
        3. Display comprehensive results (regression + classification metrics)
    
    Note:
        This block only runs when the script is executed directly,
        not when imported as a module.
    """
    
    # ========================================
    # Print Cluster Configuration
    # ========================================
    # Display cluster resources to ensure adequate capacity and enable reproducibility
    # Important for:
    # - Debugging OOM errors (check if cluster is undersized)
    # - Reproducing results (document exact cluster configuration)
    # - Cost tracking (understand resource usage)
    
    sc = spark.sparkContext
    
    # Get number of executors (exclude driver node)
    # statusTracker().getExecutorInfos() returns list of all executors
    num_executors = len(sc._jsc.sc().statusTracker().getExecutorInfos()) - 1
    
    # Get executor configuration from Spark conf
    executor_memory = sc.getConf().get("spark.executor.memory", "Unknown")
    executor_cores = sc.getConf().get("spark.executor.cores", "Unknown")
    actual_cores = sc.defaultParallelism / num_executors if num_executors > 0 else "Unknown"
    print(f"Estimated cores per executor: {int(actual_cores)}")
    
    # Print cluster configuration
    print("\n" + "="*80)
    print("CLUSTER CONFIGURATION")
    print("="*80)
    print(f"Cluster Size: {num_executors} executors, {executor_cores} cores per executor, {executor_memory} RAM per executor")
    print("="*80 + "\n")
    
    # ========================================
    # Load 1-year joined dataset
    # ========================================
    df_1y = spark.read.parquet("/mnt/mids-w261/student-groups/Group_4_2/processed/flights_weather_joined_1y")
    
    # Optional: select only the columns needed by the baseline estimator
    # (day_of_week, month, dep_time_blk, origin, dest, op_unique_carrier,
    #  hourlywindspeed, hourlyvisibility, hourlyprecipitation, distance, dep_delay)
    df_1y = df_1y.select(
        "day_of_week",
        "month",
        "dep_time_blk",
        "origin",
        "dest",
        "op_unique_carrier",
        "hourlywindspeed",
        "hourlyvisibility",
        "hourlyprecipitation",
        "distance",
        "dep_delay",
    )
    
    # ========================================
    # Run Cross-Validation
    # ========================================
    # Execute expanding window CV with comprehensive metrics on 1-year data
    # include_classification_metrics=True: Compute OTPA, SDDR, bucket accuracy
    
    best_model, val_metrics_list, test_metrics_dict, metrics_df, metrics_defs, classification_df, test_class_results = run_cv(
        df_1y,
        include_classification_metrics=True,
    )
    
    # ========================================
    # Display Regression Metrics
    # ========================================
    # Show comprehensive regression metrics across train/val/test splits
    # metrics_df: Per-fold metrics (split, fold, RMSE, MAE, R², MSE)
    # metrics_defs: LaTeX equations for each metric (for documentation)
    
    print("\n" + "="*80)
    print("REGRESSION METRICS")
    print("="*80)
    display(metrics_df)  # Use display() for interactive Databricks table
    display(metrics_defs)
    
    # ========================================
    # Display Classification Metrics
    # ========================================
    # Show operational classification metrics if computed
    # Includes OTPA (on-time accuracy), SDDR (severe delay detection),
    # and 4-bucket classification accuracy
    
    if classification_df is not None:
        print("\n" + "="*80)
        print("CLASSIFICATION METRICS SUMMARY")
        print("="*80)
        display(classification_df)  # Use display() for interactive Databricks table

In [0]:
displayHTML("""
<!DOCTYPE html>
<html>
<head>
  <script src="https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.min.js"></script>
  <script>
    mermaid.initialize({
      startOnLoad: true,
      theme: 'dark',
      themeVariables: {
        primaryColor: '#4a5568',
        primaryTextColor: '#fff',
        primaryBorderColor: '#cbd5e0',
        lineColor: '#cbd5e0',
        secondaryColor: '#2d3748',
        tertiaryColor: '#1a202c',
        background: '#1a202c',
        mainBkg: '#4a5568',
        secondBkg: '#2d3748',
        tertiaryBkg: '#1a202c'
      }
    });
  </script>
  <style>
    body { background-color: #1a202c; }
    .mermaid { background-color: #1a202c; }
  </style>
</head>
<body>
<div class="mermaid">
flowchart LR

    %% Input
    Input["<b>Input</b><br/>1-Year Joined DataFrame<br/>10 Features + Label (dep_delay)"]

    %% Stage 1: Data Preparation
    subgraph S1["<b>Stage 1: Data Preparation</b>"]
        LabelClean["Cast dep_delay to Double<br/>Filter Null/NaN Labels"]
        SelectFeat["Select 10 Baseline Features<br/>Temporal, Airport, Flight, Weather"]
        NumClean["Clean Numerical Features:<br/>Remove non-numeric chars<br/>Empty → Null<br/>Cast to Double"]
        LabelClean --> SelectFeat --> NumClean
    end

    %% Stage 2: Drop Null Numerical Rows (No Imputation)
    subgraph S2["<b>Stage 2: Drop Null Numerical Rows</b>"]
        DropNulls["Filter rows where any of:<br/>hourlywindspeed, hourlyvisibility,<br/>hourlyprecipitation, distance<br/>is Null"]
    end

    %% Stage 3: Categorical Encoding
    subgraph S3["<b>Stage 3: Categorical Encoding</b>"]
        DOW["day_of_week_clean<br/>StringIndexer + OHE"]
        Month["month_clean<br/>StringIndexer + OHE"]
        TimeBlk["dep_time_blk_clean<br/>StringIndexer + OHE"]
        Origin["origin_clean<br/>StringIndexer + OHE"]
        Dest["dest_clean<br/>StringIndexer + OHE"]
        Carrier["op_unique_carrier_clean<br/>StringIndexer + OHE"]
    end

    %% Stage 4: Feature Assembly
    subgraph S4["<b>Stage 4: Feature Assembly</b>"]
        Assemble["VectorAssembler<br/>Combine Numerical + Encoded Categorical<br/><br/>Numerical inputs:<br/>hourlywindspeed_imputed,<br/>hourlyvisibility_imputed,<br/>hourlyprecipitation_imputed,<br/>distance_imputed"]
    end

    %% Stage 5: Standardization
    subgraph S5["<b>Stage 5: Standardization</b>"]
        Scale["StandardScaler<br/>features → scaled_features<br/>withMean=True, withStd=True"]
    end

    %% Stage 6: Linear Regression
    subgraph S6["<b>Stage 6: Model</b>"]
        LR["Linear Regression<br/>featuresCol=scaled_features<br/>labelCol=dep_delay<br/>maxIter=100<br/>regParam=0.0, elasticNet=0.0"]
    end

    %% Output
    Output["<b>Output</b><br/>Predictions<br/>dep_delay in minutes"]

    %% Connections
    Input --> S1
    S1 --> S2
    S1 --> S3
    S2 --> Assemble
    S3 --> Assemble
    Assemble --> Scale --> LR --> Output

</div>
</body>
</html>
""")

# Model 5 - Cross-Validation/12-month/Data-Imputed/Custom-Data
updated to 2015 data and using imputation


In [0]:
"""
================================================================================
Cross-Validation for Baseline Linear Regression (2015 Joined Dataset)
================================================================================

Purpose:
    Implements expanding window cross-validation for flight delay prediction
    using a baseline linear regression model with comprehensive feature engineering.

Key Features:
    - Expanding window CV: Each fold trains on progressively more historical data
    - Direct use of 2015 joined flights+weather data (no pre-checkpointed folds)
    - Baseline pipeline: Numeric cleaning + median imputation + categorical OHE + standardization + LR
    - Comprehensive metrics: Regression (RMSE, MAE, R², MSE) + Classification (OTPA, SDDR, 4-bucket)
    - Memory management: Strategic caching (optional) and explicit GC to avoid OOM

Data Flow:
    1. Load 2015 joined dataset from DBFS
    2. Build time-aware expanding folds using the `month` column
    3. For each fold (except last):
       - Train model on expanding training set
       - Evaluate on validation set
       - Track best model by RMSE
    4. Evaluate best model on held-out test set
    5. Report comprehensive metrics and feature importance

Author: Emily Lieske
Date: November 2025
================================================================================
"""

# ============================================================================
# Imports
# ============================================================================

# PySpark SQL functions for data transformation
from pyspark.sql.functions import col, when, isnan, regexp_replace, trim, length
from pyspark.sql.types import DoubleType, StringType

# PySpark ML for pipeline construction and modeling
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder, StandardScaler, Imputer
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

# Standard libraries for metrics and timing
import numpy as np
import pandas as pd
import time
import gc  # Garbage collection for memory management


# ============================================================================
# Data Loading Functions
# ============================================================================

def _load_checkpointed_data(name, folder_path="dbfs:/student-groups/Group_4_2"):
    """
    Load a pre-checkpointed Parquet dataset from DBFS.
    
    Args:
        name (str): Dataset name (e.g., "OTPW_12M_FOLD_1_TRAIN")
        folder_path (str): DBFS path to the folder containing checkpointed data
        
    Returns:
        pyspark.sql.DataFrame: Loaded dataset
        
    Note:
        Pre-checkpointed data significantly speeds up CV by avoiding repeated
        data loading and splitting operations.
    """
    return spark.read.parquet(f"{folder_path}/{name}.parquet")


def _build_time_folds_from_df(df, n_val_folds=3):
    """
    Build expanding-window folds directly from a single 1-year DataFrame.
    
    We use the `month` column (1–12) to create time-aware folds:
        - Fold 1: Train on months <= 8, validate on month 9
        - Fold 2: Train on months <= 9, validate on month 10
        - Fold 3: Train on months <= 10, validate on month 11
      Held-out test: Train on months <= 11, test on month 12
    
    Returns:
        folds: list of (train_df, val_df) for validation folds
        test_fold: (train_df, test_df) for final held-out evaluation
    """
    folds = []
    # Validation folds for months 8, 9, 10, 11.
    # Each fold trains on all months up to train_last_month (inclusive)
    # and validates on the next month (val_month).
    #
    # This mirrors the 4 validation folds in Model 3:
    #   Fold 1: train <= 7,  val == 8
    #   Fold 2: train <= 8,  val == 9
    #   Fold 3: train <= 9,  val == 10
    #   Fold 4: train <= 10, val == 11
    fold_specs = [(7, 8), (8, 9), (9, 10), (10, 11)]
    for train_last_month, val_month in fold_specs:
        # All history up to train_last_month becomes the training window
        train_df = df.filter(col("month") <= train_last_month)
        # The following month becomes the validation window
        val_df = df.filter(col("month") == val_month)
        folds.append((train_df, val_df))
    
    # Held-out test fold:
    # Train on all data up to month 11, and test on month 12 only.
    test_train_df = df.filter(col("month") <= 11)
    test_df = df.filter(col("month") == 12)
    test_fold = (test_train_df, test_df)
    
    return folds, test_fold


# ============================================================================
# Baseline Estimator Class
# ============================================================================

class BaselineEstimator:
    """
    Baseline Linear Regression Estimator with Feature Engineering Pipeline.
    
    This class encapsulates the entire feature engineering and modeling pipeline:
        1. Data preparation (label cleaning, feature selection)
        2. Numerical feature cleaning (remove non-numeric chars, handle nulls)
        3. Median imputation for numerical features
        4. Categorical encoding (StringIndexer + OneHotEncoder)
        5. Feature assembly and standardization
        6. Linear regression modeling
    
    Feature Families:
        - Temporal: day_of_week, month, dep_time_blk
        - Airport: origin, dest
        - Flight: op_unique_carrier, distance
        - Weather: hourlywindspeed, hourlyvisibility, hourlyprecipitation
    
    Why This Design:
        - Encapsulation: All preprocessing logic in one place
        - Reusability: Same pipeline for train/val/test
        - Spark ML compatibility: Uses Pipeline for efficient execution
    """
    
    def __init__(self, label_col="dep_delay"):
        """
        Initialize the estimator with feature definitions.
        
        Args:
            label_col (str): Name of the target variable column
        """
        self.label_col = label_col
        self.pipeline = None
        self.model = None
        
        # Categorical features: Encoded using StringIndexer + OneHotEncoder
        # These capture temporal patterns, route characteristics, and carrier effects
        self.categorical_features = [
            "day_of_week",        # Day of week (1=Monday, 7=Sunday)
            "month",              # Month of year (1-12)
            "dep_time_blk",       # Departure time block (e.g., "0600-0659")
            "origin",             # Origin airport code
            "dest",               # Destination airport code
            "op_unique_carrier"   # Operating carrier code
        ]
        
        # Numerical features: Weather conditions and flight distance
        self.numerical_features = [
            "hourlywindspeed",       # Wind speed at origin (mph)
            "hourlyvisibility",      # Visibility at origin (miles)
            "hourlyprecipitation",   # Precipitation at origin (inches)
            "distance"               # Flight distance (miles)
        ]

    def _prepare(self, df):
        """
        Prepare the DataFrame for modeling by cleaning the label and numerical features.
        
        Steps:
            1. Cast label to DoubleType (required by Spark ML)
            2. Filter out rows with null/NaN labels (Spark ML requirement)
            3. Select only required features + label
            4. Clean numerical features:
               - Remove non-numeric characters (e.g., "12.5mph" -> "12.5")
               - Convert empty strings to null
               - Cast to DoubleType
        
        Args:
            df (pyspark.sql.DataFrame): Input DataFrame
            
        Returns:
            pyspark.sql.DataFrame: Cleaned DataFrame
            
        Why This Matters:
            - Spark ML LinearRegression requires DoubleType labels with no nulls
            - Numerical features may contain string artifacts from source data
            - Explicit type casting prevents downstream pipeline errors
        """
        # Cast label to double and filter out null/NaN values
        # Spark ML does not accept null labels
        df = df.withColumn(self.label_col, col(self.label_col).cast(DoubleType()))
        df = df.filter(~(col(self.label_col).isNull() | isnan(col(self.label_col))))

        # Select only the columns we need (features + label)
        # This reduces memory footprint and prevents accidental data leakage
        selected = [c for c in (self.categorical_features + self.numerical_features + [self.label_col]) 
                    if c in df.columns]
        df = df.select(*selected)

        # Clean numerical features: remove non-numeric characters, handle empty strings
        for f in self.numerical_features:
            if f in df.columns:
                # Step 1: Cast to string to enable regex operations
                # Step 2: Remove all non-numeric characters except +, -, and .
                df = df.withColumn(f, regexp_replace(col(f).cast(StringType()), r"[^0-9+\-\.]", ""))
                
                # Step 3: Convert empty strings to null (for imputation)
                df = df.withColumn(f, when(length(trim(col(f))) == 0, None).otherwise(col(f)))
                
                # Step 4: Cast to DoubleType for modeling
                df = df.withColumn(f, col(f).cast(DoubleType()))
        
        return df

    def _build_pipeline(self, df):
        """
        Build the Spark ML Pipeline with all feature engineering stages.
        
        Pipeline Stages:
            1. Median Imputer: Fill missing values in numerical features
            2. StringIndexers: Convert categorical strings to indices
            3. OneHotEncoders: Convert indices to binary vectors
            4. VectorAssembler: Combine all features into single vector
            5. StandardScaler: Standardize features (mean=0, std=1)
            6. LinearRegression: Train linear model
        
        Args:
            df (pyspark.sql.DataFrame): Prepared DataFrame
            
        Returns:
            pyspark.sql.DataFrame: DataFrame (may have additional columns from transformations)
            
        Why This Design:
            - Pipeline ensures consistent transformations across train/val/test
            - Median imputation preserves more rows than dropping nulls
            - OneHotEncoding with dropLast=True prevents multicollinearity
            - StandardScaler improves convergence for gradient descent
            - No regularization (regParam=0) for interpretable baseline
        """
        stages = []
        
        # ========================================
        # Stage 1: Median Imputation for Numerical Features
        # ========================================
        # Use Spark's Imputer to fill missing numerical values with the median.
        # This retains more data than dropping rows with nulls.
        input_cols = [f for f in self.numerical_features if f in df.columns]
        output_cols = [f"{f}_imputed" for f in input_cols]
        if input_cols:
            imputer = Imputer(
                strategy="median",
                inputCols=input_cols,
                outputCols=output_cols,
            )
            stages.append(imputer)

        # ========================================
        # Stage 2: Categorical Encoding (StringIndexer + OneHotEncoder)
        # ========================================
        # StringIndexer: Converts strings to numeric indices (most frequent = 0)
        # OneHotEncoder: Converts indices to binary vectors (prevents ordinal assumption)
        # handleInvalid="keep": Unseen categories in test set get their own index
        # dropLast=True: Drop last category to prevent multicollinearity
        for f in self.categorical_features:
            if f in df.columns:
                # For numeric-coded categoricals (day_of_week, month, dep_time_blk),
                # cast to string first to ensure consistent handling
                if f in ["day_of_week", "month", "dep_time_blk"]:
                    df = df.withColumn(f"{f}_clean", 
                                      when(col(f).isNull(), "UNKNOWN").otherwise(col(f).cast(StringType())))
                else:
                    # For string categoricals (ORIGIN, DEST, CARRIER), handle nulls only
                    df = df.withColumn(f"{f}_clean", 
                                      when(col(f).isNull(), "UNKNOWN").otherwise(col(f)))
                
                # Add StringIndexer stage
                stages.append(StringIndexer(inputCol=f"{f}_clean", 
                                           outputCol=f"{f}_indexed", 
                                           handleInvalid="keep"))
                
                # Add OneHotEncoder stage
                stages.append(OneHotEncoder(inputCols=[f"{f}_indexed"], 
                                           outputCols=[f"{f}_encoded"], 
                                           dropLast=True))

        # ========================================
        # Stage 3: Feature Assembly
        # ========================================
        # Combine all features (numerical + encoded categorical) into single vector
        # handleInvalid="skip": Skip rows with invalid values
        feature_columns = [f"{f}_imputed" for f in self.numerical_features if f in df.columns] + \
                          [f"{f}_encoded" for f in self.categorical_features if f in df.columns]
        assembler = VectorAssembler(inputCols=feature_columns, 
                                    outputCol="features", 
                                    handleInvalid="skip")
        
        # ========================================
        # Stage 4: Feature Standardization
        # ========================================
        # Standardize features to mean=0, std=1
        # withStd=True: Scale to unit variance
        # withMean=True: Center to zero mean
        # Why? Improves convergence and makes coefficients comparable
        scaler = StandardScaler(inputCol="features", 
                               outputCol="scaled_features", 
                               withStd=True, 
                               withMean=True)
        
        # ========================================
        # Stage 5: Linear Regression
        # ========================================
        # Baseline model: No regularization (regParam=0, elasticNetParam=0)
        # maxIter=100: Maximum iterations for convergence
        # Why no regularization? We want an interpretable baseline to understand
        # feature importance before adding complexity
        lr = LinearRegression(featuresCol="scaled_features", 
                            labelCol=self.label_col, 
                            maxIter=100, 
                            regParam=0.0, 
                            elasticNetParam=0.0)

        # Assemble all stages into pipeline
        stages.extend([assembler, scaler, lr])
        self.pipeline = Pipeline(stages=stages)
        
        return df

    def fit(self, df):
        """
        Fit the pipeline on training data.
        
        Args:
            df (pyspark.sql.DataFrame): Training DataFrame
            
        Returns:
            BaselineEstimator: self (for method chaining)
        """
        df_prep = self._prepare(df)
        df_prep = self._build_pipeline(df_prep)
        self.model = self.pipeline.fit(df_prep)
        return self

    def transform(self, df):
        """
        Transform data using the fitted pipeline.
        
        Args:
            df (pyspark.sql.DataFrame): DataFrame to transform (val/test)
            
        Returns:
            pyspark.sql.DataFrame: Transformed DataFrame with predictions
            
        Note:
            We rebuild the pipeline on the input DataFrame to ensure all
            transformation columns are present, but use the fitted model
            for predictions.
        """
        df_prep = self._prepare(df)
        df_prep = self._build_pipeline(df_prep)  # Rebuild to ensure cols present
        return self.model.transform(df_prep)


# ============================================================================
# Cross-Validation Runner
# ============================================================================

def run_cv(df_1y, include_classification_metrics=True):
    """
    Run expanding window cross-validation for the baseline model on 1-year data.
    
    Process:
        1. Build time-aware folds directly from the 1-year DataFrame (expanding window)
        2. For each validation fold:
           a. Train model on training set
           b. Evaluate on both training and validation sets
           c. Compute regression and classification metrics
           d. Track the best model (by validation RMSE)
        3. Evaluate best model on held-out test set (month = 12)
        4. Report comprehensive metrics, feature importance, and runtime
    
    Args:
        df_1y (DataFrame): Full 1-year joined dataset with `month` and `dep_delay`
        include_classification_metrics (bool): Whether to compute OTPA, SDDR, etc.
        
    Returns:
        tuple: (best_model, val_metrics_list, test_metrics_dict, metrics_df,
                metrics_defs, classification_df, test_class_results)
    
    Memory Management Strategy:
        - .cache().count(): Forces immediate materialization in controlled chunks
        - .unpersist(): Explicitly frees memory after each fold
        - This prevents OOM errors by avoiding lazy accumulation of cached data
    """
    # ========================================
    # Initialize Timer and Build Folds
    # ========================================
    overall_start_time = time.time()
    # Build time-aware (expanding window) folds from the 1-year DataFrame.
    # - folds: list of (train_df, val_df) pairs for validation months 9,10,11
    # - test_fold: (train_df, test_df) pair for held-out month 12
    folds, test_fold = _build_time_folds_from_df(df_1y)

    # ========================================
    # Initialize Evaluators
    # ========================================
    # Create evaluators for each regression metric
    # These will be reused across all folds for consistency
    eval_rmse = RegressionEvaluator(predictionCol="prediction", labelCol="dep_delay", metricName="rmse")
    eval_mae  = RegressionEvaluator(predictionCol="prediction", labelCol="dep_delay", metricName="mae")
    eval_r2   = RegressionEvaluator(predictionCol="prediction", labelCol="dep_delay", metricName="r2")
    eval_mse  = RegressionEvaluator(predictionCol="prediction", labelCol="dep_delay", metricName="mse")

    # ========================================
    # Initialize Metric Storage
    # ========================================
    metrics = []  # Validation metrics per fold (for backward compatibility)
    train_records = []  # Per-fold train metrics across months
    val_records = []    # Per-fold validation metrics across months
    classification_records = []  # Classification metrics per validation/test fold
    best_model = None  # Best model (by validation RMSE)
    best_rmse = float("inf")  # Track best validation RMSE seen so far

    # ========================================
    # Initialize Estimator
    # ========================================
    est = BaselineEstimator(label_col="dep_delay")

    # ========================================
    # Cross-Validation Loop
    # ========================================
    # Train/validate on all validation folds; a separate held-out test fold is used later.
    # Each fold corresponds to a different validation month (9, 10, 11).
    for idx, (train_df, val_df) in enumerate(folds, start=1):
        print(f"--- Fold {idx}/{len(folds)} ---")
        
        # ========================================
        # Cache DataFrames for Performance (optional)
        # ========================================
        # In this model we leave caching commented out to reduce memory pressure
        # on small clusters. Uncomment if you need more performance and have RAM.
        # ========================================
        # Train Model
        # ========================================
        model = est.fit(train_df)
        
        # ========================================
        # Generate Predictions
        # ========================================
        # Generate predictions on both validation and training sets.
        # Training metrics help detect overfitting (train << val).
        
        # Validation predictions
        val_preds = model.transform(val_df)
        # val_preds.cache().count()  # Disabled to prevent OOM
        
        # Train predictions
        train_preds = model.transform(train_df)
        # train_preds.cache().count()  # Disabled to prevent OOM

        # ========================================
        # Compute Regression Metrics (Validation)
        # ========================================
        val_rmse = eval_rmse.evaluate(val_preds)
        val_mae  = eval_mae.evaluate(val_preds)
        val_r2   = eval_r2.evaluate(val_preds)
        val_mse  = eval_mse.evaluate(val_preds)
        
        # Store validation metrics
        metrics.append({"fold": idx, "rmse": val_rmse, "mae": val_mae, "r2": val_r2, "mse": val_mse})
        val_records.append({"split": "validation", "fold": idx, "rmse": val_rmse, "mae": val_mae, "r2": val_r2, "mse": val_mse})

        # ========================================
        # Compute Regression Metrics (Training)
        # ========================================
        # Training metrics help detect overfitting
        # If train metrics >> val metrics, model is overfitting
        tr_rmse = eval_rmse.evaluate(train_preds)
        tr_mae  = eval_mae.evaluate(train_preds)
        tr_r2   = eval_r2.evaluate(train_preds)
        tr_mse  = eval_mse.evaluate(train_preds)
        train_records.append({"split": "train", "fold": idx, "rmse": tr_rmse, "mae": tr_mae, "r2": tr_r2, "mse": tr_mse})

        # ========================================
        # Track Best Model
        # ========================================
        # Select model with lowest validation RMSE
        # This model will be used for final test evaluation
        if val_rmse < best_rmse:
            best_rmse = val_rmse
            best_model = model

        # Print validation metrics
        print(f"RMSE: {val_rmse:.2f}  MAE: {val_mae:.2f}  R²: {val_r2:.4f}  MSE: {val_mse:.2f}")
        
        # ========================================
        # Compute Classification Metrics (Optional)
        # ========================================
        # Classification metrics provide operational insights:
        # - OTPA: On-Time Performance Accuracy (<15 min threshold)
        # - SDDR: Severe Delay Detection Rate (≥60 min threshold)
        # - Bucket Accuracy: 4-bucket classification (Early, OnTime, Delayed, Severe)
        if include_classification_metrics:
            val_class_results, _ = compute_classification_metrics(val_preds)
            classification_records.append({
                "split": "validation",
                "fold": idx,
                "otpa_accuracy": val_class_results["otpa_accuracy"],
                "otpa_f1": val_class_results["otpa_f1"],
                "sddr_recall": val_class_results["sddr_recall"],
                "bucket_accuracy": val_class_results["bucket_accuracy"]
            })
            print(f"  OTPA Acc: {val_class_results['otpa_accuracy']:.4f}  "
                  f"OTPA F1: {val_class_results['otpa_f1']:.4f}  "
                  f"SDDR: {val_class_results['sddr_recall']:.4f}")
        
        # ========================================
        # Free Memory
        # ========================================
        # Explicitly unpersist cached DataFrames to free memory
        # Critical for preventing OOM in subsequent folds
        # Disabled since caching is disabled
        # train_preds.unpersist()
        # val_preds.unpersist()
        # train_df.unpersist()
        # val_df.unpersist()
        
        # Force Python garbage collection to free driver memory
        gc.collect()
        
        # Clear Spark SQL cache to free executor memory
        spark.catalog.clearCache()

    # ========================================
    # Validation Summary
    # ========================================
    # Report aggregate statistics across all validation folds
    # This helps assess model stability and generalization
    print("-------------------------------")
    print("Validation Summary:")
    print(f"Best RMSE: {best_rmse:.2f}")
    print(f"Mean RMSE: {np.mean([m['rmse'] for m in metrics]):.2f}  |  Std: {np.std([m['rmse'] for m in metrics]):.2f}")
    print(f"Mean MAE:  {np.mean([m['mae'] for m in metrics]):.2f}")
    print(f"Mean R²:   {np.mean([m['r2'] for m in metrics]):.4f}")
    print(f"Mean MSE:  {np.mean([m['mse'] for m in metrics]):.2f}")

    # ========================================
    # Held-Out Test Evaluation
    # ========================================
    # Evaluate best model on held-out test set (month = 12)
    # This provides an unbiased estimate of production performance
    train_df, test_df = test_fold
    # test_df.cache().count()  # Disabled to prevent OOM
    
    # Generate predictions on test set
    test_preds = best_model.transform(test_df)
    # test_preds.cache().count()  # Disabled to prevent OOM
    
    # Compute regression metrics on test set
    test_rmse = eval_rmse.evaluate(test_preds)
    test_mae  = eval_mae.evaluate(test_preds)
    test_r2   = eval_r2.evaluate(test_preds)
    test_mse  = eval_mse.evaluate(test_preds)
    
    # Free memory
    # Disabled since caching is disabled
    # test_preds.unpersist()
    # test_df.unpersist()

    # Print test results
    print("-------------------------------")
    print("Held-out Test:")
    print(f"RMSE: {test_rmse:.2f}  MAE: {test_mae:.2f}  R²: {test_r2:.4f}  MSE: {test_mse:.2f}")
    
    # ========================================
    # Test Set Classification Metrics
    # ========================================
    # Compute classification metrics for test set if requested
    # Provides operational insights for deployment decisions
    if include_classification_metrics:
        test_class_results, test_class_df = compute_classification_metrics(test_preds)
        print_classification_summary(test_class_results, test_class_df)
        classification_records.append({
            "split": "test",
            "fold": len(folds) + 1,  # treat held-out test as the 5th fold
            "otpa_accuracy": test_class_results["otpa_accuracy"],
            "otpa_f1": test_class_results["otpa_f1"],
            "sddr_recall": test_class_results["sddr_recall"],
            "bucket_accuracy": test_class_results["bucket_accuracy"]
        })

    # ========================================
    # Build Consolidated Metrics DataFrames
    # ========================================
    # Combine train/val/test metrics into single DataFrame for easy comparison
    # Use fold index len(folds)+1 (=5) for the held-out test split to mirror 5-fold CV layout.
    test_record = [{"split": "test", "fold": len(folds) + 1, "rmse": test_rmse, "mae": test_mae, "r2": test_r2, "mse": test_mse}]
    metrics_df = pd.DataFrame(train_records + val_records + test_record)
    
    # Build classification metrics DataFrame if computed
    classification_df = pd.DataFrame(classification_records) if include_classification_metrics else None
    
    # Round to 4 decimal places for readability
    metrics_df['rmse'] = metrics_df['rmse'].round(4)
    metrics_df['mae'] = metrics_df['mae'].round(4)
    metrics_df['r2'] = metrics_df['r2'].round(4)
    metrics_df['mse'] = metrics_df['mse'].round(4)

    # ========================================
    # Metrics Definitions with LaTeX
    # ========================================
    # Provide LaTeX equations for each metric for documentation/reporting
    metrics_defs = pd.DataFrame([
        {"metric": "RMSE", "latex": r"\\mathrm{RMSE} = \\sqrt{\\tfrac{1}{N} \\sum_{i=1}^{N} (y_i - \\hat{y}_i)^2}"},
        {"metric": "MAE",  "latex": r"\\mathrm{MAE} = \\tfrac{1}{N} \\sum_{i=1}^{N} |y_i - \\hat{y}_i|"},
        {"metric": "R^2",  "latex": r"R^2 = 1 - \\dfrac{\\sum_{i=1}^{N} (y_i - \\hat{y}_i)^2}{\\sum_{i=1}^{N} (y_i - \\bar{y})^2}"},
        {"metric": "MSE",  "latex": r"\\mathrm{MSE} = \\tfrac{1}{N} \\sum_{i=1}^{N} (y_i - \\hat{y}_i)^2"},
    ])

    # ========================================
    # Extract Feature Importance (Coefficients)
    # ========================================
    # Extract and display top 10 most important features by absolute coefficient
    # Helps understand which features drive predictions
    try:
        # Extract coefficients from the linear regression model (last stage in pipeline)
        coefficients = best_model.model.stages[-1].coefficients.toArray()
        
        # Try to extract feature names from metadata (includes OHE expansions)
        feats_meta = test_preds.schema["features"].metadata
        attrs = []
        if "ml_attr" in feats_meta and "attrs" in feats_meta["ml_attr"]:
            ml_attrs = feats_meta["ml_attr"]["attrs"]
            # Collect attributes from all types (binary, numeric, nominal)
            for t in ["binary", "numeric", "nominal"]:
                if t in ml_attrs:
                    attrs.extend(ml_attrs[t])
            # Sort by vector index to match coefficient order
            attrs = sorted(attrs, key=lambda x: x["idx"])
            feature_names = [a.get("name", f"feature_{a['idx']}") for a in attrs]
        else:
            # Fallback: use generic names if metadata unavailable
            feature_names = [f"feature_{i}" for i in range(len(coefficients))]
        
        # Safety check: ensure lengths match
        if len(feature_names) != len(coefficients):
            feature_names = [f"feature_{i}" for i in range(len(coefficients))]
        
        # Create DataFrame with feature names and coefficients
        coef_df = pd.DataFrame({"feature": feature_names, "coefficient": coefficients})
        coef_df["abs_coefficient"] = coef_df["coefficient"].abs()
        coef_df = coef_df.sort_values("abs_coefficient", ascending=False)
        
        # Round coefficients to 4 decimal places
        top_coef_df = coef_df.head(10).copy()
        top_coef_df['coefficient'] = top_coef_df['coefficient'].round(4)
        top_coef_df['abs_coefficient'] = top_coef_df['abs_coefficient'].round(4)
        
        print("\nTop 10 Most Important Features (by absolute coefficient):")
        print(top_coef_df.to_string(index=False))
    except Exception as e:
        print(f"[Info] Skipping coefficient listing: {e}")
    
    # ========================================
    # Print Total Runtime
    # ========================================
    # Report total execution time for performance tracking
    overall_elapsed = time.time() - overall_start_time
    hours, remainder = divmod(overall_elapsed, 3600)
    minutes, seconds = divmod(remainder, 60)
    print("\n" + "="*80)
    print(f"TOTAL RUNTIME: {int(hours)}h {int(minutes)}m {seconds:.2f}s ({overall_elapsed:.2f} seconds)")
    print("="*80)

    # ========================================
    # Return Results
    # ========================================
    return best_model, metrics, {"rmse": test_rmse, "mae": test_mae, "r2": test_r2, "mse": test_mse}, metrics_df, metrics_defs, classification_df, test_class_results if include_classification_metrics else None


# ============================================================================
# Main Execution Block
# ============================================================================

if __name__ == "__main__":
    """
    Main execution entry point for the cross-validation script.
    
    Workflow:
        1. Print cluster configuration (for reproducibility and debugging)
        2. Run expanding window cross-validation
        3. Display comprehensive results (regression + classification metrics)
    
    Note:
        This block only runs when the script is executed directly,
        not when imported as a module.
    """
    
    # ========================================
    # Print Cluster Configuration
    # ========================================
    # Display cluster resources to ensure adequate capacity and enable reproducibility
    # Important for:
    # - Debugging OOM errors (check if cluster is undersized)
    # - Reproducing results (document exact cluster configuration)
    # - Cost tracking (understand resource usage)
    
    sc = spark.sparkContext
    
    # Get number of executors (exclude driver node)
    # statusTracker().getExecutorInfos() returns list of all executors
    num_executors = len(sc._jsc.sc().statusTracker().getExecutorInfos()) - 1
    
    # Get executor configuration from Spark conf
    executor_memory = sc.getConf().get("spark.executor.memory", "Unknown")
    executor_cores = sc.getConf().get("spark.executor.cores", "Unknown")
    actual_cores = sc.defaultParallelism / num_executors if num_executors > 0 else "Unknown"
    print(f"Estimated cores per executor: {int(actual_cores)}")
    
    # Print cluster configuration
    print("\n" + "="*80)
    print("CLUSTER CONFIGURATION")
    print("="*80)
    print(f"Cluster Size: {num_executors} executors, {executor_cores} cores per executor, {executor_memory} RAM per executor")
    print("="*80 + "\n")
    
    # ========================================
    # Load 2015 joined dataset
    # ========================================
    df_1y = spark.read.parquet("dbfs:/mnt/mids-w261/student-groups/Group_4_2/processed/flights_weather_joined_2015")
    
    # Optional: select only the columns needed by the baseline estimator
    # (day_of_week, month, dep_time_blk, origin, dest, op_unique_carrier,
    #  hourlywindspeed, hourlyvisibility, hourlyprecipitation, distance, dep_delay)
    df_1y = df_1y.select(
        "day_of_week",
        "month",
        "dep_time_blk",
        "origin",
        "dest",
        "op_unique_carrier",
        "hourlywindspeed",
        "hourlyvisibility",
        "hourlyprecipitation",
        "distance",
        "dep_delay",
    )
    
    # ========================================
    # Run Cross-Validation
    # ========================================
    # Execute expanding window CV with comprehensive metrics on 1-year data
    # include_classification_metrics=True: Compute OTPA, SDDR, bucket accuracy
    
    best_model, val_metrics_list, test_metrics_dict, metrics_df, metrics_defs, classification_df, test_class_results = run_cv(
        df_1y,
        include_classification_metrics=True,
    )
    
    # ========================================
    # Display Regression Metrics
    # ========================================
    # Show comprehensive regression metrics across train/val/test splits
    # metrics_df: Per-fold metrics (split, fold, RMSE, MAE, R², MSE)
    # metrics_defs: LaTeX equations for each metric (for documentation)
    
    print("\n" + "="*80)
    print("REGRESSION METRICS")
    print("="*80)
    display(metrics_df)  # Use display() for interactive Databricks table
    display(metrics_defs)
    
    # ========================================
    # Display Classification Metrics
    # ========================================
    # Show operational classification metrics if computed
    # Includes OTPA (on-time accuracy), SDDR (severe delay detection),
    # and 4-bucket classification accuracy
    
    if classification_df is not None:
        print("\n" + "="*80)
        print("CLASSIFICATION METRICS SUMMARY")
        print("="*80)
        display(classification_df)  # Use display() for interactive Databricks table

| feature                        |   coefficient |   abs_coefficient |
|:-------------------------------|--------------:|------------------:|
| hourlyvisibility_imputed       |       -2.3834 |            2.3834 |
| dep_time_blk_encoded_0600-0659 |       -1.8372 |            1.8372 |
| month_encoded_6                |        1.6024 |            1.6024 |
| dep_time_blk_encoded_1700-1759 |        1.5085 |            1.5085 |
| dep_time_blk_encoded_0700-0759 |       -1.4293 |            1.4293 |
| dep_time_blk_encoded_2000-2059 |        1.4151 |            1.4151 |
| dep_time_blk_encoded_0800-0859 |       -1.3648 |            1.3648 |
| dep_time_blk_encoded_0001-0559 |       -1.3449 |            1.3449 |
| month_encoded_9                |       -1.3427 |            1.3427 |
| month_encoded_10               |       -1.3389 |            1.3389 |

## Re-run model 5

In [0]:
"""
================================================================================
Cross-Validation for Baseline Linear Regression (2015 Joined Dataset)
================================================================================

Purpose:
    Implements expanding window cross-validation for flight delay prediction
    using a baseline linear regression model with comprehensive feature engineering.

Key Features:
    - Expanding window CV: Each fold trains on progressively more historical data
    - Direct use of 2015 joined flights+weather data (no pre-checkpointed folds)
    - Baseline pipeline: Numeric cleaning + median imputation + categorical OHE + standardization + LR
    - Comprehensive metrics: Regression (RMSE, MAE, R², MSE) + Classification (OTPA, SDDR, 4-bucket)
    - Memory management: Strategic caching (optional) and explicit GC to avoid OOM

Data Flow:
    1. Load 2015 joined dataset from DBFS
    2. Build time-aware expanding folds using the `month` column
    3. For each fold (except last):
       - Train model on expanding training set
       - Evaluate on validation set
       - Track best model by RMSE
    4. Evaluate best model on held-out test set
    5. Report comprehensive metrics and feature importance

Author: Emily Lieske
Date: November 2025
================================================================================
"""

# ============================================================================
# Imports
# ============================================================================

# PySpark SQL functions for data transformation
from pyspark.sql.functions import col, when, isnan, regexp_replace, trim, length
from pyspark.sql.types import DoubleType, StringType

# PySpark ML for pipeline construction and modeling
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder, StandardScaler, Imputer
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

# Standard libraries for metrics and timing
import numpy as np
import pandas as pd
import time
import gc  # Garbage collection for memory management


# ============================================================================
# Data Loading Functions
# ============================================================================

def _load_checkpointed_data(name, folder_path="dbfs:/student-groups/Group_4_2"):
    """
    Load a pre-checkpointed Parquet dataset from DBFS.
    
    Args:
        name (str): Dataset name (e.g., "OTPW_12M_FOLD_1_TRAIN")
        folder_path (str): DBFS path to the folder containing checkpointed data
        
    Returns:
        pyspark.sql.DataFrame: Loaded dataset
        
    Note:
        Pre-checkpointed data significantly speeds up CV by avoiding repeated
        data loading and splitting operations.
    """
    return spark.read.parquet(f"{folder_path}/{name}.parquet")


def _build_time_folds_from_df(df, n_val_folds=3):
    """
    Build expanding-window folds directly from a single 1-year DataFrame.
    
    We use the `month` column (1–12) to create time-aware folds:
        - Fold 1: Train on months <= 8, validate on month 9
        - Fold 2: Train on months <= 9, validate on month 10
        - Fold 3: Train on months <= 10, validate on month 11
      Held-out test: Train on months <= 11, test on month 12
    
    Returns:
        folds: list of (train_df, val_df) for validation folds
        test_fold: (train_df, test_df) for final held-out evaluation
    """
    folds = []
    # Validation folds for months 8, 9, 10, 11.
    # Each fold trains on all months up to train_last_month (inclusive)
    # and validates on the next month (val_month).
    #
    # This mirrors the 4 validation folds in Model 3:
    #   Fold 1: train <= 7,  val == 8
    #   Fold 2: train <= 8,  val == 9
    #   Fold 3: train <= 9,  val == 10
    #   Fold 4: train <= 10, val == 11
    fold_specs = [(7, 8), (8, 9), (9, 10), (10, 11)]
    for train_last_month, val_month in fold_specs:
        # All history up to train_last_month becomes the training window
        train_df = df.filter(col("month") <= train_last_month)
        # The following month becomes the validation window
        val_df = df.filter(col("month") == val_month)
        folds.append((train_df, val_df))
    
    # Held-out test fold:
    # Train on all data up to month 11, and test on month 12 only.
    test_train_df = df.filter(col("month") <= 11)
    test_df = df.filter(col("month") == 12)
    test_fold = (test_train_df, test_df)
    
    return folds, test_fold


# ============================================================================
# Baseline Estimator Class
# ============================================================================

class BaselineEstimator:
    """
    Baseline Linear Regression Estimator with Feature Engineering Pipeline.
    
    This class encapsulates the entire feature engineering and modeling pipeline:
        1. Data preparation (label cleaning, feature selection)
        2. Numerical feature cleaning (remove non-numeric chars, handle nulls)
        3. Median imputation for numerical features
        4. Categorical encoding (StringIndexer + OneHotEncoder)
        5. Feature assembly and standardization
        6. Linear regression modeling
    
    Feature Families:
        - Temporal: day_of_week, month, dep_time_blk
        - Airport: origin, dest
        - Flight: op_unique_carrier, distance
        - Weather: hourlywindspeed, hourlyvisibility, hourlyprecipitation
    
    Why This Design:
        - Encapsulation: All preprocessing logic in one place
        - Reusability: Same pipeline for train/val/test
        - Spark ML compatibility: Uses Pipeline for efficient execution
    """
    
    def __init__(self, label_col="dep_delay"):
        """
        Initialize the estimator with feature definitions.
        
        Args:
            label_col (str): Name of the target variable column
        """
        self.label_col = label_col
        self.pipeline = None
        self.model = None
        
        # Categorical features: Encoded using StringIndexer + OneHotEncoder
        # These capture temporal patterns, route characteristics, and carrier effects
        self.categorical_features = [
            "day_of_week",        # Day of week (1=Monday, 7=Sunday)
            "month",              # Month of year (1-12)
            "dep_time_blk",       # Departure time block (e.g., "0600-0659")
            "origin",             # Origin airport code
            "dest",               # Destination airport code
            "op_unique_carrier"   # Operating carrier code
        ]
        
        # Numerical features: Weather conditions and flight distance
        self.numerical_features = [
            "hourlywindspeed",       # Wind speed at origin (mph)
            "hourlyvisibility",      # Visibility at origin (miles)
            "hourlyprecipitation",   # Precipitation at origin (inches)
            "distance"               # Flight distance (miles)
        ]

    def _prepare(self, df):
        """
        Prepare the DataFrame for modeling by cleaning the label and numerical features.
        
        Steps:
            1. Cast label to DoubleType (required by Spark ML)
            2. Filter out rows with null/NaN labels (Spark ML requirement)
            3. Select only required features + label
            4. Clean numerical features:
               - Remove non-numeric characters (e.g., "12.5mph" -> "12.5")
               - Convert empty strings to null
               - Cast to DoubleType
        
        Args:
            df (pyspark.sql.DataFrame): Input DataFrame
            
        Returns:
            pyspark.sql.DataFrame: Cleaned DataFrame
            
        Why This Matters:
            - Spark ML LinearRegression requires DoubleType labels with no nulls
            - Numerical features may contain string artifacts from source data
            - Explicit type casting prevents downstream pipeline errors
        """
        # Cast label to double and filter out null/NaN values
        # Spark ML does not accept null labels
        df = df.withColumn(self.label_col, col(self.label_col).cast(DoubleType()))
        df = df.filter(~(col(self.label_col).isNull() | isnan(col(self.label_col))))

        # Select only the columns we need (features + label)
        # This reduces memory footprint and prevents accidental data leakage
        selected = [c for c in (self.categorical_features + self.numerical_features + [self.label_col]) 
                    if c in df.columns]
        df = df.select(*selected)

        # Clean numerical features: remove non-numeric characters, handle empty strings
        for f in self.numerical_features:
            if f in df.columns:
                # Step 1: Cast to string to enable regex operations
                # Step 2: Remove all non-numeric characters except +, -, and .
                df = df.withColumn(f, regexp_replace(col(f).cast(StringType()), r"[^0-9+\-\.]", ""))
                
                # Step 3: Convert empty strings to null (for imputation)
                df = df.withColumn(f, when(length(trim(col(f))) == 0, None).otherwise(col(f)))
                
                # Step 4: Cast to DoubleType for modeling
                df = df.withColumn(f, col(f).cast(DoubleType()))
        
        return df

    def _build_pipeline(self, df):
        """
        Build the Spark ML Pipeline with all feature engineering stages.
        
        Pipeline Stages:
            1. Median Imputer: Fill missing values in numerical features
            2. StringIndexers: Convert categorical strings to indices
            3. OneHotEncoders: Convert indices to binary vectors
            4. VectorAssembler: Combine all features into single vector
            5. StandardScaler: Standardize features (mean=0, std=1)
            6. LinearRegression: Train linear model
        
        Args:
            df (pyspark.sql.DataFrame): Prepared DataFrame
            
        Returns:
            pyspark.sql.DataFrame: DataFrame (may have additional columns from transformations)
            
        Why This Design:
            - Pipeline ensures consistent transformations across train/val/test
            - Median imputation preserves more rows than dropping nulls
            - OneHotEncoding with dropLast=True prevents multicollinearity
            - StandardScaler improves convergence for gradient descent
            - No regularization (regParam=0) for interpretable baseline
        """
        stages = []
        
        # ========================================
        # Stage 1: Median Imputation for Numerical Features
        # ========================================
        # Use Spark's Imputer to fill missing numerical values with the median.
        # This retains more data than dropping rows with nulls.
        input_cols = [f for f in self.numerical_features if f in df.columns]
        output_cols = [f"{f}_imputed" for f in input_cols]
        if input_cols:
            imputer = Imputer(
                strategy="median",
                inputCols=input_cols,
                outputCols=output_cols,
            )
            stages.append(imputer)

        # ========================================
        # Stage 2: Categorical Encoding (StringIndexer + OneHotEncoder)
        # ========================================
        # StringIndexer: Converts strings to numeric indices (most frequent = 0)
        # OneHotEncoder: Converts indices to binary vectors (prevents ordinal assumption)
        # handleInvalid="keep": Unseen categories in test set get their own index
        # dropLast=True: Drop last category to prevent multicollinearity
        for f in self.categorical_features:
            if f in df.columns:
                # For numeric-coded categoricals (day_of_week, month, dep_time_blk),
                # cast to string first to ensure consistent handling
                if f in ["day_of_week", "month", "dep_time_blk"]:
                    df = df.withColumn(f"{f}_clean", 
                                      when(col(f).isNull(), "UNKNOWN").otherwise(col(f).cast(StringType())))
                else:
                    # For string categoricals (ORIGIN, DEST, CARRIER), handle nulls only
                    df = df.withColumn(f"{f}_clean", 
                                      when(col(f).isNull(), "UNKNOWN").otherwise(col(f)))
                
                # Add StringIndexer stage
                stages.append(StringIndexer(inputCol=f"{f}_clean", 
                                           outputCol=f"{f}_indexed", 
                                           handleInvalid="keep"))
                
                # Add OneHotEncoder stage
                stages.append(OneHotEncoder(inputCols=[f"{f}_indexed"], 
                                           outputCols=[f"{f}_encoded"], 
                                           dropLast=True))

        # ========================================
        # Stage 3: Feature Assembly
        # ========================================
        # Combine all features (numerical + encoded categorical) into single vector
        # handleInvalid="skip": Skip rows with invalid values
        feature_columns = [f"{f}_imputed" for f in self.numerical_features if f in df.columns] + \
                          [f"{f}_encoded" for f in self.categorical_features if f in df.columns]
        assembler = VectorAssembler(inputCols=feature_columns, 
                                    outputCol="features", 
                                    handleInvalid="skip")
        
        # ========================================
        # Stage 4: Feature Standardization
        # ========================================
        # Standardize features to mean=0, std=1
        # withStd=True: Scale to unit variance
        # withMean=True: Center to zero mean
        # Why? Improves convergence and makes coefficients comparable
        scaler = StandardScaler(inputCol="features", 
                               outputCol="scaled_features", 
                               withStd=True, 
                               withMean=True)
        
        # ========================================
        # Stage 5: Linear Regression
        # ========================================
        # Baseline model: No regularization (regParam=0, elasticNetParam=0)
        # maxIter=100: Maximum iterations for convergence
        # Why no regularization? We want an interpretable baseline to understand
        # feature importance before adding complexity
        lr = LinearRegression(featuresCol="scaled_features", 
                            labelCol=self.label_col, 
                            maxIter=100, 
                            regParam=0.0, 
                            elasticNetParam=0.0)

        # Assemble all stages into pipeline
        stages.extend([assembler, scaler, lr])
        self.pipeline = Pipeline(stages=stages)
        
        return df

    def fit(self, df):
        """
        Fit the pipeline on training data.
        
        Args:
            df (pyspark.sql.DataFrame): Training DataFrame
            
        Returns:
            BaselineEstimator: self (for method chaining)
        """
        df_prep = self._prepare(df)
        df_prep = self._build_pipeline(df_prep)
        self.model = self.pipeline.fit(df_prep)
        return self

    def transform(self, df):
        """
        Transform data using the fitted pipeline.
        
        Args:
            df (pyspark.sql.DataFrame): DataFrame to transform (val/test)
            
        Returns:
            pyspark.sql.DataFrame: Transformed DataFrame with predictions
            
        Note:
            We rebuild the pipeline on the input DataFrame to ensure all
            transformation columns are present, but use the fitted model
            for predictions.
        """
        df_prep = self._prepare(df)
        df_prep = self._build_pipeline(df_prep)  # Rebuild to ensure cols present
        return self.model.transform(df_prep)


# ============================================================================
# Cross-Validation Runner
# ============================================================================

def run_cv(df_1y, include_classification_metrics=True):
    """
    Run expanding window cross-validation for the baseline model on 1-year data.
    
    Process:
        1. Build time-aware folds directly from the 1-year DataFrame (expanding window)
        2. For each validation fold:
           a. Train model on training set
           b. Evaluate on both training and validation sets
           c. Compute regression and classification metrics
           d. Track the best model (by validation RMSE)
        3. Evaluate best model on held-out test set (month = 12)
        4. Report comprehensive metrics, feature importance, and runtime
    
    Args:
        df_1y (DataFrame): Full 1-year joined dataset with `month` and `dep_delay`
        include_classification_metrics (bool): Whether to compute OTPA, SDDR, etc.
        
    Returns:
        tuple: (best_model, val_metrics_list, test_metrics_dict, metrics_df,
                metrics_defs, classification_df, test_class_results)
    
    Memory Management Strategy:
        - .cache().count(): Forces immediate materialization in controlled chunks
        - .unpersist(): Explicitly frees memory after each fold
        - This prevents OOM errors by avoiding lazy accumulation of cached data
    """
    # ========================================
    # Initialize Timer and Build Folds
    # ========================================
    overall_start_time = time.time()
    # Build time-aware (expanding window) folds from the 1-year DataFrame.
    # - folds: list of (train_df, val_df) pairs for validation months 9,10,11
    # - test_fold: (train_df, test_df) pair for held-out month 12
    folds, test_fold = _build_time_folds_from_df(df_1y)

    # ========================================
    # Initialize Evaluators
    # ========================================
    # Create evaluators for each regression metric
    # These will be reused across all folds for consistency
    eval_rmse = RegressionEvaluator(predictionCol="prediction", labelCol="dep_delay", metricName="rmse")
    eval_mae  = RegressionEvaluator(predictionCol="prediction", labelCol="dep_delay", metricName="mae")
    eval_r2   = RegressionEvaluator(predictionCol="prediction", labelCol="dep_delay", metricName="r2")
    eval_mse  = RegressionEvaluator(predictionCol="prediction", labelCol="dep_delay", metricName="mse")

    # ========================================
    # Initialize Metric Storage
    # ========================================
    metrics = []  # Validation metrics per fold (for backward compatibility)
    train_records = []  # Per-fold train metrics across months
    val_records = []    # Per-fold validation metrics across months
    classification_records = []  # Classification metrics per validation/test fold
    best_model = None  # Best model (by validation RMSE)
    best_rmse = float("inf")  # Track best validation RMSE seen so far

    # ========================================
    # Initialize Estimator
    # ========================================
    est = BaselineEstimator(label_col="dep_delay")

    # ========================================
    # Cross-Validation Loop
    # ========================================
    # Train/validate on all validation folds; a separate held-out test fold is used later.
    # Each fold corresponds to a different validation month (9, 10, 11).
    for idx, (train_df, val_df) in enumerate(folds, start=1):
        print(f"--- Fold {idx}/{len(folds)} ---")
        
        # ========================================
        # Cache DataFrames for Performance (optional)
        # ========================================
        # In this model we leave caching commented out to reduce memory pressure
        # on small clusters. Uncomment if you need more performance and have RAM.
        # ========================================
        # Train Model
        # ========================================
        model = est.fit(train_df)
        
        # ========================================
        # Generate Predictions
        # ========================================
        # Generate predictions on both validation and training sets.
        # Training metrics help detect overfitting (train << val).
        
        # Validation predictions
        val_preds = model.transform(val_df)
        # val_preds.cache().count()  # Disabled to prevent OOM
        
        # Train predictions
        train_preds = model.transform(train_df)
        # train_preds.cache().count()  # Disabled to prevent OOM

        # ========================================
        # Compute Regression Metrics (Validation)
        # ========================================
        val_rmse = eval_rmse.evaluate(val_preds)
        val_mae  = eval_mae.evaluate(val_preds)
        val_r2   = eval_r2.evaluate(val_preds)
        val_mse  = eval_mse.evaluate(val_preds)
        
        # Store validation metrics
        metrics.append({"fold": idx, "rmse": val_rmse, "mae": val_mae, "r2": val_r2, "mse": val_mse})
        val_records.append({"split": "validation", "fold": idx, "rmse": val_rmse, "mae": val_mae, "r2": val_r2, "mse": val_mse})

        # ========================================
        # Compute Regression Metrics (Training)
        # ========================================
        # Training metrics help detect overfitting
        # If train metrics >> val metrics, model is overfitting
        tr_rmse = eval_rmse.evaluate(train_preds)
        tr_mae  = eval_mae.evaluate(train_preds)
        tr_r2   = eval_r2.evaluate(train_preds)
        tr_mse  = eval_mse.evaluate(train_preds)
        train_records.append({"split": "train", "fold": idx, "rmse": tr_rmse, "mae": tr_mae, "r2": tr_r2, "mse": tr_mse})

        # ========================================
        # Track Best Model
        # ========================================
        # Select model with lowest validation RMSE
        # This model will be used for final test evaluation
        if val_rmse < best_rmse:
            best_rmse = val_rmse
            best_model = model

        # Print validation metrics
        print(f"RMSE: {val_rmse:.2f}  MAE: {val_mae:.2f}  R²: {val_r2:.4f}  MSE: {val_mse:.2f}")
        
        # ========================================
        # Compute Classification Metrics (Optional)
        # ========================================
        # Classification metrics provide operational insights:
        # - OTPA: On-Time Performance Accuracy (<15 min threshold)
        # - SDDR: Severe Delay Detection Rate (≥60 min threshold)
        # - Bucket Accuracy: 4-bucket classification (Early, OnTime, Delayed, Severe)
        if include_classification_metrics:
            val_class_results, _ = compute_classification_metrics(val_preds)
            classification_records.append({
                "split": "validation",
                "fold": idx,
                "otpa_accuracy": val_class_results["otpa_accuracy"],
                "otpa_f1": val_class_results["otpa_f1"],
                "sddr_recall": val_class_results["sddr_recall"],
                "bucket_accuracy": val_class_results["bucket_accuracy"]
            })
            print(f"  OTPA Acc: {val_class_results['otpa_accuracy']:.4f}  "
                  f"OTPA F1: {val_class_results['otpa_f1']:.4f}  "
                  f"SDDR: {val_class_results['sddr_recall']:.4f}")
        
        # ========================================
        # Free Memory
        # ========================================
        # Explicitly unpersist cached DataFrames to free memory
        # Critical for preventing OOM in subsequent folds
        # Disabled since caching is disabled
        # train_preds.unpersist()
        # val_preds.unpersist()
        # train_df.unpersist()
        # val_df.unpersist()
        
        # Force Python garbage collection to free driver memory
        gc.collect()
        
        # Clear Spark SQL cache to free executor memory
        spark.catalog.clearCache()

    # ========================================
    # Validation Summary
    # ========================================
    # Report aggregate statistics across all validation folds
    # This helps assess model stability and generalization
    print("-------------------------------")
    print("Validation Summary:")
    print(f"Best RMSE: {best_rmse:.2f}")
    print(f"Mean RMSE: {np.mean([m['rmse'] for m in metrics]):.2f}  |  Std: {np.std([m['rmse'] for m in metrics]):.2f}")
    print(f"Mean MAE:  {np.mean([m['mae'] for m in metrics]):.2f}")
    print(f"Mean R²:   {np.mean([m['r2'] for m in metrics]):.4f}")
    print(f"Mean MSE:  {np.mean([m['mse'] for m in metrics]):.2f}")

    # ========================================
    # Held-Out Test Evaluation
    # ========================================
    # Evaluate best model on held-out test set (month = 12)
    # This provides an unbiased estimate of production performance
    train_df, test_df = test_fold
    # test_df.cache().count()  # Disabled to prevent OOM
    
    # Generate predictions on test set
    test_preds = best_model.transform(test_df)
    # test_preds.cache().count()  # Disabled to prevent OOM
    
    # Compute regression metrics on test set
    test_rmse = eval_rmse.evaluate(test_preds)
    test_mae  = eval_mae.evaluate(test_preds)
    test_r2   = eval_r2.evaluate(test_preds)
    test_mse  = eval_mse.evaluate(test_preds)
    
    # Free memory
    # Disabled since caching is disabled
    # test_preds.unpersist()
    # test_df.unpersist()

    # Print test results
    print("-------------------------------")
    print("Held-out Test:")
    print(f"RMSE: {test_rmse:.2f}  MAE: {test_mae:.2f}  R²: {test_r2:.4f}  MSE: {test_mse:.2f}")
    
    # ========================================
    # Test Set Classification Metrics
    # ========================================
    # Compute classification metrics for test set if requested
    # Provides operational insights for deployment decisions
    if include_classification_metrics:
        test_class_results, test_class_df = compute_classification_metrics(test_preds)
        print_classification_summary(test_class_results, test_class_df)
        classification_records.append({
            "split": "test",
            "fold": len(folds) + 1,  # treat held-out test as the 5th fold
            "otpa_accuracy": test_class_results["otpa_accuracy"],
            "otpa_f1": test_class_results["otpa_f1"],
            "sddr_recall": test_class_results["sddr_recall"],
            "bucket_accuracy": test_class_results["bucket_accuracy"]
        })

    # ========================================
    # Build Consolidated Metrics DataFrames
    # ========================================
    # Combine train/val/test metrics into single DataFrame for easy comparison
    # Use fold index len(folds)+1 (=5) for the held-out test split to mirror 5-fold CV layout.
    test_record = [{"split": "test", "fold": len(folds) + 1, "rmse": test_rmse, "mae": test_mae, "r2": test_r2, "mse": test_mse}]
    metrics_df = pd.DataFrame(train_records + val_records + test_record)
    
    # Build classification metrics DataFrame if computed
    classification_df = pd.DataFrame(classification_records) if include_classification_metrics else None
    
    # Round to 4 decimal places for readability
    metrics_df['rmse'] = metrics_df['rmse'].round(4)
    metrics_df['mae'] = metrics_df['mae'].round(4)
    metrics_df['r2'] = metrics_df['r2'].round(4)
    metrics_df['mse'] = metrics_df['mse'].round(4)

    # ========================================
    # Metrics Definitions with LaTeX
    # ========================================
    # Provide LaTeX equations for each metric for documentation/reporting
    metrics_defs = pd.DataFrame([
        {"metric": "RMSE", "latex": r"\\mathrm{RMSE} = \\sqrt{\\tfrac{1}{N} \\sum_{i=1}^{N} (y_i - \\hat{y}_i)^2}"},
        {"metric": "MAE",  "latex": r"\\mathrm{MAE} = \\tfrac{1}{N} \\sum_{i=1}^{N} |y_i - \\hat{y}_i|"},
        {"metric": "R^2",  "latex": r"R^2 = 1 - \\dfrac{\\sum_{i=1}^{N} (y_i - \\hat{y}_i)^2}{\\sum_{i=1}^{N} (y_i - \\bar{y})^2}"},
        {"metric": "MSE",  "latex": r"\\mathrm{MSE} = \\tfrac{1}{N} \\sum_{i=1}^{N} (y_i - \\hat{y}_i)^2"},
    ])

    # ========================================
    # Extract Feature Importance (Coefficients)
    # ========================================
    # Extract and display top 10 most important features by absolute coefficient
    # Helps understand which features drive predictions
    try:
        # Extract coefficients from the linear regression model (last stage in pipeline)
        coefficients = best_model.model.stages[-1].coefficients.toArray()
        
        # Try to extract feature names from metadata (includes OHE expansions)
        feats_meta = test_preds.schema["features"].metadata
        attrs = []
        if "ml_attr" in feats_meta and "attrs" in feats_meta["ml_attr"]:
            ml_attrs = feats_meta["ml_attr"]["attrs"]
            # Collect attributes from all types (binary, numeric, nominal)
            for t in ["binary", "numeric", "nominal"]:
                if t in ml_attrs:
                    attrs.extend(ml_attrs[t])
            # Sort by vector index to match coefficient order
            attrs = sorted(attrs, key=lambda x: x["idx"])
            feature_names = [a.get("name", f"feature_{a['idx']}") for a in attrs]
        else:
            # Fallback: use generic names if metadata unavailable
            feature_names = [f"feature_{i}" for i in range(len(coefficients))]
        
        # Safety check: ensure lengths match
        if len(feature_names) != len(coefficients):
            feature_names = [f"feature_{i}" for i in range(len(coefficients))]
        
        # Create DataFrame with feature names and coefficients
        coef_df = pd.DataFrame({"feature": feature_names, "coefficient": coefficients})
        coef_df["abs_coefficient"] = coef_df["coefficient"].abs()
        coef_df = coef_df.sort_values("abs_coefficient", ascending=False)
        
        # Round coefficients to 4 decimal places
        top_coef_df = coef_df.head(10).copy()
        top_coef_df['coefficient'] = top_coef_df['coefficient'].round(4)
        top_coef_df['abs_coefficient'] = top_coef_df['abs_coefficient'].round(4)
        
        print("\nTop 10 Most Important Features (by absolute coefficient):")
        print(top_coef_df.to_string(index=False))
    except Exception as e:
        print(f"[Info] Skipping coefficient listing: {e}")
    
    # ========================================
    # Print Total Runtime
    # ========================================
    # Report total execution time for performance tracking
    overall_elapsed = time.time() - overall_start_time
    hours, remainder = divmod(overall_elapsed, 3600)
    minutes, seconds = divmod(remainder, 60)
    print("\n" + "="*80)
    print(f"TOTAL RUNTIME: {int(hours)}h {int(minutes)}m {seconds:.2f}s ({overall_elapsed:.2f} seconds)")
    print("="*80)

    # ========================================
    # Return Results
    # ========================================
    return best_model, metrics, {"rmse": test_rmse, "mae": test_mae, "r2": test_r2, "mse": test_mse}, metrics_df, metrics_defs, classification_df, test_class_results if include_classification_metrics else None


# ============================================================================
# Main Execution Block
# ============================================================================

if __name__ == "__main__":
    """
    Main execution entry point for the cross-validation script.
    
    Workflow:
        1. Print cluster configuration (for reproducibility and debugging)
        2. Run expanding window cross-validation
        3. Display comprehensive results (regression + classification metrics)
    
    Note:
        This block only runs when the script is executed directly,
        not when imported as a module.
    """
    
    # ========================================
    # Print Cluster Configuration
    # ========================================
    # Display cluster resources to ensure adequate capacity and enable reproducibility
    # Important for:
    # - Debugging OOM errors (check if cluster is undersized)
    # - Reproducing results (document exact cluster configuration)
    # - Cost tracking (understand resource usage)
    
    sc = spark.sparkContext
    
    # Get number of executors (exclude driver node)
    # statusTracker().getExecutorInfos() returns list of all executors
    num_executors = len(sc._jsc.sc().statusTracker().getExecutorInfos()) - 1
    
    # Get executor configuration from Spark conf
    executor_memory = sc.getConf().get("spark.executor.memory", "Unknown")
    executor_cores = sc.getConf().get("spark.executor.cores", "Unknown")
    actual_cores = sc.defaultParallelism / num_executors if num_executors > 0 else "Unknown"
    print(f"Estimated cores per executor: {int(actual_cores)}")
    
    # Print cluster configuration
    print("\n" + "="*80)
    print("CLUSTER CONFIGURATION")
    print("="*80)
    print(f"Cluster Size: {num_executors} executors, {executor_cores} cores per executor, {executor_memory} RAM per executor")
    print("="*80 + "\n")
    
    # ========================================
    # Load 2015 joined dataset
    # ========================================
    df_1y = spark.read.parquet("dbfs:/mnt/mids-w261/student-groups/Group_4_2/processed/flights_weather_joined_2015")
    
    # Optional: select only the columns needed by the baseline estimator
    # (day_of_week, month, dep_time_blk, origin, dest, op_unique_carrier,
    #  hourlywindspeed, hourlyvisibility, hourlyprecipitation, distance, dep_delay)
    df_1y = df_1y.select(
        "day_of_week",
        "month",
        "dep_time_blk",
        "origin",
        "dest",
        "op_unique_carrier",
        "hourlywindspeed",
        "hourlyvisibility",
        "hourlyprecipitation",
        "distance",
        "dep_delay",
    )
    
    # ========================================
    # Run Cross-Validation
    # ========================================
    # Execute expanding window CV with comprehensive metrics on 1-year data
    # include_classification_metrics=True: Compute OTPA, SDDR, bucket accuracy
    
    best_model, val_metrics_list, test_metrics_dict, metrics_df, metrics_defs, classification_df, test_class_results = run_cv(
        df_1y,
        include_classification_metrics=True,
    )
    
    # ========================================
    # Display Regression Metrics
    # ========================================
    # Show comprehensive regression metrics across train/val/test splits
    # metrics_df: Per-fold metrics (split, fold, RMSE, MAE, R², MSE)
    # metrics_defs: LaTeX equations for each metric (for documentation)
    
    print("\n" + "="*80)
    print("REGRESSION METRICS")
    print("="*80)
    display(metrics_df)  # Use display() for interactive Databricks table
    display(metrics_defs)
    
    # ========================================
    # Display Classification Metrics
    # ========================================
    # Show operational classification metrics if computed
    # Includes OTPA (on-time accuracy), SDDR (severe delay detection),
    # and 4-bucket classification accuracy
    
    if classification_df is not None:
        print("\n" + "="*80)
        print("CLASSIFICATION METRICS SUMMARY")
        print("="*80)
        display(classification_df)  # Use display() for interactive Databricks table

In [0]:
df_1y.count()

In [0]:
displayHTML("""

<!DOCTYPE html>
<html>
<head>
  <script src="https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.min.js"></script>
  <script>
    mermaid.initialize({
      startOnLoad: true,
      theme: 'dark',
      themeVariables: {
        primaryColor: '#4a5568',
        primaryTextColor: '#fff',
        primaryBorderColor: '#cbd5e0',
        lineColor: '#cbd5e0',
        secondaryColor: '#2d3748',
        tertiaryColor: '#1a202c',
        background: '#1a202c',
        mainBkg: '#4a5568',
        secondBkg: '#2d3748',
        tertiaryBkg: '#1a202c'
      }
    });
  </script>
  <style>
    body { background-color: #1a202c; }
    .mermaid { background-color: #1a202c; }
  </style>
</head>
<body>

<div class="mermaid">
flowchart LR

    %% Input
    Input["<b>Input</b><br/>2015 Joined DataFrame<br/>(flights_weather_joined_2015)<br/>Temporal, Airport, Flight, Weather<br/>+ Label: dep_delay"]

    %% Stage 0: Time-Aware Expanding Folds (CV)
    subgraph S0["<b>Stage 0: Expanding-Window CV Splits</b>"]
        Split["Use month (1–12) to build folds:<br/><br/>Fold 1: Train ≤ 7, Val = 8<br/>Fold 2: Train ≤ 8, Val = 9<br/>Fold 3: Train ≤ 9, Val = 10<br/>Fold 4: Train ≤ 10, Val = 11<br/><br/>Held-out Test: Train ≤ 11, Test = 12"]
    end

    %% Stage 1: Data Preparation
    subgraph S1["<b>Stage 1: Data Preparation</b>"]
        LabelClean["Cast dep_delay to Double<br/>Filter Null/NaN Labels"]
        SelectFeat["Select Baseline Features<br/>Temporal, Airport, Flight, Weather"]
        NumClean["Clean Numerical Features:<br/>hourlywindspeed, hourlyvisibility,<br/>hourlyprecipitation, distance<br/>Remove non-numeric chars<br/>Empty → Null<br/>Cast to Double"]
        LabelClean --> SelectFeat --> NumClean
    end

    %% Stage 2: Median Imputation for Numeric Features
    subgraph S2["<b>Stage 2: Median Imputation</b>"]
        Impute["Imputer(strategy='median')<br/>Inputs:<br/>hourlywindspeed, hourlyvisibility,<br/>hourlyprecipitation, distance<br/><br/>Outputs:<br/>hourlywindspeed_imputed,<br/>hourlyvisibility_imputed,<br/>hourlyprecipitation_imputed,<br/>distance_imputed"]
    end

    %% Stage 3: Categorical Encoding
    subgraph S3["<b>Stage 3: Categorical Encoding</b>"]
        DOW["day_of_week_clean<br/>StringIndexer + OHE"]
        Month["month_clean<br/>StringIndexer + OHE"]
        TimeBlk["dep_time_blk_clean<br/>StringIndexer + OHE"]
        Origin["origin_clean<br/>StringIndexer + OHE"]
        Dest["dest_clean<br/>StringIndexer + OHE"]
        Carrier["op_unique_carrier_clean<br/>StringIndexer + OHE"]
    end

    %% Stage 4: Feature Assembly
    subgraph S4["<b>Stage 4: Feature Assembly</b>"]
        Assemble["VectorAssembler<br/>Combine Numerical + Encoded Categorical<br/><br/>Numerical inputs:<br/>hourlywindspeed_imputed,<br/>hourlyvisibility_imputed,<br/>hourlyprecipitation_imputed,<br/>distance_imputed"]
    end

    %% Stage 5: Standardization
    subgraph S5["<b>Stage 5: Standardization</b>"]
        Scale["StandardScaler<br/>features → scaled_features<br/>withMean=True, withStd=True"]
    end

    %% Stage 6: Linear Regression
    subgraph S6["<b>Stage 6: Model</b>"]
        LR["Linear Regression<br/>featuresCol=scaled_features<br/>labelCol=dep_delay<br/>maxIter=100<br/>regParam=0.0, elasticNet=0.0"]
    end

    %% Output & Metrics
    Output["<b>Output</b><br/>Per-Fold Predictions & Metrics<br/><br/>Validation (4 folds):<br/>RMSE, MAE, R², MSE<br/>OTPA (Accuracy, F1)<br/>SDDR Recall<br/>4-Bucket Accuracy<br/><br/>Held-out Test (month 12):<br/>Same metrics + Top 10 Coefficients"]

    %% Connections
    Input --> Split
    Split --> S1
    S1 --> Impute
    S1 --> S3
    Impute --> Assemble
    S3 --> Assemble
    Assemble --> Scale --> LR --> Output

</div>

</body>
</html>

""")

# Model 6: No-Cross-Validation/Train-Test-Split/12-month/Dropped-Null/Custom-Data
One final model for the results section not using cross-validation and only using the first three quarters as train and the last quarter as test

Please report the results of training (first three-quarters of the one-year dataset), and blind testing (the last quarter of the available one-year dataset).

In [0]:
"""
================================================================================
Time-Based Train/Test Split for Baseline Linear Regression (12-Month Dataset)
================================================================================

Purpose:
    Implements a simple time-based split (first 3/4 of year for training,
    last 1/4 for testing) for flight delay prediction using a baseline
    linear regression model with comprehensive feature engineering.

Key Features:
    - Time-based split: Train on months 1-9, test on months 10-12
    - Baseline pipeline: Numeric cleaning + drop nulls + categorical OHE + standardization + LR
    - Comprehensive metrics: Regression (RMSE, MAE, R², MSE) + Classification (OTPA, SDDR, 4-bucket)
    - Memory management: Minimal caching to prevent OOM errors

Data Flow:
    1. Load 1-year joined dataset from DBFS
    2. Split by month: Train (months 1-9), Test (months 10-12)
    3. Train model on training set
    4. Evaluate on both training and test sets
    5. Report comprehensive metrics and feature importance

Author: Emily Lieske
Date: November 2025
================================================================================
"""

# ============================================================================
# Imports
# ============================================================================

# PySpark SQL functions for data transformation
from pyspark.sql.functions import col, when, isnan, regexp_replace, trim, length
from pyspark.sql.types import DoubleType, StringType

# PySpark ML for pipeline construction and modeling
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder, StandardScaler, Imputer
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

# Standard libraries for metrics and timing
import numpy as np
import pandas as pd
import time
import gc  # Garbage collection for memory management

# ============================================================================
# Data Loading Functions
# ============================================================================

def _load_checkpointed_data(name, folder_path="dbfs:/student-groups/Group_4_2"):
    """
    Load a pre-checkpointed Parquet dataset from DBFS.
    
    Args:
        name (str): Dataset name (e.g., "OTPW_12M_FOLD_1_TRAIN")
        folder_path (str): DBFS path to the folder containing checkpointed data
        
    Returns:
        pyspark.sql.DataFrame: Loaded dataset
        
    Note:
        Pre-checkpointed data significantly speeds up CV by avoiding repeated
        data loading and splitting operations.
    """
    return spark.read.parquet(f"{folder_path}/{name}.parquet")


def _build_time_split(df, train_months_end=9):
    """
    Build a simple time-based train/test split from a 1-year DataFrame.
    
    Uses the `month` column (1–12) to create a chronological split:
        - Training set: months 1 through train_months_end (inclusive)
        - Test set: months (train_months_end + 1) through 12
    
    Args:
        df (DataFrame): Full 1-year dataset with `month` column
        train_months_end (int): Last month to include in training (default: 9 for 3/4 split)
    
    Returns:
        tuple: (train_df, test_df) - Training and test DataFrames
    
    Example:
        With train_months_end=9:
        - Train: months 1-9 (first 3/4 of year)
        - Test: months 10-12 (last 1/4 of year)
    """
    train_df = df.filter(col("month") <= train_months_end)
    test_df = df.filter(col("month") > train_months_end)
    return train_df, test_df


# ============================================================================
# Baseline Estimator Class
# ============================================================================

class BaselineEstimator:
    """
    Baseline Linear Regression Estimator with Feature Engineering Pipeline.
    
    This class encapsulates the entire feature engineering and modeling pipeline:
        1. Data preparation (label cleaning, feature selection)
        2. Numerical feature cleaning (remove non-numeric chars, handle nulls)
        3. Median imputation for numerical features
        4. Categorical encoding (StringIndexer + OneHotEncoder)
        5. Feature assembly and standardization
        6. Linear regression modeling
    
    Feature Families:
        - Temporal: day_of_week, month, dep_time_blk
        - Airport: origin, dest
        - Flight: op_unique_carrier, distance
        - Weather: hourlywindspeed, hourlyvisibility, hourlyprecipitation
    
    Why This Design:
        - Encapsulation: All preprocessing logic in one place
        - Reusability: Same pipeline for train/val/test
        - Spark ML compatibility: Uses Pipeline for efficient execution
    """
    
    def __init__(self, label_col="dep_delay"):
        """
        Initialize the estimator with feature definitions.
        
        Args:
            label_col (str): Name of the target variable column
        """
        self.label_col = label_col
        self.pipeline = None
        self.model = None
        
        # Categorical features: Encoded using StringIndexer + OneHotEncoder
        # These capture temporal patterns, route characteristics, and carrier effects
        self.categorical_features = [
            "day_of_week",        # Day of week (1=Monday, 7=Sunday)
            "month",              # Month of year (1-12)
            "dep_time_blk",       # Departure time block (e.g., "0600-0659")
            "origin",             # Origin airport code
            "dest",               # Destination airport code
            "op_unique_carrier"   # Operating carrier code
        ]
        
        # Numerical features: Weather conditions and flight distance
        self.numerical_features = [
            "hourlywindspeed",       # Wind speed at origin (mph)
            "hourlyvisibility",      # Visibility at origin (miles)
            "hourlyprecipitation",   # Precipitation at origin (inches)
            "distance"               # Flight distance (miles)
        ]

    def _prepare(self, df):
        """
        Prepare the DataFrame for modeling by cleaning the label and numerical features.
        
        Steps:
            1. Cast label to DoubleType (required by Spark ML)
            2. Filter out rows with null/NaN labels (Spark ML requirement)
            3. Select only required features + label
            4. Clean numerical features:
               - Remove non-numeric characters (e.g., "12.5mph" -> "12.5")
               - Convert empty strings to null
               - Cast to DoubleType
        
        Args:
            df (pyspark.sql.DataFrame): Input DataFrame
            
        Returns:
            pyspark.sql.DataFrame: Cleaned DataFrame
            
        Why This Matters:
            - Spark ML LinearRegression requires DoubleType labels with no nulls
            - Numerical features may contain string artifacts from source data
            - Explicit type casting prevents downstream pipeline errors
        """
        # Cast label to double and filter out null/NaN values
        # Spark ML does not accept null labels
        df = df.withColumn(self.label_col, col(self.label_col).cast(DoubleType()))
        df = df.filter(~(col(self.label_col).isNull() | isnan(col(self.label_col))))

        # Select only the columns we need (features + label)
        # This reduces memory footprint and prevents accidental data leakage
        selected = [c for c in (self.categorical_features + self.numerical_features + [self.label_col]) 
                    if c in df.columns]
        df = df.select(*selected)

        # Clean numerical features: remove non-numeric characters, handle empty strings
        for f in self.numerical_features:
            if f in df.columns:
                # Step 1: Cast to string to enable regex operations
                # Step 2: Remove all non-numeric characters except +, -, and .
                df = df.withColumn(f, regexp_replace(col(f).cast(StringType()), r"[^0-9+\-\.]", ""))
                
                # Step 3: Convert empty strings to null (for imputation)
                df = df.withColumn(f, when(length(trim(col(f))) == 0, None).otherwise(col(f)))
                
                # Step 4: Cast to DoubleType for modeling
                df = df.withColumn(f, col(f).cast(DoubleType()))
        
        return df

    def _build_pipeline(self, df):
        """
        Build the Spark ML Pipeline with all feature engineering stages.
        
        Pipeline Stages:
            1. Drop nulls: Remove rows with missing numerical features
            2. StringIndexers: Convert categorical strings to indices
            3. OneHotEncoders: Convert indices to binary vectors
            4. VectorAssembler: Combine all features into single vector
            5. StandardScaler: Standardize features (mean=0, std=1)
            6. LinearRegression: Train linear model
        
        Args:
            df (pyspark.sql.DataFrame): Prepared DataFrame
            
        Returns:
            pyspark.sql.DataFrame: DataFrame (may have additional columns from transformations)
            
        Why This Design:
            - Pipeline ensures consistent transformations across train/val/test
            - Dropping nulls instead of imputing for cleaner baseline
            - OneHotEncoding with dropLast=True prevents multicollinearity
            - StandardScaler improves convergence for gradient descent
            - No regularization (regParam=0) for interpretable baseline
        """
        stages = []
        
        # ========================================
        # Stage 1: Drop Null Values in Numerical Features
        # ========================================
        # Drop rows with any null values in numerical features
        # This ensures we only train on complete cases
        for f in self.numerical_features:
            if f in df.columns:
                df = df.filter(col(f).isNotNull())
        
        # Repartition after filtering to rebalance data across executors
        # Filtering can create skewed partitions (some empty, some overloaded)
        # Repartitioning ensures efficient parallel processing
        # Use 8 partitions to match available cores (2 executors × 4 cores = 8 total)
        df = df.repartition(8)
        
        # Rename numerical features to match expected naming convention
        # (no "_imputed" suffix since we're not imputing)
        for f in self.numerical_features:
            if f in df.columns:
                df = df.withColumnRenamed(f, f"{f}_imputed")

        # ========================================
        # Stage 2: Categorical Encoding (StringIndexer + OneHotEncoder)
        # ========================================
        # StringIndexer: Converts strings to numeric indices (most frequent = 0)
        # OneHotEncoder: Converts indices to binary vectors (prevents ordinal assumption)
        # handleInvalid="keep": Unseen categories in test set get their own index
        # dropLast=True: Drop last category to prevent multicollinearity
        for f in self.categorical_features:
            if f in df.columns:
                # For numeric-coded categoricals (day_of_week, month, dep_time_blk),
                # cast to string first to ensure consistent handling
                if f in ["day_of_week", "month", "dep_time_blk"]:
                    df = df.withColumn(f"{f}_clean", 
                                      when(col(f).isNull(), "UNKNOWN").otherwise(col(f).cast(StringType())))
                else:
                    # For string categoricals (ORIGIN, DEST, CARRIER), handle nulls only
                    df = df.withColumn(f"{f}_clean", 
                                      when(col(f).isNull(), "UNKNOWN").otherwise(col(f)))
                
                # Add StringIndexer stage
                stages.append(StringIndexer(inputCol=f"{f}_clean", 
                                           outputCol=f"{f}_indexed", 
                                           handleInvalid="keep"))
                
                # Add OneHotEncoder stage
                stages.append(OneHotEncoder(inputCols=[f"{f}_indexed"], 
                                           outputCols=[f"{f}_encoded"], 
                                           dropLast=True))

        # ========================================
        # Stage 3: Feature Assembly
        # ========================================
        # Combine all features (numerical + encoded categorical) into single vector
        # handleInvalid="skip": Skip rows with invalid values
        feature_columns = [f"{f}_imputed" for f in self.numerical_features if f in df.columns] + \
                          [f"{f}_encoded" for f in self.categorical_features if f in df.columns]
        assembler = VectorAssembler(inputCols=feature_columns, 
                                    outputCol="features", 
                                    handleInvalid="skip")
        
        # ========================================
        # Stage 4: Feature Standardization
        # ========================================
        # Standardize features to mean=0, std=1
        # withStd=True: Scale to unit variance
        # withMean=True: Center to zero mean
        # Why? Improves convergence and makes coefficients comparable
        scaler = StandardScaler(inputCol="features", 
                               outputCol="scaled_features", 
                               withStd=True, 
                               withMean=True)
        
        # ========================================
        # Stage 5: Linear Regression
        # ========================================
        # Baseline model: No regularization (regParam=0, elasticNetParam=0)
        # maxIter=100: Maximum iterations for convergence
        # Why no regularization? We want an interpretable baseline to understand
        # feature importance before adding complexity
        lr = LinearRegression(featuresCol="scaled_features", 
                            labelCol=self.label_col, 
                            maxIter=100, 
                            regParam=0.0, 
                            elasticNetParam=0.0)

        # Assemble all stages into pipeline
        stages.extend([assembler, scaler, lr])
        self.pipeline = Pipeline(stages=stages)
        
        return df

    def fit(self, df):
        """
        Fit the pipeline on training data.
        
        Args:
            df (pyspark.sql.DataFrame): Training DataFrame
            
        Returns:
            BaselineEstimator: self (for method chaining)
        """
        df_prep = self._prepare(df)
        df_prep = self._build_pipeline(df_prep)
        self.model = self.pipeline.fit(df_prep)
        return self

    def transform(self, df):
        """
        Transform data using the fitted pipeline.
        
        Args:
            df (pyspark.sql.DataFrame): DataFrame to transform (val/test)
            
        Returns:
            pyspark.sql.DataFrame: Transformed DataFrame with predictions
            
        Note:
            We rebuild the pipeline on the input DataFrame to ensure all
            transformation columns are present, but use the fitted model
            for predictions.
        """
        df_prep = self._prepare(df)
        df_prep = self._build_pipeline(df_prep)  # Rebuild to ensure cols present
        return self.model.transform(df_prep)


# ============================================================================
# Train/Test Split Runner
# ============================================================================

def run_train_test_split(df_1y, train_months_end=9, include_classification_metrics=True):
    """
    Run a simple time-based train/test split for the baseline model on 1-year data.
    
    Process:
        1. Split 1-year DataFrame by month: Train (months 1-9), Test (months 10-12)
        2. Train model on training set
        3. Evaluate on both training and test sets
        4. Compute regression and classification metrics
        5. Report comprehensive metrics, feature importance, and runtime
    
    Args:
        df_1y (DataFrame): Full 1-year joined dataset with `month` and `dep_delay`
        train_months_end (int): Last month to include in training (default: 9 for 3/4 split)
        include_classification_metrics (bool): Whether to compute OTPA, SDDR, etc.
        
    Returns:
        tuple: (model, train_metrics_dict, test_metrics_dict, metrics_df,
                metrics_defs, classification_df, test_class_results)
    
    Memory Management Strategy:
        - Minimal caching to prevent OOM errors
        - Explicit garbage collection after predictions
    """
    # ========================================
    # Initialize Timer and Build Split
    # ========================================
    overall_start_time = time.time()
    # Build time-based split: Train on first 3/4 of year, test on last 1/4
    train_df, test_df = _build_time_split(df_1y, train_months_end=train_months_end)
    
    print(f"Training set: months 1-{train_months_end}")
    print(f"Test set: months {train_months_end + 1}-12")
    print(f"Training rows: {train_df.count():,}")
    print(f"Test rows: {test_df.count():,}")

    # ========================================
    # Initialize Evaluators
    # ========================================
    # Create evaluators for each regression metric
    # These will be reused for both train and test sets
    eval_rmse = RegressionEvaluator(predictionCol="prediction", labelCol="dep_delay", metricName="rmse")
    eval_mae  = RegressionEvaluator(predictionCol="prediction", labelCol="dep_delay", metricName="mae")
    eval_r2   = RegressionEvaluator(predictionCol="prediction", labelCol="dep_delay", metricName="r2")
    eval_mse  = RegressionEvaluator(predictionCol="prediction", labelCol="dep_delay", metricName="mse")

    # ========================================
    # Initialize Estimator
    # ========================================
    est = BaselineEstimator(label_col="dep_delay")

    # ========================================
    # Train Model
    # ========================================
    print("\n" + "="*80)
    print("TRAINING MODEL")
    print("="*80)
    model = est.fit(train_df)
    print("Model training completed!")

    # ========================================
    # Generate Predictions
    # ========================================
    # Generate predictions on both training and test sets
    # Training metrics help detect overfitting (train << test indicates generalization)
    
    print("\n" + "="*80)
    print("GENERATING PREDICTIONS")
    print("="*80)
    
    # Training predictions
    train_preds = model.transform(train_df)
    # train_preds.cache().count()  # Disabled to prevent OOM
    
    # Test predictions
    test_preds = model.transform(test_df)
    # test_preds.cache().count()  # Disabled to prevent OOM

    # ========================================
    # Compute Regression Metrics (Training)
    # ========================================
    # Training metrics help detect overfitting
    # If train metrics << test metrics, model is overfitting
    print("\n" + "="*80)
    print("TRAINING SET METRICS")
    print("="*80)
    train_rmse = eval_rmse.evaluate(train_preds)
    train_mae  = eval_mae.evaluate(train_preds)
    train_r2   = eval_r2.evaluate(train_preds)
    train_mse  = eval_mse.evaluate(train_preds)
    
    print(f"RMSE: {train_rmse:.2f}  MAE: {train_mae:.2f}  R²: {train_r2:.4f}  MSE: {train_mse:.2f}")

    # ========================================
    # Compute Regression Metrics (Test)
    # ========================================
    # Test metrics provide an unbiased estimate of production performance
    print("\n" + "="*80)
    print("TEST SET METRICS")
    print("="*80)
    test_rmse = eval_rmse.evaluate(test_preds)
    test_mae  = eval_mae.evaluate(test_preds)
    test_r2   = eval_r2.evaluate(test_preds)
    test_mse  = eval_mse.evaluate(test_preds)
    
    print(f"RMSE: {test_rmse:.2f}  MAE: {test_mae:.2f}  R²: {test_r2:.4f}  MSE: {test_mse:.2f}")
    
    # ========================================
    # Compute Classification Metrics (Optional)
    # ========================================
    # Classification metrics provide operational insights:
    # - OTPA: On-Time Performance Accuracy (<15 min threshold)
    # - SDDR: Severe Delay Detection Rate (≥60 min threshold)
    # - Bucket Accuracy: 4-bucket classification (Early, OnTime, Delayed, Severe)
    classification_records = []
    
    if include_classification_metrics:
        # Training set classification metrics
        print("\n" + "="*80)
        print("TRAINING SET - CLASSIFICATION METRICS")
        print("="*80)
        train_class_results, train_class_df = compute_classification_metrics(
            train_preds, label_col="dep_delay", prediction_col="prediction"
        )
        print_classification_summary(train_class_results, train_class_df)
        classification_records.append({
            "split": "train",
            "otpa_accuracy": train_class_results["otpa_accuracy"],
            "otpa_f1": train_class_results["otpa_f1"],
            "sddr_recall": train_class_results["sddr_recall"],
            "bucket_accuracy": train_class_results["bucket_accuracy"]
        })
        
        # Test set classification metrics
        print("\n" + "="*80)
        print("TEST SET - CLASSIFICATION METRICS")
        print("="*80)
        test_class_results, test_class_df = compute_classification_metrics(
            test_preds, label_col="dep_delay", prediction_col="prediction"
        )
        print_classification_summary(test_class_results, test_class_df)
        classification_records.append({
            "split": "test",
            "otpa_accuracy": test_class_results["otpa_accuracy"],
            "otpa_f1": test_class_results["otpa_f1"],
            "sddr_recall": test_class_results["sddr_recall"],
            "bucket_accuracy": test_class_results["bucket_accuracy"]
        })
    else:
        test_class_results = None

    # ========================================
    # Build Consolidated Metrics DataFrames
    # ========================================
    # Combine train/test metrics into single DataFrame for easy comparison
    metrics_df = pd.DataFrame([
        {"split": "train", "rmse": train_rmse, "mae": train_mae, "r2": train_r2, "mse": train_mse},
        {"split": "test", "rmse": test_rmse, "mae": test_mae, "r2": test_r2, "mse": test_mse}
    ])
    
    # Build classification metrics DataFrame if computed
    classification_df = pd.DataFrame(classification_records) if include_classification_metrics else None
    
    # Round to 4 decimal places for readability
    metrics_df['rmse'] = metrics_df['rmse'].round(4)
    metrics_df['mae'] = metrics_df['mae'].round(4)
    metrics_df['r2'] = metrics_df['r2'].round(4)
    metrics_df['mse'] = metrics_df['mse'].round(4)

    # ========================================
    # Metrics Definitions with LaTeX
    # ========================================
    # Provide LaTeX equations for each metric for documentation/reporting
    metrics_defs = pd.DataFrame([
        {"metric": "RMSE", "latex": r"\\mathrm{RMSE} = \\sqrt{\\tfrac{1}{N} \\sum_{i=1}^{N} (y_i - \\hat{y}_i)^2}"},
        {"metric": "MAE",  "latex": r"\\mathrm{MAE} = \\tfrac{1}{N} \\sum_{i=1}^{N} |y_i - \\hat{y}_i|"},
        {"metric": "R^2",  "latex": r"R^2 = 1 - \\dfrac{\\sum_{i=1}^{N} (y_i - \\hat{y}_i)^2}{\\sum_{i=1}^{N} (y_i - \\bar{y})^2}"},
        {"metric": "MSE",  "latex": r"\\mathrm{MSE} = \\tfrac{1}{N} \\sum_{i=1}^{N} (y_i - \\hat{y}_i)^2"},
    ])

    # ========================================
    # Extract Feature Importance (Coefficients)
    # ========================================
    # Extract and display top 10 most important features by absolute coefficient
    # Helps understand which features drive predictions
    try:
        # Extract coefficients from the linear regression model (last stage in pipeline)
        coefficients = model.model.stages[-1].coefficients.toArray()
        
        # Try to extract feature names from metadata (includes OHE expansions)
        feats_meta = test_preds.schema["features"].metadata
        attrs = []
        if "ml_attr" in feats_meta and "attrs" in feats_meta["ml_attr"]:
            ml_attrs = feats_meta["ml_attr"]["attrs"]
            # Collect attributes from all types (binary, numeric, nominal)
            for t in ["binary", "numeric", "nominal"]:
                if t in ml_attrs:
                    attrs.extend(ml_attrs[t])
            # Sort by vector index to match coefficient order
            attrs = sorted(attrs, key=lambda x: x["idx"])
            feature_names = [a.get("name", f"feature_{a['idx']}") for a in attrs]
        else:
            # Fallback: use generic names if metadata unavailable
            feature_names = [f"feature_{i}" for i in range(len(coefficients))]
        
        # Safety check: ensure lengths match
        if len(feature_names) != len(coefficients):
            feature_names = [f"feature_{i}" for i in range(len(coefficients))]
        
        # Create DataFrame with feature names and coefficients
        coef_df = pd.DataFrame({"feature": feature_names, "coefficient": coefficients})
        coef_df["abs_coefficient"] = coef_df["coefficient"].abs()
        coef_df = coef_df.sort_values("abs_coefficient", ascending=False)
        
        # Round coefficients to 4 decimal places
        top_coef_df = coef_df.head(10).copy()
        top_coef_df['coefficient'] = top_coef_df['coefficient'].round(4)
        top_coef_df['abs_coefficient'] = top_coef_df['abs_coefficient'].round(4)
        
        print("\n" + "="*80)
        print("Top 10 Most Important Features (by absolute coefficient):")
        print("="*80)
        print(top_coef_df.to_string(index=False))
    except Exception as e:
        print(f"[Info] Skipping coefficient listing: {e}")
    
    # ========================================
    # Print Total Runtime
    # ========================================
    # Report total execution time for performance tracking
    overall_elapsed = time.time() - overall_start_time
    hours, remainder = divmod(overall_elapsed, 3600)
    minutes, seconds = divmod(remainder, 60)
    print("\n" + "="*80)
    print(f"TOTAL RUNTIME: {int(hours)}h {int(minutes)}m {seconds:.2f}s ({overall_elapsed:.2f} seconds)")
    print("="*80)

    # ========================================
    # Free Memory
    # ========================================
    # Force Python garbage collection to free driver memory
    gc.collect()
    
    # Clear Spark SQL cache to free executor memory
    spark.catalog.clearCache()

    # ========================================
    # Return Results
    # ========================================
    train_metrics_dict = {"rmse": train_rmse, "mae": train_mae, "r2": train_r2, "mse": train_mse}
    test_metrics_dict = {"rmse": test_rmse, "mae": test_mae, "r2": test_r2, "mse": test_mse}
    return model, train_metrics_dict, test_metrics_dict, metrics_df, metrics_defs, classification_df, test_class_results if include_classification_metrics else None


# ============================================================================
# Main Execution Block
# ============================================================================

if __name__ == "__main__":
    """
    Main execution entry point for the cross-validation script.
    
    Workflow:
        1. Print cluster configuration (for reproducibility and debugging)
        2. Run expanding window cross-validation
        3. Display comprehensive results (regression + classification metrics)
    
    Note:
        This block only runs when the script is executed directly,
        not when imported as a module.
    """
    
    # ========================================
    # Print Cluster Configuration
    # ========================================
    # Display cluster resources to ensure adequate capacity and enable reproducibility
    # Important for:
    # - Debugging OOM errors (check if cluster is undersized)
    # - Reproducing results (document exact cluster configuration)
    # - Cost tracking (understand resource usage)
    
    sc = spark.sparkContext
    
    # Get number of executors (exclude driver node)
    # statusTracker().getExecutorInfos() returns list of all executors
    num_executors = len(sc._jsc.sc().statusTracker().getExecutorInfos()) - 1
    
    # Get executor configuration from Spark conf
    executor_memory = sc.getConf().get("spark.executor.memory", "Unknown")
    executor_cores = sc.getConf().get("spark.executor.cores", "Unknown")
    actual_cores = sc.defaultParallelism / num_executors if num_executors > 0 else "Unknown"
    print(f"Estimated cores per executor: {int(actual_cores)}")
    
    # Print cluster configuration
    print("\n" + "="*80)
    print("CLUSTER CONFIGURATION")
    print("="*80)
    print(f"Cluster Size: {num_executors} executors, {executor_cores} cores per executor, {executor_memory} RAM per executor")
    print("="*80 + "\n")
    
    # ========================================
    # Load 1-year joined dataset
    # ========================================
    df_1y = spark.read.parquet("/mnt/mids-w261/student-groups/Group_4_2/processed/flights_weather_joined_1y")
    
    # Optional: select only the columns needed by the baseline estimator
    # (day_of_week, month, dep_time_blk, origin, dest, op_unique_carrier,
    #  hourlywindspeed, hourlyvisibility, hourlyprecipitation, distance, dep_delay)
    df_1y = df_1y.select(
        "day_of_week",
        "month",
        "dep_time_blk",
        "origin",
        "dest",
        "op_unique_carrier",
        "hourlywindspeed",
        "hourlyvisibility",
        "hourlyprecipitation",
        "distance",
        "dep_delay",
    )
    
    # ========================================
    # Run Train/Test Split
    # ========================================
    # Execute time-based split (first 3/4 for training, last 1/4 for testing)
    # include_classification_metrics=True: Compute OTPA, SDDR, bucket accuracy
    
    model, train_metrics_dict, test_metrics_dict, metrics_df, metrics_defs, classification_df, test_class_results = run_train_test_split(
        df_1y,
        train_months_end=9,  # Train on months 1-9, test on months 10-12
        include_classification_metrics=True,
    )
    
    # ========================================
    # Display Regression Metrics
    # ========================================
    # Show comprehensive regression metrics across train/val/test splits
    # metrics_df: Per-fold metrics (split, fold, RMSE, MAE, R², MSE)
    # metrics_defs: LaTeX equations for each metric (for documentation)
    
    print("\n" + "="*80)
    print("REGRESSION METRICS")
    print("="*80)
    display(metrics_df)  # Use display() for interactive Databricks table
    display(metrics_defs)
    
    # ========================================
    # Display Classification Metrics
    # ========================================
    # Show operational classification metrics if computed
    # Includes OTPA (on-time accuracy), SDDR (severe delay detection),
    # and 4-bucket classification accuracy
    
    if classification_df is not None:
        print("\n" + "="*80)
        print("CLASSIFICATION METRICS SUMMARY")
        print("="*80)
        display(classification_df)  # Use display() for interactive Databricks table

In [0]:
displayHTML("""

<!DOCTYPE html>
<html>
<head>
  <script src="https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.min.js"></script>
  <script>
    mermaid.initialize({
      startOnLoad: true,
      theme: 'dark',
      themeVariables: {
        primaryColor: '#4a5568',
        primaryTextColor: '#fff',
        primaryBorderColor: '#cbd5e0',
        lineColor: '#cbd5e0',
        secondaryColor: '#2d3748',
        tertiaryColor: '#1a202c',
        background: '#1a202c',
        mainBkg: '#4a5568',
        secondBkg: '#2d3748',
        tertiaryBkg: '#1a202c'
      }
    });
  </script>
  <style>
    body { background-color: #1a202c; }
    .mermaid { background-color: #1a202c; }
  </style>
</head>
<body>

<div class="mermaid">
flowchart LR

    %% Input
    Input["<b>Input</b><br/>1-Year Joined DataFrame (df_1y)<br/>Features:<br/>day_of_week, month, dep_time_blk,<br/>origin, dest, op_unique_carrier,<br/>hourlywindspeed, hourlyvisibility,<br/>hourlyprecipitation, distance<br/>Label: dep_delay"]

    %% Stage 0: Time-Based Split
    subgraph S0["<b>Stage 0: Time-Based Split</b>"]
        Split["Split by month:<br/><b>Train</b>: months 1–9<br/><b>Test</b>: months 10–12"]
    end

    %% Stage 1: Data Preparation
    subgraph S1["<b>Stage 1: Data Preparation</b>"]
        LabelClean["Cast dep_delay to Double<br/>Filter Null/NaN Labels"]
        SelectFeat["Select Baseline Features<br/>Temporal, Airport, Flight, Weather"]
        NumClean["Clean Numerical Features:<br/>hourlywindspeed, hourlyvisibility,<br/>hourlyprecipitation, distance<br/>Remove non-numeric chars<br/>Empty → Null<br/>Cast to Double"]
        LabelClean --> SelectFeat --> NumClean
    end

    %% Stage 2: Drop Null Numerical Rows (No Imputation)
    subgraph S2["<b>Stage 2: Drop Null Numerical Rows</b>"]
        DropNulls["Filter rows where any of:<br/>hourlywindspeed, hourlyvisibility,<br/>hourlyprecipitation, distance<br/>is Null<br/><br/>Rename to:<br/>*_imputed (no actual imputation)"]
    end

    %% Stage 3: Categorical Encoding
    subgraph S3["<b>Stage 3: Categorical Encoding</b>"]
        DOW["day_of_week_clean<br/>StringIndexer + OHE"]
        Month["month_clean<br/>StringIndexer + OHE"]
        TimeBlk["dep_time_blk_clean<br/>StringIndexer + OHE"]
        Origin["origin_clean<br/>StringIndexer + OHE"]
        Dest["dest_clean<br/>StringIndexer + OHE"]
        Carrier["op_unique_carrier_clean<br/>StringIndexer + OHE"]
    end

    %% Stage 4: Feature Assembly
    subgraph S4["<b>Stage 4: Feature Assembly</b>"]
        Assemble["VectorAssembler<br/>Combine Numerical + Encoded Categorical<br/><br/>Numerical inputs:<br/>hourlywindspeed_imputed,<br/>hourlyvisibility_imputed,<br/>hourlyprecipitation_imputed,<br/>distance_imputed"]
    end

    %% Stage 5: Standardization
    subgraph S5["<b>Stage 5: Standardization</b>"]
        Scale["StandardScaler<br/>features → scaled_features<br/>withMean=True, withStd=True"]
    end

    %% Stage 6: Linear Regression
    subgraph S6["<b>Stage 6: Model</b>"]
        LR["Linear Regression<br/>featuresCol=scaled_features<br/>labelCol=dep_delay<br/>maxIter=100<br/>regParam=0.0, elasticNet=0.0"]
    end

    %% Output & Metrics
    Output["<b>Output</b><br/>Train & Test Predictions<br/>dep_delay in minutes<br/><br/>Metrics:<br/>RMSE, MAE, R², MSE<br/>OTPA (Accuracy, F1)<br/>SDDR Recall<br/>4-Bucket Accuracy<br/>Top 10 Coefficients"]

    %% Connections
    Input --> Split
    Split --> S1
    S1 --> S2
    S1 --> S3
    S2 --> Assemble
    S3 --> Assemble
    Assemble --> Scale --> LR --> Output

</div>

</body>
</html>

""")

# Model 6 Model 6 - No-Cross-Validation/3/4Train-1/4Test-Split/12-month/Imputed-Null/Default-Data
Updated: 3/4 Train 1/4 Test on default data null imputed

In [0]:
# Read the gzipped CSV from the class mount
df_otpw12 = (spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("compression", "gzip")
    .load("dbfs:/mnt/mids-w261/OTPW_12M/OTPW_12M/OTPW_12M_2015.csv.gz")
)

# Write Parquet into your team folder
out_path = "dbfs:/student-groups/Group_4_2/OTPW_12M_2015_parquet"

df_otpw12.write.mode("overwrite").parquet(out_path)

In [0]:
df_otpw12_parq = spark.read.parquet("dbfs:/student-groups/Group_4_2/OTPW_12M_2015_parquet")

In [0]:
print(f"Total rows: {df_otpw12_parq.count()}")
display(df_otpw12_parq.limit(10))

In [0]:
"""
================================================================================
Time-Based Train/Test Split for Baseline Linear Regression (12-Month Dataset)
================================================================================

Purpose:
    Implements a simple time-based split (first 3/4 of year for training,
    last 1/4 for testing) for flight delay prediction using a baseline
    linear regression model with comprehensive feature engineering.

Key Features:
    - Time-based split: Train on months 1-9, test on months 10-12
    - Baseline pipeline: Numeric cleaning + drop nulls + categorical OHE + standardization + LR
    - Comprehensive metrics: Regression (RMSE, MAE, R², MSE) + Classification (OTPA, SDDR, 4-bucket)
    - Memory management: Minimal caching to prevent OOM errors

Data Flow:
    1. Load 1-year joined dataset from DBFS
    2. Split by month: Train (months 1-9), Test (months 10-12)
    3. Train model on training set
    4. Evaluate on both training and test sets
    5. Report comprehensive metrics and feature importance

Author: Emily Lieske
Date: November 2025
================================================================================
"""

# ============================================================================
# Imports
# ============================================================================

# PySpark SQL functions for data transformation
from pyspark.sql.functions import col, when, isnan, regexp_replace, trim, length
from pyspark.sql.types import DoubleType, StringType

# PySpark ML for pipeline construction and modeling
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder, StandardScaler, Imputer
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

# Standard libraries for metrics and timing
import numpy as np
import pandas as pd
import time
import gc  # Garbage collection for memory management

# ============================================================================
# Data Loading Functions
# ============================================================================

def _load_checkpointed_data(name, folder_path="dbfs:/student-groups/Group_4_2"):
    """
    Load a pre-checkpointed Parquet dataset from DBFS.
    
    Args:
        name (str): Dataset name (e.g., "OTPW_12M_FOLD_1_TRAIN")
        folder_path (str): DBFS path to the folder containing checkpointed data
        
    Returns:
        pyspark.sql.DataFrame: Loaded dataset
        
    Note:
        Pre-checkpointed data significantly speeds up CV by avoiding repeated
        data loading and splitting operations.
    """
    return spark.read.parquet(f"{folder_path}/{name}.parquet")


def _build_time_split(df, train_months_end=9):
    """
    Build a simple time-based train/test split from a 1-year (12-month) DataFrame.
    
    Uses the month column (1–12) to create a chronological split:
        - Training set: months 1 through train_months_end (inclusive)
        - Test set: months (train_months_end + 1) through 12
    
    The function is resilient to different month column names and will use
    "month" if present, otherwise "MONTH".
    
    Args:
        df (DataFrame): Full 1-year dataset with a month column
        train_months_end (int): Last month to include in training (default: 9 for 3/4 split)
    
    Returns:
        tuple: (train_df, test_df) - Training and test DataFrames
    """
    if "month" in df.columns:
        month_col = "month"
    elif "MONTH" in df.columns:
        month_col = "MONTH"
    else:
        raise ValueError("No month / MONTH column found for time-based split.")
    
    train_df = df.filter(col(month_col) <= train_months_end)
    test_df = df.filter(col(month_col) > train_months_end)
    return train_df, test_df


# ============================================================================
# Baseline Estimator Class
# ============================================================================

class BaselineEstimator:
    """
    Baseline Linear Regression Estimator with Feature Engineering Pipeline.
    
    This class encapsulates the entire feature engineering and modeling pipeline:
        1. Data preparation (label cleaning, feature selection)
        2. Numerical feature cleaning (remove non-numeric chars, handle nulls)
        3. Median imputation for numerical features
        4. Categorical encoding (StringIndexer + OneHotEncoder)
        5. Feature assembly and standardization
        6. Linear regression modeling
    
    Feature Families (OTPW 12M, 2015 schema):
        - Temporal: DAY_OF_WEEK, MONTH, DEP_TIME_BLK
        - Airport: ORIGIN, DEST
        - Flight: OP_UNIQUE_CARRIER, DISTANCE
        - Weather: HourlyWindSpeed, HourlyVisibility, HourlyPrecipitation
    
    Why This Design:
        - Encapsulation: All preprocessing logic in one place
        - Reusability: Same pipeline for train/val/test
        - Spark ML compatibility: Uses Pipeline for efficient execution
    """
    
    def __init__(self, label_col="DEP_DELAY"):
        """
        Initialize the estimator with feature definitions.
        
        Args:
            label_col (str): Name of the target variable column
        """
        self.label_col = label_col
        self.pipeline = None
        self.model = None
        
        # Categorical features for OTPW 12M (uppercase schema)
        self.categorical_features = [
            "DAY_OF_WEEK",        # Day of week (1=Monday, 7=Sunday)
            "MONTH",              # Month of year (1-12)
            "DEP_TIME_BLK",       # Departure time block (e.g., "0600-0659")
            "ORIGIN",             # Origin airport code
            "DEST",               # Destination airport code
            "OP_UNIQUE_CARRIER"   # Operating carrier code
        ]
        
        # Numerical features (uppercase schema)
        self.numerical_features = [
            "HourlyWindSpeed",       # Wind speed at origin (mph)
            "HourlyVisibility",      # Visibility at origin (miles)
            "HourlyPrecipitation",   # Precipitation at origin (inches)
            "DISTANCE"               # Flight distance (miles)
        ]

    def _prepare(self, df):
        """
        Prepare the DataFrame for modeling by cleaning the label and numerical features.
        
        Steps:
            1. Cast label to DoubleType (required by Spark ML)
            2. Filter out rows with null/NaN labels (Spark ML requirement)
            3. Select only required features + label
            4. Clean numerical features:
               - Remove non-numeric characters (e.g., "12.5mph" -> "12.5")
               - Convert empty strings to null
               - Cast to DoubleType
        
        Args:
            df (pyspark.sql.DataFrame): Input DataFrame
            
        Returns:
            pyspark.sql.DataFrame: Cleaned DataFrame
            
        Why This Matters:
            - Spark ML LinearRegression requires DoubleType labels with no nulls
            - Numerical features may contain string artifacts from source data
            - Explicit type casting prevents downstream pipeline errors
        """
        # Cast label to double and filter out null/NaN values
        df = df.withColumn(self.label_col, col(self.label_col).cast(DoubleType()))
        df = df.filter(~(col(self.label_col).isNull() | isnan(col(self.label_col))))

        # Select only the columns we need (features + label)
        selected = [
            c
            for c in (self.categorical_features + self.numerical_features + [self.label_col])
            if c in df.columns
        ]
        df = df.select(*selected)

        # Clean numerical features: remove non-numeric characters, handle empty strings
        for f in self.numerical_features:
            if f in df.columns:
                # Step 1: Cast to string to enable regex operations
                # Step 2: Remove all non-numeric characters except +, -, and .
                df = df.withColumn(f, regexp_replace(col(f).cast(StringType()), r"[^0-9+\-\.]", ""))
                # Step 3: Convert empty strings to null (for imputation)
                df = df.withColumn(f, when(length(trim(col(f))) == 0, None).otherwise(col(f)))
                # Step 4: Cast to DoubleType for modeling
                df = df.withColumn(f, col(f).cast(DoubleType()))
        
        return df

    def _build_pipeline(self, df):
        """
        Build the Spark ML Pipeline with all feature engineering stages.
        
        Pipeline Stages:
            1. Imputers: Median imputation for numerical features
            2. StringIndexers: Convert categorical strings to indices
            3. OneHotEncoders: Convert indices to binary vectors
            4. VectorAssembler: Combine all features into single vector
            5. StandardScaler: Standardize features (mean=0, std=1)
            6. LinearRegression: Train linear model
        
        Args:
            df (pyspark.sql.DataFrame): Prepared DataFrame
            
        Returns:
            pyspark.sql.DataFrame: DataFrame (may have additional columns from transformations)
            
        Why This Design:
            - Matches Model 3: median imputation instead of dropping rows
            - Pipeline ensures consistent transformations across train/test
            - OneHotEncoding with dropLast=True prevents multicollinearity
            - StandardScaler improves convergence for gradient descent
            - No regularization (regParam=0) for interpretable baseline
        """
        stages = []
        
        # ========================================
        # Stage 1: Median Imputation for Numerical Features
        # ========================================
        imputers = [
            Imputer(inputCols=[f], outputCols=[f"{f}_imputed"], strategy="median")
            for f in self.numerical_features
            if f in df.columns
        ]
        stages.extend(imputers)

        # ========================================
        # Stage 2: Categorical Encoding (StringIndexer + OneHotEncoder)
        # ========================================
        # StringIndexer: Converts strings to numeric indices (most frequent = 0)
        # OneHotEncoder: Converts indices to binary vectors (prevents ordinal assumption)
        # handleInvalid="keep": Unseen categories in test set get their own index
        # dropLast=True: Drop last category to prevent multicollinearity
        for f in self.categorical_features:
            if f in df.columns:
                # For numeric-coded categoricals (DAY_OF_WEEK, MONTH, DEP_TIME_BLK),
                # cast to string first to ensure consistent handling
                if f in ["DAY_OF_WEEK", "MONTH", "DEP_TIME_BLK"]:
                    df = df.withColumn(
                        f"{f}_clean",
                        when(col(f).isNull(), "UNKNOWN").otherwise(col(f).cast(StringType())),
                    )
                else:
                    # For string categoricals (ORIGIN, DEST, OP_UNIQUE_CARRIER), handle nulls only
                    df = df.withColumn(
                        f"{f}_clean",
                        when(col(f).isNull(), "UNKNOWN").otherwise(col(f)),
                    )
                
                # Add StringIndexer stage
                stages.append(StringIndexer(inputCol=f"{f}_clean", 
                                           outputCol=f"{f}_indexed", 
                                           handleInvalid="keep"))
                
                # Add OneHotEncoder stage
                stages.append(OneHotEncoder(inputCols=[f"{f}_indexed"], 
                                           outputCols=[f"{f}_encoded"], 
                                           dropLast=True))

        # ========================================
        # Stage 3: Feature Assembly
        # ========================================
        # Combine all features (imputed numerical + encoded categorical) into single vector
        # handleInvalid="skip": Skip rows with invalid values
        feature_columns = [
            f"{f}_imputed" for f in self.numerical_features if f in df.columns
        ] + [
            f"{f}_encoded" for f in self.categorical_features if f in df.columns
        ]
        assembler = VectorAssembler(
            inputCols=feature_columns,
            outputCol="features",
            handleInvalid="skip",
        )
        
        # ========================================
        # Stage 4: Feature Standardization
        # ========================================
        # Standardize features to mean=0, std=1
        # withStd=True: Scale to unit variance
        # withMean=True: Center to zero mean
        # Why? Improves convergence and makes coefficients comparable
        scaler = StandardScaler(inputCol="features", 
                               outputCol="scaled_features", 
                               withStd=True, 
                               withMean=True)
        
        # ========================================
        # Stage 5: Linear Regression
        # ========================================
        # Baseline model: No regularization (regParam=0, elasticNetParam=0)
        # maxIter=100: Maximum iterations for convergence
        # Why no regularization? We want an interpretable baseline to understand
        # feature importance before adding complexity
        lr = LinearRegression(featuresCol="scaled_features", 
                            labelCol=self.label_col, 
                            maxIter=100, 
                            regParam=0.0, 
                            elasticNetParam=0.0)

        # Assemble all stages into pipeline
        stages.extend([assembler, scaler, lr])
        self.pipeline = Pipeline(stages=stages)
        
        return df

    def fit(self, df):
        """
        Fit the pipeline on training data.
        
        Args:
            df (pyspark.sql.DataFrame): Training DataFrame
            
        Returns:
            BaselineEstimator: self (for method chaining)
        """
        df_prep = self._prepare(df)
        df_prep = self._build_pipeline(df_prep)
        self.model = self.pipeline.fit(df_prep)
        return self

    def transform(self, df):
        """
        Transform data using the fitted pipeline.
        
        Args:
            df (pyspark.sql.DataFrame): DataFrame to transform (val/test)
            
        Returns:
            pyspark.sql.DataFrame: Transformed DataFrame with predictions
            
        Note:
            We rebuild the pipeline on the input DataFrame to ensure all
            transformation columns are present, but use the fitted model
            for predictions.
        """
        df_prep = self._prepare(df)
        df_prep = self._build_pipeline(df_prep)  # Rebuild to ensure cols present
        return self.model.transform(df_prep)


# ============================================================================
# Train/Test Split Runner
# ============================================================================

def run_train_test_split(df_1y, train_months_end=9, include_classification_metrics=True):
    """
    Run a simple time-based train/test split for the baseline model on 1-year data.
    
    Process:
        1. Split 1-year DataFrame by month: Train (months 1-9), Test (months 10-12)
        2. Train model on training set
        3. Evaluate on both training and test sets
        4. Compute regression and classification metrics
        5. Report comprehensive metrics, feature importance, and runtime
    
    Args:
        df_1y (DataFrame): Full 1-year joined dataset with `month` and `dep_delay`
        train_months_end (int): Last month to include in training (default: 9 for 3/4 split)
        include_classification_metrics (bool): Whether to compute OTPA, SDDR, etc.
        
    Returns:
        tuple: (model, train_metrics_dict, test_metrics_dict, metrics_df,
                metrics_defs, classification_df, test_class_results)
    
    Memory Management Strategy:
        - Minimal caching to prevent OOM errors
        - Explicit garbage collection after predictions
    """
    # ========================================
    # Initialize Timer and Build Split
    # ========================================
    overall_start_time = time.time()
    # Build time-based split: Train on first 3/4 of year, test on last 1/4
    train_df, test_df = _build_time_split(df_1y, train_months_end=train_months_end)
    
    print(f"Training set: months 1-{train_months_end}")
    print(f"Test set: months {train_months_end + 1}-12")
    print(f"Training rows: {train_df.count():,}")
    print(f"Test rows: {test_df.count():,}")

    # Determine label column name (supports DEP_DELAY or dep_delay)
    if "DEP_DELAY" in df_1y.columns:
        label_col = "DEP_DELAY"
    elif "dep_delay" in df_1y.columns:
        label_col = "dep_delay"
    else:
        raise ValueError("Expected DEP_DELAY or dep_delay column for label, but neither was found.")

    # ========================================
    # Initialize Evaluators
    # ========================================
    eval_rmse = RegressionEvaluator(predictionCol="prediction", labelCol=label_col, metricName="rmse")
    eval_mae  = RegressionEvaluator(predictionCol="prediction", labelCol=label_col, metricName="mae")
    eval_r2   = RegressionEvaluator(predictionCol="prediction", labelCol=label_col, metricName="r2")
    eval_mse  = RegressionEvaluator(predictionCol="prediction", labelCol=label_col, metricName="mse")

    # ========================================
    # Initialize Estimator
    # ========================================
    est = BaselineEstimator(label_col=label_col)

    # ========================================
    # Train Model
    # ========================================
    print("\n" + "="*80)
    print("TRAINING MODEL")
    print("="*80)
    model = est.fit(train_df)
    print("Model training completed!")

    # ========================================
    # Generate Predictions
    # ========================================
    # Generate predictions on both training and test sets
    # Training metrics help detect overfitting (train << test indicates generalization)
    
    print("\n" + "="*80)
    print("GENERATING PREDICTIONS")
    print("="*80)
    
    # Training predictions
    train_preds = model.transform(train_df)
    # train_preds.cache().count()  # Disabled to prevent OOM
    
    # Test predictions
    test_preds = model.transform(test_df)
    # test_preds.cache().count()  # Disabled to prevent OOM

    # ========================================
    # Compute Regression Metrics (Training)
    # ========================================
    # Training metrics help detect overfitting
    # If train metrics << test metrics, model is overfitting
    print("\n" + "="*80)
    print("TRAINING SET METRICS")
    print("="*80)
    train_rmse = eval_rmse.evaluate(train_preds)
    train_mae  = eval_mae.evaluate(train_preds)
    train_r2   = eval_r2.evaluate(train_preds)
    train_mse  = eval_mse.evaluate(train_preds)
    
    print(f"RMSE: {train_rmse:.2f}  MAE: {train_mae:.2f}  R²: {train_r2:.4f}  MSE: {train_mse:.2f}")

    # ========================================
    # Compute Regression Metrics (Test)
    # ========================================
    # Test metrics provide an unbiased estimate of production performance
    print("\n" + "="*80)
    print("TEST SET METRICS")
    print("="*80)
    test_rmse = eval_rmse.evaluate(test_preds)
    test_mae  = eval_mae.evaluate(test_preds)
    test_r2   = eval_r2.evaluate(test_preds)
    test_mse  = eval_mse.evaluate(test_preds)
    
    print(f"RMSE: {test_rmse:.2f}  MAE: {test_mae:.2f}  R²: {test_r2:.4f}  MSE: {test_mse:.2f}")
    
    # ========================================
    # Compute Classification Metrics (Optional)
    # ========================================
    # Classification metrics provide operational insights:
    # - OTPA: On-Time Performance Accuracy (<15 min threshold)
    # - SDDR: Severe Delay Detection Rate (≥60 min threshold)
    # - Bucket Accuracy: 4-bucket classification (Early, OnTime, Delayed, Severe)
    classification_records = []
    
    if include_classification_metrics:
        # Training set classification metrics
        print("\n" + "="*80)
        print("TRAINING SET - CLASSIFICATION METRICS")
        print("="*80)
        train_class_results, train_class_df = compute_classification_metrics(
            train_preds, label_col=label_col, prediction_col="prediction"
        )
        print_classification_summary(train_class_results, train_class_df)
        classification_records.append({
            "split": "train",
            "otpa_accuracy": train_class_results["otpa_accuracy"],
            "otpa_f1": train_class_results["otpa_f1"],
            "sddr_recall": train_class_results["sddr_recall"],
            "bucket_accuracy": train_class_results["bucket_accuracy"]
        })
        
        # Test set classification metrics
        print("\n" + "="*80)
        print("TEST SET - CLASSIFICATION METRICS")
        print("="*80)
        test_class_results, test_class_df = compute_classification_metrics(
            test_preds, label_col=label_col, prediction_col="prediction"
        )
        print_classification_summary(test_class_results, test_class_df)
        classification_records.append({
            "split": "test",
            "otpa_accuracy": test_class_results["otpa_accuracy"],
            "otpa_f1": test_class_results["otpa_f1"],
            "sddr_recall": test_class_results["sddr_recall"],
            "bucket_accuracy": test_class_results["bucket_accuracy"]
        })
    else:
        test_class_results = None

    # ========================================
    # Build Consolidated Metrics DataFrames
    # ========================================
    # Combine train/test metrics into single DataFrame for easy comparison
    metrics_df = pd.DataFrame([
        {"split": "train", "rmse": train_rmse, "mae": train_mae, "r2": train_r2, "mse": train_mse},
        {"split": "test", "rmse": test_rmse, "mae": test_mae, "r2": test_r2, "mse": test_mse}
    ])
    
    # Build classification metrics DataFrame if computed
    classification_df = pd.DataFrame(classification_records) if include_classification_metrics else None
    
    # Round to 4 decimal places for readability
    metrics_df['rmse'] = metrics_df['rmse'].round(4)
    metrics_df['mae'] = metrics_df['mae'].round(4)
    metrics_df['r2'] = metrics_df['r2'].round(4)
    metrics_df['mse'] = metrics_df['mse'].round(4)

    # ========================================
    # Metrics Definitions with LaTeX
    # ========================================
    # Provide LaTeX equations for each metric for documentation/reporting
    metrics_defs = pd.DataFrame([
        {"metric": "RMSE", "latex": r"\\mathrm{RMSE} = \\sqrt{\\tfrac{1}{N} \\sum_{i=1}^{N} (y_i - \\hat{y}_i)^2}"},
        {"metric": "MAE",  "latex": r"\\mathrm{MAE} = \\tfrac{1}{N} \\sum_{i=1}^{N} |y_i - \\hat{y}_i|"},
        {"metric": "R^2",  "latex": r"R^2 = 1 - \\dfrac{\\sum_{i=1}^{N} (y_i - \\hat{y}_i)^2}{\\sum_{i=1}^{N} (y_i - \\bar{y})^2}"},
        {"metric": "MSE",  "latex": r"\\mathrm{MSE} = \\tfrac{1}{N} \\sum_{i=1}^{N} (y_i - \\hat{y}_i)^2"},
    ])

    # ========================================
    # Extract Feature Importance (Coefficients)
    # ========================================
    # Extract and display top 10 most important features by absolute coefficient
    # Helps understand which features drive predictions
    try:
        # Extract coefficients from the linear regression model (last stage in pipeline)
        coefficients = model.model.stages[-1].coefficients.toArray()
        
        # Try to extract feature names from metadata (includes OHE expansions)
        feats_meta = test_preds.schema["features"].metadata
        attrs = []
        if "ml_attr" in feats_meta and "attrs" in feats_meta["ml_attr"]:
            ml_attrs = feats_meta["ml_attr"]["attrs"]
            # Collect attributes from all types (binary, numeric, nominal)
            for t in ["binary", "numeric", "nominal"]:
                if t in ml_attrs:
                    attrs.extend(ml_attrs[t])
            # Sort by vector index to match coefficient order
            attrs = sorted(attrs, key=lambda x: x["idx"])
            feature_names = [a.get("name", f"feature_{a['idx']}") for a in attrs]
        else:
            # Fallback: use generic names if metadata unavailable
            feature_names = [f"feature_{i}" for i in range(len(coefficients))]
        
        # Safety check: ensure lengths match
        if len(feature_names) != len(coefficients):
            feature_names = [f"feature_{i}" for i in range(len(coefficients))]
        
        # Create DataFrame with feature names and coefficients
        coef_df = pd.DataFrame({"feature": feature_names, "coefficient": coefficients})
        coef_df["abs_coefficient"] = coef_df["coefficient"].abs()
        coef_df = coef_df.sort_values("abs_coefficient", ascending=False)
        
        # Round coefficients to 4 decimal places
        top_coef_df = coef_df.head(10).copy()
        top_coef_df['coefficient'] = top_coef_df['coefficient'].round(4)
        top_coef_df['abs_coefficient'] = top_coef_df['abs_coefficient'].round(4)
        
        print("\n" + "="*80)
        print("Top 10 Most Important Features (by absolute coefficient):")
        print("="*80)
        print(top_coef_df.to_string(index=False))
    except Exception as e:
        print(f"[Info] Skipping coefficient listing: {e}")
    
    # ========================================
    # Print Total Runtime
    # ========================================
    # Report total execution time for performance tracking
    overall_elapsed = time.time() - overall_start_time
    hours, remainder = divmod(overall_elapsed, 3600)
    minutes, seconds = divmod(remainder, 60)
    print("\n" + "="*80)
    print(f"TOTAL RUNTIME: {int(hours)}h {int(minutes)}m {seconds:.2f}s ({overall_elapsed:.2f} seconds)")
    print("="*80)

    # ========================================
    # Free Memory
    # ========================================
    # Force Python garbage collection to free driver memory
    gc.collect()
    
    # Clear Spark SQL cache to free executor memory
    spark.catalog.clearCache()

    # ========================================
    # Return Results
    # ========================================
    train_metrics_dict = {"rmse": train_rmse, "mae": train_mae, "r2": train_r2, "mse": train_mse}
    test_metrics_dict = {"rmse": test_rmse, "mae": test_mae, "r2": test_r2, "mse": test_mse}
    return model, train_metrics_dict, test_metrics_dict, metrics_df, metrics_defs, classification_df, test_class_results if include_classification_metrics else None


# ============================================================================
# Main Execution Block
# ============================================================================

if __name__ == "__main__":
    """
    Main execution entry point for the script.
    
    Workflow:
        1. Print cluster configuration (for reproducibility and debugging)
        2. Load the OTPW 12M 2015 Parquet dataset from the team folder
        3. Run a time-based train/test split:
           - Training: first three-quarters of the 1-year dataset (months 1–9)
           - Blind testing: last quarter of the 1-year dataset (months 10–12)
        4. Display regression and classification metrics.
    """
    
    # ========================================
    # Print Cluster Configuration
    # ========================================
    sc = spark.sparkContext
    
    num_executors = len(sc._jsc.sc().statusTracker().getExecutorInfos()) - 1
    executor_memory = sc.getConf().get("spark.executor.memory", "Unknown")
    executor_cores = sc.getConf().get("spark.executor.cores", "Unknown")
    actual_cores = sc.defaultParallelism / num_executors if num_executors > 0 else "Unknown"
    print(f"Estimated cores per executor: {(actual_cores)}")
    
    print("\n" + "="*80)
    print("CLUSTER CONFIGURATION")
    print("="*80)
    print(f"Cluster Size: {num_executors} executors, {executor_cores} cores per executor, {executor_memory} RAM per executor")
    print("="*80 + "\n")
    
    # ========================================
    # Load OTPW 12M 2015 Parquet dataset
    # ========================================
    df_otpw12_parq = spark.read.parquet("dbfs:/student-groups/Group_4_2/OTPW_12M_2015_parquet")
    
    # ========================================
    # Run Train/Test Split (3/4 vs 1/4)
    # ========================================
    model, train_metrics_dict, test_metrics_dict, metrics_df, metrics_defs, classification_df, test_class_results = run_train_test_split(
        df_otpw12_parq,
        train_months_end=9,  # Train on months 1-9, test on months 10-12
        include_classification_metrics=True,
    )
    
    # ========================================
    # Display Regression Metrics
    # ========================================
    print("\n" + "="*80)
    print("REGRESSION METRICS")
    print("="*80)
    display(metrics_df)
    display(metrics_defs)
    
    # ========================================
    # Display Classification Metrics
    # ========================================
    if classification_df is not None:
        print("\n" + "="*80)
        print("CLASSIFICATION METRICS SUMMARY")
        print("="*80)
        display(classification_df)

In [0]:
displayHTML("""
<!DOCTYPE html>
<html>
<head>
  <script src="https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.min.js"></script>
  <script>
    mermaid.initialize({
      startOnLoad: true,
      theme: 'dark',
      themeVariables: {
        primaryColor: '#4a5568',
        primaryTextColor: '#fff',
        primaryBorderColor: '#cbd5e0',
        lineColor: '#cbd5e0',
        secondaryColor: '#2d3748',
        tertiaryColor: '#1a202c',
        background: '#1a202c',
        mainBkg: '#4a5568',
        secondBkg: '#2d3748',
        tertiaryBkg: '#1a202c'
      }
    });
  </script>
  <style>
    body { background-color: #1a202c; }
    .mermaid {
      background-color: #1a202c;
      font-size: 24px;
    }
  </style>
</head>
<body>
<div class="mermaid">
flowchart LR

    %% Input
    Input["<b>Input</b><br/>OTPW 12M 2015 Parquet<br/>10 Features + DEP_DELAY"]

    %% Stage 1: Data Preparation
    subgraph S1["<b>Stage 1: Data Preparation</b>"]
        LabelClean["Cast DEP_DELAY to Double<br/>Filter Null/NaN Labels"]
        SelectFeat["Select 10 Baseline Features<br/>Temporal, Airport, Flight, Weather"]
        NumClean["Clean Numerical Features:<br/>Remove non-numeric chars<br/>Empty → Null<br/>Cast to Double"]
        LabelClean --> SelectFeat --> NumClean
    end

    %% Stage 2: Numerical Imputation (Median)
    subgraph S2["<b>Stage 2: Numerical Imputation (Median)</b>"]
        ImpWind["HourlyWindSpeed_imputed<br/>median(HourlyWindSpeed)"]
        ImpVis["HourlyVisibility_imputed<br/>median(HourlyVisibility)"]
        ImpPrec["HourlyPrecipitation_imputed<br/>median(HourlyPrecipitation)"]
        ImpDist["DISTANCE_imputed<br/>median(DISTANCE)"]
    end

    %% Stage 3: Categorical Encoding
    subgraph S3["<b>Stage 3: Categorical Encoding</b>"]
        DOW["DAY_OF_WEEK_clean<br/>StringIndexer + OHE"]
        Month["MONTH_clean<br/>StringIndexer + OHE"]
        TimeBlk["DEP_TIME_BLK_clean<br/>StringIndexer + OHE"]
        Origin["ORIGIN_clean<br/>StringIndexer + OHE"]
        Dest["DEST_clean<br/>StringIndexer + OHE"]
        Carrier["OP_UNIQUE_CARRIER_clean<br/>StringIndexer + OHE"]
    end

    %% Stage 4: Feature Assembly
    subgraph S4["<b>Stage 4: Feature Assembly</b>"]
        Assemble["VectorAssembler<br/>Combine Imputed Numerics + Encoded Categoricals<br/>→ features"]
    end

    %% Stage 5: Standardization
    subgraph S5["<b>Stage 5: Standardization</b>"]
        Scale["StandardScaler<br/>features → scaled_features<br/>withMean=True, withStd=True"]
    end

    %% Stage 6: Linear Regression
    subgraph S6["<b>Stage 6: Model</b>"]
        LR["Linear Regression<br/>featuresCol=scaled_features<br/>labelCol=DEP_DELAY<br/>maxIter=100<br/>regParam=0.0, elasticNet=0.0"]
    end

    %% Time-based Split (Evaluation)
    subgraph S7["<b>Evaluation Split</b>"]
        Split["Time-Based Split<br/>Train: Months 1–9<br/>Blind Test: Months 10–12"]
    end

    %% Output
    Output["<b>Output</b><br/>Predictions<br/>DEP_DELAY in minutes<br/>+ OTPA / SDDR / 4-Bucket Metrics"]

    %% Connections
    Input --> S1
    S1 --> S2
    S1 --> S3
    S2 --> Assemble
    S3 --> Assemble
    Assemble --> Scale --> LR --> Split --> Output

</div>
</body>
</html>
""")

In [0]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

data = [
    {"Model": "Model 1\n3M / No CV / Imputed",        "rmse": 37.1695, "mae": 18.7141, "r2": 0.0381, "mse": 1381.5691},
    {"Model": "Model 2\n3M / CV / Imputed",          "rmse": 34.2761, "mae": 17.6190, "r2": 0.0140, "mse": 1174.8532},
    {"Model": "Model 3\n12M / CV / Imputed",         "rmse": 35.9611, "mae": 18.2930, "r2": 0.0077, "mse": 1293.2014},
    {"Model": "Model 4\n12M / CV / DropNull",        "rmse": 34.9891, "mae": 17.7445, "r2": 0.0034, "mse": 1224.2345},
    {"Model": "Model 5\n12M / CV / Custom",          "rmse": 43.2399, "mae": 21.4274, "r2": 0.0171, "mse": 1869.6893},
    {"Model": "Model 6\n12M / 3/4–1/4 / Imputed",    "rmse": 36.0030, "mae": 17.8670, "r2": 0.0133, "mse": 1296.2151},
]

df = pd.DataFrame(data)
models = df["Model"].tolist()
metrics = ["rmse", "mae", "r2", "mse"]

# Fixed color palette, one color per model
colors = ["#4c72b0", "#55a868", "#c44e52", "#8172b3", "#ccb974", "#64b5cd"]

for metric in metrics:
    fig, ax = plt.subplots(figsize=(10, 5))
    x = range(len(models))
    values = df[metric].values

    bars = ax.bar(x, values, color=colors)

    ax.set_title(f"{metric.upper()} by Model")
    ax.set_ylabel(metric.upper())
    ax.set_xticks(x)
    ax.set_xticklabels(models, rotation=20, ha="right")

    # Legend: one entry per model, matching bar colors
    legend_patches = [
        mpatches.Patch(color=colors[i], label=models[i]) for i in range(len(models))
    ]
    ax.legend(handles=legend_patches, title="Model", bbox_to_anchor=(1.05, 1), loc="upper left")

    plt.tight_layout()
    display(fig)
    plt.close(fig)

In [0]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

data_cls = [
    {"Model": "Model 1\n3M / No CV / Imputed",        "otpa_accuracy": 0.7162, "otpa_f1": 0.8170, "sddr_recall": 0.0,    "bucket_accuracy": 0.2824},
    {"Model": "Model 2\n3M / CV / Imputed",          "otpa_accuracy": 0.7086, "otpa_f1": 0.8163, "sddr_recall": 0.0,    "bucket_accuracy": 0.2736},
    {"Model": "Model 3\n12M / CV / Imputed",         "otpa_accuracy": 0.6952, "otpa_f1": 0.8038, "sddr_recall": 0.0003, "bucket_accuracy": 0.2508},
    {"Model": "Model 4\n12M / CV / DropNull",        "otpa_accuracy": 0.7137, "otpa_f1": 0.8202, "sddr_recall": 0.0,    "bucket_accuracy": 0.2497},
    {"Model": "Model 5\n12M / CV / Custom",          "otpa_accuracy": 0.6702, "otpa_f1": 0.7782, "sddr_recall": 0.0,    "bucket_accuracy": 0.2585},
    {"Model": "Model 6\n12M / 3/4–1/4 / Imputed",    "otpa_accuracy": 0.7231, "otpa_f1": 0.8272, "sddr_recall": 0.0002, "bucket_accuracy": 0.2578},
]

df_cls = pd.DataFrame(data_cls)
models = df_cls["Model"].tolist()
metrics_cls = ["otpa_accuracy", "otpa_f1", "sddr_recall", "bucket_accuracy"]

# Same fixed color palette as before
colors = ["#4c72b0", "#55a868", "#c44e52", "#8172b3", "#ccb974", "#64b5cd"]

for metric in metrics_cls:
    fig, ax = plt.subplots(figsize=(10, 5))
    x = range(len(models))
    values = df_cls[metric].values

    bars = ax.bar(x, values, color=colors)

    ax.set_title(f"{metric.replace('_', ' ').upper()} by Model")
    ax.set_ylabel(metric.replace('_', ' ').upper())
    ax.set_xticks(x)
    ax.set_xticklabels(models, rotation=20, ha="right")

    # Legend: one entry per model
    legend_patches = [
        mpatches.Patch(color=colors[i], label=models[i]) for i in range(len(models))
    ]
    ax.legend(handles=legend_patches, title="Model", bbox_to_anchor=(1.05, 1), loc="upper left")

    plt.tight_layout()
    display(fig)
    plt.close(fig)